# Vroegsignalering — Effectiveness Analysis

Early-warning systems aim to reach households with debt before things escalate — but reaching
someone is only step one. This notebook asks *"did it work?"*: success means a household does
**not** come back with a new signal after its case is closed.

It uses three core tabs — **Signalen** (signals), **Meldingen** (cases), **Melders** (reporters) —
plus the contact-attempt tabs, linked by shared reference numbers.
**How to run:** set `FILE` in the setup cell to the export, then use **Restart Kernel and Run
All**. If a section fails, the error is written into the report and the run continues; the
report starts with a banner saying COMPLETE or INCOMPLETE, listing failed or missing sections,
data warnings (e.g. a tab that was not found) and the input file that was read.

**Output:** mostly aggregate statistics and charts. §14e profiles the ten most-signalling
households individually (labelled by rank, no identifiers, with dated signal timelines), and some
tables show small groups.


## 0. Install Dependencies

Run once to install the required Python libraries. Skipped automatically if they are already present.


In [ ]:
%pip install pandas numpy matplotlib seaborn openpyxl statsmodels scipy --quiet


## 1. Setup

Imports all libraries, sets chart styling, and installs the report capture: everything printed,
displayed or plotted below is also collected into one HTML results file (written in §13).
**Update the `FILE` variable** in the cell below if the export has a different name or location.


In [ ]:
# Force a NON-blocking, headless plotting backend. Without this, plt.show() can
# block forever waiting for a GUI window (TkAgg/QtAgg/MacOSX) — which makes the
# first plotting cell appear to "hang". 'inline' renders in the notebook and is
# non-blocking; if IPython is unavailable we fall back to the headless 'Agg'.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    import matplotlib
    matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
###################################################
FILE     = Path('bi_export_sample.xlsx')   # ← update to real file path if needed
###################################################
DATE_FMT = '%d-%m-%Y'
# Event dates after DATA_CUTOFF are treated as impossible (set to missing). Set it to the
# extraction date, e.g. pd.Timestamp('2026-07-03'), for reproducible runs; None = the run date.
DATA_CUTOFF = None
# Columns holding PLANNED dates (may lie in the future) are exempt from that guard.
PLANNED_DATE_COLS = {'Op welke datum'}
# OBSERVATION_END = the date through which signal reporting is COMPLETE (confirm with the data
# owner - the extraction date is not automatically complete). It ends follow-up, censoring and
# eligibility everywhere. None = fall back to the latest signal date in the file (printed).
OBSERVATION_END = None
# ANALYSIS_END = the last date the analyses use. Around 27-09-2025 the household key in the BI system
# changed (seen in Maastricht and Venlo on the same date): later signals no longer link to the same
# household, so returns after it are invisible. Rows after this date are left out of ALL analyses
# (printed in 1a, with a check whether the change is visible in this export). None = use all data.
ANALYSIS_END = '2025-09-27'
# Contact results dated AFTER a case was closed (some municipalities enter results later) still count
# for reach, help and final result, as long as they are dated BEFORE the household's next case opened;
# a later result belongs to the new case, not to this one. 1b shows how many and how late.
# LATE_RECORD_DAYS = optional extra limit in days after closing (None = no limit; 0 = never use
# records dated after closing, as in 3.8 and before).
LATE_RECORD_DAYS = None
# Names in THIS export that differ from the usual ones. Leave empty at first: section 0 matches
# capital / spacing variants itself, recognises the contact tabs by their columns, and lists every
# tab and column it could not match (with the closest names), so these can be filled in for a re-run.
#   TAB_NAMES    = {'Tussenresultaat': 'Resultaat', 'Eindresultaat': 'Resultaat1'}   usual tab: tab in the file
#   COLUMN_NAMES = {'Tussenresultaat': {'Contactdatum': 'Datum contactpoging'}}      tab: {column in the file: usual name}
TAB_NAMES = {}
COLUMN_NAMES = {}

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid')
COLORS = sns.color_palette('Set2')

# ══════════════════════════════════════════════════════════════════════════════
#  RESULTS-FILE CAPTURE
#  Every print(), every table shown with display(), and every chart shown with
#  plt.show() is automatically recorded AND written to a single self-contained
#  HTML file (elsa_results_report.html) at the end (Section 15).
#  Send that one file back — it opens in any browser and embeds all charts.
#  Cell outputs still appear in the notebook as normal.
# ══════════════════════════════════════════════════════════════════════════════
import io as _io, base64 as _b64, html as _html, datetime as _dt, builtins as _bi
from IPython.display import display as _ipy_display   # the REAL display (safe to re-import)

REPORT_PATH   = (FILE.parent if hasattr(FILE, 'parent') else Path('.')) / 'elsa_results_report.html'
_REPORT_BLOCKS = []

def _report_add(html_str):
    _REPORT_BLOCKS.append(html_str)

# ── Run bookkeeping: each run gets a new id; every section logs started / ok / FAILED ──
_RUN_ID = _dt.datetime.now().strftime('%Y%m%d-%H%M%S-%f')
_SECTION_LOG = {}          # section title -> (status, message)
_DATA_WARNINGS = []
# Sections this notebook contains (sections added later still run; they are just not
# checked for 'not run' in the banner unless added here).
EXPECTED_SECTIONS = ['0. Tab & Column Check', '1a. Load Data', '1b. Shared Case Definitions & Data Checks', '2a. Schema — tabs, columns, dtypes, missingness, cardinality', '2b. Exact Category Labels & Frequencies', '2c. Numeric Ranges & Date Coverage', '2d. Join Integrity & Structural Distributions', '2e. R0 - What does the system Recidive flag mean?', '2f. R1 - Recidive Fingerprint Census', '3. Signal Volume & Distribution', '3a. Reporter (Melder) Analysis', '4. Signal → Report Conversion', '5. Contact Effectiveness', '6. Client Response & Outcomes', '7. Case Closure & Timing', '7a. Recidive — Repeated Signals', '8. Signal-to-Case Conversion and Case Funnel', '9. Effectiveness Framework — data preparation', '9. Contact Decomposition - Does Contact Reduce Return?', '9b. Chained Cases - Contact Method x Final Result Patterns', '9c. Same-Clock Check - outcome measured from the signal date', '9d. Calendar Comparability Check', '10. Reach vs Effectiveness & Resolution Quality', '11. Targeting - Who Returns? (built-in Recidive flag)', '12. Return Timing, Contact Effort & Recurrence Patterns', '13. Module 1 - Within-Household Comparisons: preparation', '13a. E1 - Within-Household Comparison', '13b. E2 - State Dependence vs Chronic Vulnerability', '13c. E3 - Trajectory Around Help Acceptance', '13d. G4 - Silent Success vs Disappearance', '14a. G1 - Debt Cascade: Transitions Between Creditor Types', '14b. G2 - Breadth vs Depth', '14c. G3 - Trajectory Typology', '14d. G5 - Early-Warning Value per Signal Partner', '14e. Deep-Dive: The 10 Most-Signalling Households', '15a. H2 - Reporting Cadence per Partner', '15b. F1 - Energy Crisis 2022: Comparative Trends', '15c. F2 - Seasonality by Mechanism', '15d. H1 - Risk Stratification at First Contact', '15e. W - Workflow: Receptiveness Timing & Method Allocation']

def _report_section(title):
    _SECTION_LOG[str(title)] = ('started', '')
    _report_add(f"<h2 class='sec'>{_html.escape(str(title))}</h2>")

def _section_done(title):
    _SECTION_LOG[str(title)] = ('ok', '')

def _section_failed(title, err):
    # Records WHERE it failed: the line of this section's cell, plus the last steps of the trace
    # (helpers defined in other sections, library code). Code and error text only - no data tables.
    import traceback as _tb, sys as _sy
    msg = f'{type(err).__name__}: {err}'[:300]
    _fr = _tb.extract_tb(err.__traceback__)
    _own = _fr[0].filename if _fr else None            # first frame = this section's own cell
    _libs = tuple({_sy.prefix, _sy.base_prefix, _sy.exec_prefix})
    def _is_lib(f):
        return ('site-packages' in f.filename or f.filename.startswith(_libs)
                or f.filename.endswith(('.pyx', '.pxd', '.pxi')))   # compiled library code
    def _where(f):
        if not _is_lib(f):
            return ('this section' if f.filename == _own else 'helper from another section') + f', line {f.lineno}'
        p = f.filename.replace('\\', '/')
        p = p.split('site-packages/')[-1] if 'site-packages/' in p else '/'.join(p.split('/')[-3:])
        return f'{p}, line {f.lineno}'
    _at = [f for f in _fr if f.filename == _own]
    _at = _at[-1] if _at else None
    _loc = (f"line {_at.lineno} of this section: {(_at.line or '(code not available)').strip()[:160]}"
            if _at else 'line unknown')
    _SECTION_LOG[str(title)] = ('FAILED', f'{msg}  [at {_loc}]')
    print(f'  SECTION FAILED - {title}: {msg}')
    print(f'    failed at {_loc}')
    print('    trace (last steps, most recent last):')
    for f in _fr[-4:]:
        print(f"      {_where(f)}: {(f.line or '').strip()[:140]}")
    print('  The run continues; the banner at the top of the report lists every failed section.')
    plt.close('all')

def _require_data():
    if globals().get('DATA_LOADED_RUN') != _RUN_ID:
        raise RuntimeError('the data were not loaded in this run (section 1a did not finish) - '
                           'use Restart Kernel and Run All')

def _warn(msg):
    _DATA_WARNINGS.append(str(msg))
    print(f'  WARNING: {msg}')

# Tee print() → real stdout + report. builtins.print is always the genuine print,
# so re-running this cell never stacks wrappers on top of each other.
_orig_print = _bi.print
def print(*args, **kwargs):
    _orig_print(*args, **kwargs)
    sep = kwargs.get('sep', ' '); end = kwargs.get('end', '\n')
    _report_add(f"<pre>{_html.escape(sep.join(str(a) for a in args) + end)}</pre>")

# Tee display() → real display + report. _ipy_display is imported straight from
# IPython above, NOT taken from the namespace, so it can never become this wrapper
# (which previously caused infinite recursion when the cell was run twice).
def display(obj, *args, **kwargs):
    _ipy_display(obj, *args, **kwargs)
    try:
        if isinstance(obj, pd.DataFrame):
            _report_add(obj.to_html(border=0))
        elif isinstance(obj, pd.Series):
            _report_add(obj.to_frame().to_html(border=0))
        else:
            _report_add(f"<pre>{_html.escape(str(obj))}</pre>")
    except Exception:
        pass

# Tee plt.show() → embed the current figure as a PNG. Guarded so that re-running
# this cell does not wrap an already-wrapped show() (which would double-capture).
if not getattr(plt.show, '_elsa_wrapped', False):
    _orig_show = plt.show
    def _report_show(*args, **kwargs):
        fig = plt.gcf()
        try:
            buf = _io.BytesIO()
            fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
            buf.seek(0)
            _report_add(f"<img src='data:image/png;base64,{_b64.b64encode(buf.read()).decode()}'/>")
        except Exception:
            pass
        _orig_show(*args, **kwargs)
    _report_show._elsa_wrapped = True
    plt.show = _report_show

# Model warnings (no convergence, perfect separation, singular Hessian) stay VISIBLE and are
# written into the report next to the model they belong to; other library noise stays hidden.
from statsmodels.tools import sm_exceptions as _sme
for _wcat in ('ConvergenceWarning', 'PerfectSeparationWarning', 'HessianInversionWarning'):
    if hasattr(_sme, _wcat):
        warnings.filterwarnings('always', category=getattr(_sme, _wcat))
def _report_warning(message, category, filename, lineno, file=None, line=None):
    print(f"  MODEL WARNING ({category.__name__}): {message}")
warnings.showwarning = _report_warning

# Run environment, written into the report header (reproducibility)
import sys as _sys, statsmodels as _sm_pkg
RUN_ENV = (f"Python {_sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | "
           f"statsmodels {_sm_pkg.__version__}")


### 0. Tab & Column Check

Tab and column names can differ per municipality. Before anything is loaded, this section reads only the
**names** of the tabs and columns and:

1. lists every tab with its columns;
2. matches each tab to its role — names declared in `TAB_NAMES` (setup cell) first, then the usual name
   (ignoring capitals, spaces and punctuation), then a key word in the name. Contact tabs whose name is
   not recognised (e.g. Venlo's `Resultaat` / `Resultaat1`) are recognised by having the contact columns,
   and told apart by how often the final result is filled in (the final-result tab far more often);
3. checks every column the analyses need — declared `COLUMN_NAMES` first, then the usual name ignoring
   capitals, spaces and punctuation — and lists what is missing with the closest column names in that
   tab, also in the banner. Missing columns are added empty later, so the sections that need them show
   little; fill in `COLUMN_NAMES` with the right names and run again.

Section 1a loads the tabs exactly as matched here and renames the columns.


In [ ]:
_report_section('0. Tab & Column Check')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    import re, difflib
    xls = pd.ExcelFile(FILE)
    def _nm(s):
        return re.sub(r'[^a-z0-9]', '', str(s).lower())
    # Columns the analyses need, per tab (missing ones are added empty in 1b)
    NEEDED_COLUMNS = {
        'Signalen': ['SignaalID', 'Hash', 'Datum melding', 'Meldingnummer', 'Melder ID', 'Doorgezet', 'Recidive',
                     *[f'Recidive maand-{_i}' for _i in range(1, 7)], 'Leeftijd', 'Minderjarige kinderen',
                     'Type melding', 'Achterstandsbedrag'],
        'Meldingen': ['Meldingnummer', 'Afgerond', 'Afgerond d.d.', 'Ingetrokken', 'Onterecht'],
        'Melders': ['ID', 'Melder', 'Type melder'],
        'Tussenresultaat': ['Meldingnummer', 'Datum contactpoging', 'Wijze van contactpoging', 'Persoon bereikt'],
        'Eindresultaat': ['Meldingnummer', 'Datum contactpoging', 'Wijze van contactpoging', 'Persoon bereikt',
                          'Wil de klant hulp', 'Wat is het eindresultaat'],
        'Follow-Up resultaat': ['Meldingnummer', 'Datum'],
    }
    _ROLE_KEYS = [('Signalen', 'signal'), ('Meldingen', 'melding'), ('Melders', 'melder'),
                  ('Tussenresultaat', 'tussen'), ('Eindresultaat', 'eind'), ('Follow-Up resultaat', 'follow')]
    _COLS = {sh: [str(c) for c in pd.read_excel(xls, sh, nrows=0).columns] for sh in xls.sheet_names}
    print('Tabs and columns in the file:')
    for sh, cs in _COLS.items():
        print(f"  '{sh}' ({len(cs)} columns): {cs}")

    # ── 1. tabs ──
    _tabs, _how, _free = {}, {}, list(xls.sheet_names)
    for _role, _name in (globals().get('TAB_NAMES') or {}).items():          # declared in the setup cell
        if _name in _free:
            _tabs[_role], _how[_role] = _name, 'TAB_NAMES'; _free.remove(_name)
        else:
            print(f"  NOTE: TAB_NAMES gives '{_name}' for {_role}, but the file has no such tab")
    for _pass, _label in ((1, 'usual name'), (2, 'key word in the name')):
        for _role, _key in _ROLE_KEYS:
            if _role in _tabs:
                continue
            _hit = [s for s in _free if (_nm(s) == _nm(_role) if _pass == 1 else _key in _nm(s))]
            if _hit:
                _tabs[_role], _how[_role] = _hit[0], _label; _free.remove(_hit[0])
    # contact tabs not found by name: tabs with the contact columns; final-result fill tells them apart
    _need_c = [r for r in ('Tussenresultaat', 'Eindresultaat') if r not in _tabs]
    _cand = [s for s in _free if {'meldingnummer', 'datumcontactpoging'} <= {_nm(c) for c in _COLS[s]}]
    if _need_c and _cand:
        def _fill(sh):
            _c = [c for c in _COLS[sh] if _nm(c) == 'watisheteindresultaat']
            if not _c:
                return 0.0
            _v = pd.read_excel(xls, sh, usecols=[_c[0]])[_c[0]]
            return float((_v.notna() & (_v.astype(str).str.strip() != '')).mean()) if len(_v) else 0.0
        _f = {s: _fill(s) for s in _cand}
        print('  Contact tab(s) recognised by their columns - final result filled in: '
              + ', '.join(f"'{s}' {_f[s]*100:.1f}%" for s in _cand))
        if len(_need_c) == 2 and len(_cand) >= 2:
            _e = max(_cand, key=_f.get)
            _t = max([s for s in _cand if s != _e], key=lambda s: len(pd.read_excel(xls, s, usecols=[0])))
            _tabs['Eindresultaat'], _how['Eindresultaat'] = _e, 'contact columns, final result filled most'
            _tabs['Tussenresultaat'], _how['Tussenresultaat'] = _t, 'contact columns, final result mostly empty'
        elif len(_need_c) == 1:
            _pick = max(_cand, key=_f.get) if _need_c[0] == 'Eindresultaat' else min(_cand, key=_f.get)
            _tabs[_need_c[0]], _how[_need_c[0]] = _pick, 'contact columns'
        else:
            print(f"  NOTE: only one contact tab ('{_cand[0]}') - it cannot be both attempts and final results; "
                  "set TAB_NAMES")
        _free = [s for s in _free if s not in _tabs.values()]
    print('\nTab matching (role <- tab in the file):')
    for _role, _ in _ROLE_KEYS:
        print(f"  {_role:<20} <- " + (f"'{_tabs[_role]}' ({_how[_role]})" if _role in _tabs else 'NOT FOUND'))
    if _free:
        print(f"  Tab(s) not used: {_free}")
    _tmiss = [r for r, _ in _ROLE_KEYS if r not in _tabs]
    if _tmiss:
        _warn(f"tab(s) not found: {_tmiss} - the file has {list(xls.sheet_names)}; set TAB_NAMES in the setup "
              "cell (section 0 lists every tab with its columns)")

    # ── 2. columns ──
    _rename, _rows = {}, []
    for _role, _sh in _tabs.items():
        _cs = _COLS[_sh]
        _rn = {}
        for _orig, _usual in ((globals().get('COLUMN_NAMES') or {}).get(_role) or {}).items():   # declared
            if _orig in _cs:
                _rn[_orig] = _usual
            else:
                print(f"  NOTE: COLUMN_NAMES gives '{_orig}' for {_role}, but tab '{_sh}' has no such column")
        _have = {_nm(_rn.get(c, c)): c for c in _cs}
        _miss = []
        for _need in NEEDED_COLUMNS[_role]:
            if _need in _rn.values():
                _rows.append((_role, _need, [o for o, u in _rn.items() if u == _need][0], 'COLUMN_NAMES'))
            elif _need in _cs:
                pass                                                      # exact name: not listed
            elif _nm(_need) in _have:
                _rn[_have[_nm(_need)]] = _need
                _rows.append((_role, _need, _have[_nm(_need)], 'capitals / spacing'))
            else:
                _miss.append(_need)
        for _need in _miss:
            _used = set(_rn) | set(NEEDED_COLUMNS[_role])
            _close = difflib.get_close_matches(_nm(_need), [_nm(c) for c in _cs if c not in _used], n=3, cutoff=0.5)
            _closest = [c for c in _cs if _nm(c) in _close]
            _rows.append((_role, _need, 'NOT FOUND', f"closest in this tab: {_closest}" if _closest else 'no similar name'))
        if _miss:
            _warn(f"{_role} (tab '{_sh}'): column(s) not found: {_miss} - see section 0 for the closest names; "
                  "set COLUMN_NAMES in the setup cell")
        if _rn:
            _rename[_role] = _rn
    if _rows:
        print('\nNeeded columns NOT under their usual name (tab, usual name, name in the file, how):')
        display(pd.DataFrame(_rows, columns=['tab', 'usual name', 'in the file', 'how / closest']))
    else:
        print('\nAll needed columns found under their usual names.')
    NAME_CHECK_TABS, NAME_CHECK_COLS = _tabs, _rename
    NAME_CHECK_RUN = _RUN_ID          # 1a uses this matching only when it comes from THIS run
    _section_done('0. Tab & Column Check')
except Exception as _err:
    _section_failed('0. Tab & Column Check', _err)


### 1a. Load Data

Tabs are loaded as matched in section 0 and columns renamed to the usual names; the report warns if
the two contact tabs look swapped (the final-result tab should have the final result filled in far
more often).

Reads every tab from the Excel file — tabs and columns are recognised regardless of capitals,
spaces and small variations (e.g. "Tussenresultaten"); the report lists which tab was used for
what, and a missing contact tab gives a warning in the banner — parses date columns into a consistent format, and defines
the helper that converts Dutch `Ja`/`Nee` values to booleans. Row counts are printed in total and per year
(each tab on its own date column) — no data values.
Dates are read in stages (DD-MM-YYYY, then Excel serial numbers, then ISO YYYY-MM-DD, then any
other day-first format), so a minority format is not lost; non-empty values that still cannot be read are
counted. Data-quality guards: placeholder dates (before 2000) and event dates after the data
cutoff are set to missing, with an audit note printed into the report. The cutoff is
`DATA_CUTOFF` from the setup cell (the extraction date, for reproducible runs) or, if not set,
the run date; planned-date columns (`PLANNED_DATE_COLS`, e.g. a scheduled follow-up) are exempt.


In [ ]:
_report_section('1a. Load Data')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    import re, os
    xls = pd.ExcelFile(FILE)
    _fst = os.stat(FILE)
    INPUT_INFO = (f"{Path(FILE).name} | {_fst.st_size:,} bytes | "
                  f"modified {_dt.datetime.fromtimestamp(_fst.st_mtime):%Y-%m-%d %H:%M}")
    print(f"  Input file: {INPUT_INFO}")

    # ── Tabs: matched ignoring capitals / spaces / punctuation, then by key word ──
    def _ntab(s):
        return re.sub(r'[^a-z]', '', str(s).lower())
    _ROLE_KEYS = [('Signalen', 'signal'), ('Meldingen', 'melding'), ('Melders', 'melder'),
                  ('Tussenresultaat', 'tussen'), ('Eindresultaat', 'eind'), ('Follow-Up resultaat', 'follow')]
    _free = list(xls.sheet_names)
    TAB_SOURCE = {}
    if globals().get('NAME_CHECK_RUN') == _RUN_ID:   # tabs as matched in section 0 (this run)
        TAB_SOURCE = dict(NAME_CHECK_TABS)
        _free = [s for s in _free if s not in TAB_SOURCE.values()]
    for _role, _name in (globals().get('TAB_NAMES') or {}).items():   # 0) names declared in the setup cell
        if _role not in TAB_SOURCE and _name in _free:
            TAB_SOURCE[_role] = _name; _free.remove(_name)
    for _role, _key in _ROLE_KEYS:                       # 1) same name apart from capitals / spaces
        if _role in TAB_SOURCE:
            continue
        _hit = [s for s in _free if _ntab(s) == _ntab(_role)]
        if _hit:
            TAB_SOURCE[_role] = _hit[0]; _free.remove(_hit[0])
    for _role, _key in _ROLE_KEYS:                       # 2) close variant containing the key word
        if _role not in TAB_SOURCE:
            _hit = [s for s in _free if _key in _ntab(s)]
            if _hit:
                TAB_SOURCE[_role] = _hit[0]; _free.remove(_hit[0])
    print(f"  Tabs in the file: {list(xls.sheet_names)}")
    for _role, _ in _ROLE_KEYS:
        if _role in TAB_SOURCE:
            print(f"  {_role:<20} <- tab '{TAB_SOURCE[_role]}'")
    _miss_req = [r for r in ('Signalen', 'Meldingen', 'Melders') if r not in TAB_SOURCE]
    if _miss_req:
        raise RuntimeError(f"required tab(s) {_miss_req} not found - the file has {list(xls.sheet_names)}")
    for _role in ('Tussenresultaat', 'Eindresultaat', 'Follow-Up resultaat'):
        if _role not in TAB_SOURCE and globals().get('NAME_CHECK_RUN') != _RUN_ID:
            _warn(f"tab '{_role}' not found (the file has {list(xls.sheet_names)}) - "
                  + ('all contact-based results will be empty' if _role != 'Follow-Up resultaat'
                     else 'the follow-up tab is not used'))

    signalen  = pd.read_excel(xls, TAB_SOURCE['Signalen'])
    meldingen = pd.read_excel(xls, TAB_SOURCE['Meldingen'])
    melders   = pd.read_excel(xls, TAB_SOURCE['Melders'])
    extra_tabs = {r: pd.read_excel(xls, TAB_SOURCE[r])
                  for r in ('Tussenresultaat', 'Eindresultaat', 'Follow-Up resultaat') if r in TAB_SOURCE}
    extra_tabs.update({s: pd.read_excel(xls, s) for s in _free})     # any other tab, under its own name

    # Check that attempts and final results are not swapped: the final-result tab should have
    # 'Wat is het eindresultaat' filled in far more often than the attempts tab.
    def _efill(df):
        _c = [c for c in df.columns if re.sub(r'[^a-z]', '', str(c).lower()) == 'watisheteindresultaat']
        if not len(df):
            return float('nan')
        if not _c:
            return 0.0                                   # no final-result column at all
        _v = df[_c[0]]
        return float((_v.notna() & (_v.astype(str).str.strip() != '')).mean())
    if 'Tussenresultaat' in extra_tabs and 'Eindresultaat' in extra_tabs:
        _ft, _fe = _efill(extra_tabs['Tussenresultaat']), _efill(extra_tabs['Eindresultaat'])
        print(f"  Final result filled in: {_ft*100:.1f}% of rows in '{TAB_SOURCE['Tussenresultaat']}' (attempts), "
              f"{_fe*100:.1f}% in '{TAB_SOURCE['Eindresultaat']}' (final results)")
        if _ft == _ft and _fe == _fe and _ft > _fe:
            _warn(f"the contact tabs may be swapped: '{TAB_SOURCE['Tussenresultaat']}' (used as attempts) has the "
                  f"final result filled in more often than '{TAB_SOURCE['Eindresultaat']}' - check TAB_NAMES")

    # ── Column names as matched in section 0 (declared COLUMN_NAMES, capital / spacing variants) ──
    if globals().get('NAME_CHECK_RUN') == _RUN_ID:
        for _role, _df in [('Signalen', signalen), ('Meldingen', meldingen), ('Melders', melders),
                           *extra_tabs.items()]:
            _rn = NAME_CHECK_COLS.get(_role, {})
            if _rn:
                _df.rename(columns=_rn, inplace=True)
                print(f"  {_role}: column(s) renamed as matched in section 0: {_rn}")

    # ── Column names: matched ignoring capitals / spaces / punctuation ────────────
    _KNOWN_COLS = ['Aantal melders', 'Achterstandsbedrag', 'Afgerond', 'Afgerond d.d.', 'Bakje', 'Belangrijkste oorzaak schulden', 'Bereikt', 'Bij wie', 'Dagdeel', 'Datum', 'Datum contactpoging', 'Datum doorgezet', 'Datum melding', 'Doorgezet', 'Gemeente', 'Gezien', 'Gezien d.d.', 'Hash', 'ID', 'Ingetrokken', 'Inkomensbronnen', 'Leeftijd', 'Match', 'Melder', 'Melder (submerk)', 'Melder ID', 'Meldingnummer', 'Minderjarige kinderen', 'Naam medewerker', 'Onderwerp', 'Onterecht', 'Op welke datum', 'Persoon bereikt', 'Postcode', 'Recidive', 'Recidive maand-1', 'Recidive maand-2', 'Recidive maand-3', 'Recidive maand-4', 'Recidive maand-5', 'Recidive maand-6', 'Schuiver', 'SignaalID', 'Soort melding', 'Soort verwijzing', 'Startdatum contact', 'Termijnbedrag', 'Toelichting bij de FollowUp', 'Type melder', 'Type melding', 'Uitvoerder', 'Verwijzing anders namelijk', 'Waarnaar is verwezen', 'Wat is de voortgang', 'Wat is globaal het totale schuldbedrag', 'Wat is het eindresultaat', 'Wijk', 'Wijze van contact', 'Wijze van contactpoging', 'Wil de klant hulp', 'Wilt u de situatie invullen', 'Wilt u een nieuwe FollowUp inplannen', 'Zijn er minderjarige kinderen', 'Zijn er nog meer schuldeisers', 'Zijn of worden er inkomensverruimende voorzieningen toegepast']
    def _ncol(s):
        return re.sub(r'[^a-z0-9]', '', str(s).lower())
    _canon = {_ncol(c): c for c in _KNOWN_COLS}
    for _df in [signalen, meldingen, melders, *extra_tabs.values()]:
        _ren = {c: _canon[_ncol(c)] for c in _df.columns if _ncol(c) in _canon and c != _canon[_ncol(c)]}
        if _ren:
            print(f"  NOTE: column name(s) matched to the expected spelling: {_ren}")
            _df.rename(columns=_ren, inplace=True)
    for _role, _cols in [('Tussenresultaat', ['Meldingnummer', 'Datum contactpoging', 'Wijze van contactpoging',
                                              'Persoon bereikt']),
                         ('Eindresultaat', ['Meldingnummer', 'Datum contactpoging', 'Wijze van contactpoging',
                                            'Persoon bereikt', 'Wil de klant hulp', 'Wat is het eindresultaat'])]:
        if _role in extra_tabs:
            _m = [c for c in _cols if c not in extra_tabs[_role].columns]
            if _m and globals().get('NAME_CHECK_RUN') != _RUN_ID:
                _warn(f"tab '{TAB_SOURCE[_role]}' ({_role}) is missing column(s) {_m}")

    # ── Parse date columns (robust to already-datetime cells and other formats) ───
    RAW_DATE_SAMPLES = {}   # original text of date columns, captured BEFORE parsing (for v2)
    def parse_dates(df, cols):
        # Stages: DD-MM-YYYY -> Excel serial day numbers -> ISO (YYYY-MM-DD) -> any other day-first
        # format (value by value), so a minority format is not lost. Blank cells are not failures.
        for c in cols:
            if c not in df.columns:
                continue
            s = df[c]
            if pd.api.types.is_datetime64_any_dtype(s):
                continue                      # Excel already gave real date cells
            ex = s.dropna().astype(str).head(3).tolist()   # example raw values reveal the format
            if ex:
                RAW_DATE_SAMPLES.setdefault(c, ex)
            txt = s.astype(str).str.strip()
            nonblank = s.notna() & (txt != '') & (txt.str.lower() != 'nan')
            parsed = pd.to_datetime(txt.where(nonblank), format=DATE_FMT, errors='coerce')
            rest = nonblank & parsed.isna()
            if rest.any():                    # Excel serial day numbers (1954-2119)
                num = pd.to_numeric(s[rest], errors='coerce')
                num = num[num.between(20000, 80000)]
                if len(num):
                    parsed.loc[num.index] = pd.to_datetime(num, unit='D', origin='1899-12-30')
                rest = nonblank & parsed.isna()
            if rest.any():                    # ISO dates (YYYY-MM-DD ...) - before the day-first fallback,
                parsed.loc[rest] = pd.to_datetime(txt[rest], format='ISO8601', errors='coerce')  # which would swap them
                rest = nonblank & parsed.isna()
            if rest.any():                    # any other day-first format
                try:
                    parsed.loc[rest] = pd.to_datetime(txt[rest], format='mixed', dayfirst=True, errors='coerce')
                except (TypeError, ValueError):
                    parsed.loc[rest] = pd.to_datetime(txt[rest], dayfirst=True, errors='coerce')
            _fail = int((nonblank & parsed.isna()).sum())
            if _fail:
                print(f"  NOTE: {c}: {_fail} non-empty value(s) could not be read as a date (set to missing)")
            df[c] = parsed

    parse_dates(signalen,  ['Datum melding', 'Datum doorgezet', 'Startdatum contact'])
    parse_dates(meldingen, ['Afgerond d.d.', 'Gezien d.d.'])
    for df in extra_tabs.values():
        date_cols = [c for c in df.columns if 'datum' in c.lower() or c.lower() == 'datum']
        parse_dates(df, date_cols)

    # ── Null out placeholder dates (real export contains e.g. 1000-01-01) ─────────
    _MIN_VALID_DATE = pd.Timestamp('2000-01-01')
    for _name, _df in [('Signalen', signalen), ('Meldingen', meldingen), *extra_tabs.items()]:
        for _c in _df.columns:
            if pd.api.types.is_datetime64_any_dtype(_df[_c]):
                _n = int((_df[_c] < _MIN_VALID_DATE).sum())
                if _n:
                    print(f"  NOTE: {_name}.{_c}: {_n} placeholder date(s) before "
                          f"{_MIN_VALID_DATE.date()} set to missing")
                    _df.loc[_df[_c] < _MIN_VALID_DATE, _c] = pd.NaT

    # ── Null out impossible event dates (after the data cutoff) ───────────────────
    # DATA_CUTOFF (setup cell) = extraction date if declared, otherwise the run date. Planned-date
    # columns (PLANNED_DATE_COLS, e.g. a scheduled follow-up) may lie in the future and are exempt.
    _CUTOFF = (pd.Timestamp(DATA_CUTOFF) if globals().get('DATA_CUTOFF') is not None
               else pd.Timestamp.today()).normalize()
    _MAX_VALID_DATE = _CUTOFF + pd.Timedelta(days=1)       # EXCLUSIVE: midnight after the cutoff day
    _PLANNED = set(globals().get('PLANNED_DATE_COLS', set()))
    print(f"  Date guard: event dates after {_CUTOFF.date()} are set to missing "
          f"({'declared DATA_CUTOFF' if globals().get('DATA_CUTOFF') is not None else 'run date - declare DATA_CUTOFF for reproducible runs'}); "
          f"planned-date columns exempt: {sorted(_PLANNED) or 'none'}")
    for _name, _df in [('Signalen', signalen), ('Meldingen', meldingen), *extra_tabs.items()]:
        for _c in _df.columns:
            if _c in _PLANNED:
                continue
            if pd.api.types.is_datetime64_any_dtype(_df[_c]):
                _n = int((_df[_c] >= _MAX_VALID_DATE).sum())
                if _n:
                    print(f"  NOTE: {_name}.{_c}: {_n} date(s) after {_CUTOFF.date()} set to missing")
                    _df.loc[_df[_c] >= _MAX_VALID_DATE, _c] = pd.NaT

    # ── Analysis end date (ANALYSIS_END, setup cell): rows after it are left out of all analyses ──
    if globals().get('ANALYSIS_END') is not None:
        _end = pd.Timestamp(ANALYSIS_END).normalize()
        _hi = _end + pd.Timedelta(days=1)
        # Is the household-key change visible here? Carry-over = share of households with a signal in the
        # 90 days before a date that have a signal again in the 90 days after it.
        _sg = signalen.dropna(subset=['Hash', 'Datum melding'])
        def _carry(b):
            _A = set(_sg.loc[(_sg['Datum melding'] >= b - pd.Timedelta(days=90)) & (_sg['Datum melding'] < b), 'Hash'])
            _B = set(_sg.loc[(_sg['Datum melding'] >= b) & (_sg['Datum melding'] < b + pd.Timedelta(days=90)), 'Hash'])
            return (len(_A & _B) / len(_A)) if len(_A) >= 30 and len(_B) >= 30 else np.nan
        _c_end = _carry(_hi)
        _c_usual = pd.Series([_carry(_hi - pd.DateOffset(months=m)) for m in range(4, 16)]).median()
        if _c_end == _c_end:
            print(f"  Household carry-over across {_end.date()}: {_c_end*100:.0f}% of households seen in the 90 days "
                  f"before are seen again in the 90 days after (in the year before: usually {_c_usual*100:.0f}%)"
                  + (" - the household-key change is visible in this export" if _c_end < 0.5 * _c_usual else ''))
        else:
            print(f"  Household carry-over across {_end.date()}: not enough signals on both sides to check.")
        _n0 = {'Signalen': len(signalen), 'Meldingen': len(meldingen),
               **{r: len(d) for r, d in extra_tabs.items() if r in ('Tussenresultaat', 'Eindresultaat', 'Follow-Up resultaat')}}
        # a case belongs to the date of its first linked signal (else its first dated contact record)
        _fs = (signalen.dropna(subset=['Meldingnummer', 'Datum melding']).groupby('Meldingnummer')['Datum melding'].min()
               if 'Meldingnummer' in signalen.columns else pd.Series(dtype='datetime64[ns]'))
        _start = pd.to_datetime(meldingen['Meldingnummer'].map(_fs.to_dict()), errors='coerce')
        _fc = [d.dropna(subset=['Meldingnummer', 'Datum contactpoging']).groupby('Meldingnummer')['Datum contactpoging'].min()
               for r, d in extra_tabs.items() if r in ('Tussenresultaat', 'Eindresultaat')
               and {'Meldingnummer', 'Datum contactpoging'} <= set(d.columns)]
        if _fc:
            _start = _start.fillna(pd.to_datetime(meldingen['Meldingnummer'].map(
                pd.concat(_fc).groupby(level=0).min().to_dict()), errors='coerce'))
        _late_cases = set(meldingen.loc[_start >= _hi, 'Meldingnummer'])
        _sd = signalen['Datum melding']
        signalen = signalen[~((_sd.notna() & (_sd >= _hi)) | signalen['Meldingnummer'].isin(_late_cases))].reset_index(drop=True)
        meldingen = meldingen[~meldingen['Meldingnummer'].isin(_late_cases)].reset_index(drop=True)
        _open = meldingen['Afgerond d.d.'] >= _hi                 # closed after the end -> still open at the end
        meldingen.loc[_open, 'Afgerond d.d.'] = pd.NaT
        if 'Afgerond' in meldingen.columns:
            meldingen['Afgerond'] = meldingen['Afgerond'].astype(object)
            meldingen.loc[_open, 'Afgerond'] = 'Nee'
        for _r in [r for r in extra_tabs if r in ('Tussenresultaat', 'Eindresultaat', 'Follow-Up resultaat')]:
            _d = extra_tabs[_r]
            _keep = ~_d['Meldingnummer'].isin(_late_cases) if 'Meldingnummer' in _d.columns else pd.Series(True, index=_d.index)
            extra_tabs[_r] = _d[_keep].reset_index(drop=True)      # records of the remaining cases are all kept
        print(f"  ANALYSIS_END = {_end.date()}: rows after it are left out of all analyses "
              f"(cases closed later count as still open at that date: {int(_open.sum()):,})")
        for _r in _n0:
            print(f"    {_r:<22} {_n0[_r]:>8,} -> {(len(signalen) if _r == 'Signalen' else len(meldingen) if _r == 'Meldingen' else len(extra_tabs[_r])):>8,}")

    # ── Ja/Nee → boolean helper ───────────────────────────────────────────────────
    def ja_nee(series):
        return series.astype(str).str.strip().str.lower().map({'ja': True, 'nee': False})

    # ── Confirm tabs loaded (row counts only — no raw data) ───────────────────────
    all_tabs = [('Signalen', signalen), ('Meldingen', meldingen), ('Melders', melders),
                *extra_tabs.items()]
    for name, df in all_tabs:
        print(f"  {name:<35} {len(df):>7,} rows")

    # ── Rows per year, per tab (each counted on its own primary date column) ──────
    _DATECOL = {'Signalen': 'Datum melding', 'Meldingen': 'Afgerond d.d.',
                'Tussenresultaat': 'Datum contactpoging', 'Eindresultaat': 'Datum contactpoging',
                'Follow-Up resultaat': 'Datum'}
    _ycols = {}
    for name, df in all_tabs:
        _dc = _DATECOL.get(name)
        if _dc and _dc in df.columns and pd.api.types.is_datetime64_any_dtype(df[_dc]):
            _vc = df[_dc].dt.year.value_counts()
            if len(_vc):
                _ycols[name] = _vc
    if _ycols:
        _yt = pd.DataFrame(_ycols).fillna(0).astype(int).sort_index()
        _yt.index = _yt.index.astype(int)
        print("\n  Rows per year, per tab:")
        print(_yt.to_string())
        print("  (Signalen counted on signal date; Meldingen on close date; contact tabs on")
        print("   attempt date; Melders is a dateless registry. Rows with missing dates excluded.)")
    DATA_LOADED_RUN = _RUN_ID     # marks the data as loaded in THIS run
    _section_done('1a. Load Data')
except Exception as _err:
    _section_failed('1a. Load Data', _err)


### 1b. Shared Case Definitions & Data Checks

One place defines what every later section means by *contact method*, *reached*, *help
accepted* and *final result*, so all sections use the same rules:

- **Contact methods** are **outbound** (the municipality reaches out: letter, email, phone,
  WhatsApp/SMS, home visit, appointment), **inbound** (the client contacted the municipality —
  the label states what prompted it; "after email or letter" stays one combined category because
  the label combines them) or **administrative** (a result entered without a contact attempt).
- **Reached** = at least one attempt with `Persoon bereikt = Ja`, or an inbound contact.
  **Not reached** = no such evidence, and either a "no contact" final result or at least one
  attempt recorded as `Nee`. Everything else is **unknown**; a case with evidence of contact
  *and* a "no contact" final result is **conflicting**. Unknown and conflicting are never counted
  as "not reached".
- **Help**: accepted (`Ja`, follow-up appointment made, wants contact with VST — an appointment
  is not proof that help was received), declined (`Nee …`), hesitating, or unknown.
- **Final results** are mapped label by label. Cases closed as "invalid / not picked up" are left
  out of the effectiveness analyses; households already known to debt counselling or another
  team are kept apart from the contact comparison.
- **Time window:** only **dated** contact records between the case's **index signal** (its earliest
  linked signal) and its **closure** are used to classify reach, help and the final result.
  Records before the signal, after closure or without a date are kept and counted in an audit,
  together with the cases whose status — and, in §9, whose group — would change if they were used.
  Results entered after closing (some municipalities enter them later) still count when they are dated before the household's next case opened; a later result belongs to the new case, so it cannot move the old case into a group. 1b shows how many such records there are, how late, and which were used.

Every label and how it was mapped is printed with its count, so the mapping can be checked —
and corrected in this one cell. The cell also checks required columns (missing ones are added
empty), duplicate case and reporter numbers (exact copies are dropped; numbers that appear with
*conflicting* rows are quarantined — left out entirely), signals without a household and cases
linked to several households.

In [ ]:
_report_section('1b. Shared Case Definitions & Data Checks')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run

    # ── Required columns: report what is missing and add it empty (dates as empty dates) ──
    _REQUIRED = {'Signalen': (signalen, ['SignaalID', 'Hash', 'Datum melding', 'Meldingnummer', 'Melder ID', 'Doorgezet',
                                         'Recidive', 'Leeftijd', 'Minderjarige kinderen', 'Type melding']),
                 'Meldingen': (meldingen, ['Meldingnummer', 'Afgerond', 'Afgerond d.d.', 'Ingetrokken', 'Onterecht']),
                 'Melders': (melders, ['ID', 'Melder', 'Type melder'])}
    for _tab, (_df, _cols) in _REQUIRED.items():
        _miss = [c for c in _cols if c not in _df.columns]
        if _miss:
            _warn(f"{_tab} is missing column(s) {_miss} - added empty; sections that need them "
                  "will show little or nothing.")
            for _c in _miss:
                _df[_c] = (pd.Series(pd.NaT, index=_df.index, dtype='datetime64[ns]')
                           if _c in ('Datum melding', 'Afgerond d.d.') else np.nan)

    # ── Duplicate keys: exact copies are dropped; CONFLICTING rows (same key, different content)
    #    are quarantined - left out entirely, because nothing says which row is correct ──────────
    def _dedupe(df, key, label):
        k = df[key]
        dup = k.notna() & k.duplicated(keep=False)
        if not dup.any():
            return df, set()
        _versions = df[dup].astype(str).drop_duplicates()[key].value_counts()   # distinct row versions per key
        _conf = set(_versions[_versions > 1].index)
        _is_conf = k.notna() & k.astype(str).isin(_conf)
        _conf_vals = set(df.loc[_is_conf, key])
        out = df[~_is_conf]
        _exact = out[key].notna() & out[key].duplicated(keep='first')
        if int(_exact.sum()):
            print(f"  NOTE: {int(_exact.sum())} exact duplicate row(s) of the same {label} dropped.")
        if _conf:
            print(f"  NOTE: {len(_conf)} {label}(s) appear with CONFLICTING rows - all {int(_is_conf.sum())} "
                  "of their rows are left out (quarantined); nothing says which row is correct.")
        return out[~_exact].reset_index(drop=True), _conf_vals
    meldingen, QUARANTINED_CASES = _dedupe(meldingen, 'Meldingnummer', 'case number (Meldingen)')
    melders, QUARANTINED_REPORTERS = _dedupe(melders, 'ID', 'reporter ID (Melders)')
    all_tabs = [('Signalen', signalen), ('Meldingen', meldingen), ('Melders', melders), *extra_tabs.items()]

    print(f"  Signals without a household (Hash): {int(signalen['Hash'].isna().sum()):,}")
    _hpc0 = signalen.dropna(subset=['Meldingnummer', 'Hash']).groupby('Meldingnummer')['Hash'].nunique()
    AMBIGUOUS_CASES = set(_hpc0[_hpc0 > 1].index)
    print(f"  Cases linked to more than one household: {len(AMBIGUOUS_CASES):,} (left out of case analyses)")

    # ── Label mappings ─────────────────────────────────────────────────────────────
    def _norm(v):
        return ' '.join(str(v).strip().lower().split()) if pd.notna(v) else ''

    def method_kind(label):
        # -> (kind, channel); kind: outbound | inbound | admin | unknown
        s = _norm(label)
        if not s:
            return 'unknown', '(blank)'
        if s.startswith('klant nam contact op'):
            if 'flyer' in s:
                return 'inbound', 'after flyer'
            if 'brief' in s or 'mail' in s:
                return 'inbound', 'after letter/email'
            if 'telefo' in s or 'whatsapp' in s or 'sms' in s:
                return 'inbound', 'after phone/WhatsApp/SMS'
            return 'inbound', 'prompt not stated'
        if 'zonder contact' in s:
            return 'admin', 'result entered without contact attempt'
        if s.startswith('geen contact gehad'):          # Venlo: an outcome in the method field
            return 'unknown', 'no contact - method not recorded'
        if 'al onder begeleiding' in s:
            return 'admin', 'client already in support'
        for _key, _ch in [('huisbezoek', 'home visit'), ('afspraak op locatie', 'appointment at location'),
                          ('telefo', 'phone'), ('whatsapp', 'WhatsApp/SMS'), ('sms', 'WhatsApp/SMS'),
                          ('mail', 'email'), ('brief', 'letter'), ('consulent', 'other outreach')]:
            if _key in s:
                return 'outbound', _ch
        return 'unknown', s

    def result_class(label):
        s = _norm(label)
        if not s:
            return 'not recorded'
        if 'ongeldig' in s or 'niet opgepakt' in s:
            return 'invalid / not picked up'
        if 'geen contact' in s:
            return 'no contact'
        if 'niet betaald' in s or 'onbetaald' in s:
            return 'other (check mapping)'
        if 'betalingsregeling' in s:
            return 'payment arrangement'
        if 'betaald' in s:
            return 'paid'
        if 'zelf' in s and ('probeert' in s or 'lossen' in s):
            return 'self-resolving'
        if 'geen behoefte' in s:
            return 'no need for help'
        if 'quick fix' in s:
            return 'quick fix'
        if 'reeds bekend' in s or s.startswith('is bekend'):
            return 'already in care'
        if 'verwijzing' in s or 'aangemeld' in s or 'doorverwezen' in s:
            return 'referred / registered'
        return 'other (check mapping)'

    def help_status(label):
        s = _norm(label)
        if not s:
            return 'unknown'
        if s in {'ja', 'vervolg afspraak gemaakt', 'wenst contact met vst'}:
            return 'accepted'
        if s.startswith('nee'):
            return 'declined'
        if 'aarzelt' in s:
            return 'hesitating'
        return 'unknown'

    # ── Contact records (all attempt tabs) and each record's TIME WINDOW ──────────
    _cp = []
    for _tab in ['Tussenresultaat', 'Eindresultaat']:
        if _tab in extra_tabs and 'Meldingnummer' in extra_tabs[_tab].columns:
            _d = extra_tabs[_tab].copy(); _d['_tab'] = _tab; _cp.append(_d)
    contacts_all = pd.concat(_cp, ignore_index=True) if _cp else pd.DataFrame(columns=['Meldingnummer', '_tab'])
    for _c in ['Wijze van contactpoging', 'Persoon bereikt', 'Datum contactpoging',
               'Wil de klant hulp', 'Wat is het eindresultaat', 'Dagdeel']:
        if _c not in contacts_all.columns:
            contacts_all[_c] = np.nan
    contacts_all['Datum contactpoging'] = pd.to_datetime(contacts_all['Datum contactpoging'], errors='coerce')
    contacts_all[['method_kind', 'channel']] = pd.DataFrame(
        [method_kind(v) for v in contacts_all['Wijze van contactpoging']],
        columns=['method_kind', 'channel'], index=contacts_all.index)
    contacts_all['reached_flag'] = ja_nee(contacts_all['Persoon bereikt'])
    # window = from the case's index signal (earliest linked signal) to its closure. Only DATED records
    # inside it classify reach, help and final result; all other records are kept for the audit.
    _idx_sig = (signalen.dropna(subset=['Meldingnummer', 'Datum melding'])
                .groupby('Meldingnummer')['Datum melding'].min())
    _close_by_case = meldingen.dropna(subset=['Meldingnummer']).set_index('Meldingnummer')['Afgerond d.d.']
    # dict lookups: robust when a lookup table is empty or case numbers differ in number type
    contacts_all['_index'] = pd.to_datetime(contacts_all['Meldingnummer'].map(_idx_sig.to_dict()), errors='coerce')
    contacts_all['_close'] = pd.to_datetime(contacts_all['Meldingnummer'].map(_close_by_case.to_dict()), errors='coerce')
    _dtc = contacts_all['Datum contactpoging']
    # Records dated AFTER closing count when dated before the household's NEXT case opened (case open =
    # its first linked signal); a later record belongs to the new case. LATE_RECORD_DAYS (setup cell) =
    # optional extra limit in days (None = none; 0 = never use records after closing).
    _LATE = globals().get('LATE_RECORD_DAYS')
    _ch = (signalen.dropna(subset=['Meldingnummer', 'Hash']).drop_duplicates('Meldingnummer')
           .set_index('Meldingnummer')['Hash'])
    _co = pd.DataFrame({'Hash': _ch, 'open': _idx_sig.reindex(_ch.index)}).dropna().sort_values(['Hash', 'open'])
    _co['next_open'] = _co.groupby('Hash')['open'].shift(-1)          # the household's next case
    contacts_all['_next_open'] = pd.to_datetime(contacts_all['Meldingnummer'].map(_co['next_open'].to_dict()),
                                                errors='coerce')
    _after = contacts_all['_close'].notna() & (_dtc > contacts_all['_close'])
    _days_late = (_dtc - contacts_all['_close']).dt.days
    _ok_late = (_after & (contacts_all['_next_open'].isna() | (_dtc < contacts_all['_next_open']))
                & (True if _LATE is None else (_days_late <= int(_LATE))))
    _LATE_LABEL = 'after closure, before the next case (counted)'
    contacts_all['window'] = np.select(
        [_dtc.isna(), contacts_all['_index'].isna(), _dtc < contacts_all['_index'], _ok_late, _after],
        ['undated', 'case has no dated signal', 'before the signal', _LATE_LABEL, 'after closure'],
        default='in window')
    contacts_all['in_window'] = contacts_all['window'].isin(['in window', _LATE_LABEL])
    if _after.any():
        _lb = pd.cut(_days_late[_after], bins=[0, 7, 14, 30, 60, 180, 100000],
                     labels=['1-7', '8-14', '15-30', '31-60', '61-180', '>180'])
        _st = np.where(_ok_late[_after], 'counted', 'not used (next case already open'
                       + ('' if _LATE is None else f' or more than {_LATE} days') + ')')
        _lt = (pd.DataFrame({'days after closing': _lb, 'used': _st,
                             'Meldingnummer': contacts_all.loc[_after, 'Meldingnummer']})
               .groupby(['days after closing', 'used'], observed=False)['Meldingnummer']
               .agg(records='size', cases='nunique').unstack('used', fill_value=0))
        print("Contact records dated AFTER the case was closed - how late, and whether they count "
              "(they count when dated before the household's next case opened):")
        display(_lt)
        _cy = (contacts_all.loc[_after].assign(_yr=contacts_all.loc[_after, '_close'].dt.year)
               .groupby('_yr')['Meldingnummer'].nunique())
        _ny = pd.to_datetime(meldingen['Afgerond d.d.'], errors='coerce').dt.year.value_counts()
        print('Cases with records dated after closing, by close year (% of cases closed that year):')
        print((_cy / _ny.reindex(_cy.index) * 100).round(1).rename('% of closed cases').to_string())
    contacts_all['after_close'] = contacts_all['window'].eq('after closure')    # too late: not used

    def _last_nonblank(s):
        s = s.dropna().astype(str).str.strip()
        s = s[s != '']
        return s.iloc[-1] if len(s) else np.nan

    def _case_status(rows):
        # One row per case number in Meldingen, classified from the GIVEN contact records only.
        r = rows.copy()
        r['_ja'] = r['reached_flag'] == True
        r['_in'] = r['method_kind'] == 'inbound'
        r['_out'] = r['method_kind'] == 'outbound'
        r['_rec'] = r['reached_flag'].notna()
        out = pd.DataFrame(index=pd.Index(meldingen['Meldingnummer'].dropna().unique(), name='Meldingnummer'))
        out = out.join(r.groupby('Meldingnummer').agg(n_records=('_tab', 'size'), n_outbound=('_out', 'sum'),
                                                      any_ja=('_ja', 'max'), any_inbound=('_in', 'max'),
                                                      any_reach_recorded=('_rec', 'max')))
        for c in ['n_records', 'n_outbound', 'any_ja', 'any_inbound', 'any_reach_recorded']:
            if c not in out.columns:
                out[c] = np.nan
        for c in ['n_records', 'n_outbound']:
            out[c] = pd.to_numeric(out[c], errors='coerce').fillna(0).astype(int)
        for c in ['any_ja', 'any_inbound', 'any_reach_recorded']:
            out[c] = out[c].eq(True)
        succ = r[r['_ja'] | r['_in']].sort_values('Datum contactpoging', na_position='last', kind='stable')
        fs = succ.drop_duplicates('Meldingnummer', keep='first').set_index('Meldingnummer')
        out['first_reach_date'] = fs['Datum contactpoging']
        out['reach_channel'] = pd.Series(
            np.select([fs['method_kind'].eq('inbound'), fs['method_kind'].eq('outbound')],
                      ['callback ' + fs['channel'].astype(str), fs['channel'].astype(str)],
                      default='(' + fs['method_kind'].astype(str) + ' entry)'), index=fs.index, dtype=object)
        # final word = last non-blank value; undated rows sort first, so a dated value wins
        e = r[r['_tab'] == 'Eindresultaat'].sort_values('Datum contactpoging', na_position='first', kind='stable')
        eg = e.groupby('Meldingnummer')
        out['help_label'] = eg['Wil de klant hulp'].agg(_last_nonblank) if len(e) else np.nan
        out['result_label'] = eg['Wat is het eindresultaat'].agg(_last_nonblank) if len(e) else np.nan
        eh = e[e['Wil de klant hulp'].notna() & (e['Wil de klant hulp'].astype(str).str.strip() != '')]
        out['help_date'] = eh.drop_duplicates('Meldingnummer', keep='last').set_index('Meldingnummer')['Datum contactpoging']
        out['result_class'] = [result_class(v) for v in out['result_label']]
        out['help_status'] = [help_status(v) for v in out['help_label']]
        evid = out['any_ja'] | out['any_inbound']
        noc = out['result_class'].eq('no contact')
        out['reach_status'] = np.select([evid & noc, evid, noc | out['any_reach_recorded']],
                                        ['conflicting', 'reached', 'not reached'], default='unknown')
        return out

    CASES = _case_status(contacts_all[contacts_all['in_window']])     # used everywhere
    _ALLREC = _case_status(contacts_all)                              # same rules, ALL records - audit only
    for _c in ['reach_status', 'help_status', 'result_class']:
        CASES[_c + '_allrec'] = _ALLREC[_c]
    CASES['n_attempts'] = CASES['n_records']                          # dated contact records in the window
    _wc = contacts_all.groupby(['Meldingnummer', 'window']).size().unstack(fill_value=0)
    CASES['n_after_close'] = (_wc['after closure'].reindex(CASES.index).fillna(0).astype(int)
                              if 'after closure' in _wc.columns else 0)

    # ── Audit tables ───────────────────────────────────────────────────────────────
    if len(contacts_all):
        print('\nContact records by time window (only "in window" records classify reach, help and final result):')
        display(pd.crosstab(contacts_all['_tab'], contacts_all['window'], margins=True, margins_name='total'))
    print('Contact methods - how each label is classified (attempt rows):')
    display(contacts_all.assign(label=contacts_all['Wijze van contactpoging'].fillna('(blank)'))
            .groupby(['method_kind', 'channel', 'label']).size().rename('rows').reset_index()
            .sort_values(['method_kind', 'rows'], ascending=[True, False]).reset_index(drop=True))
    print('Final results - how each label is classified (cases; records in the window):')
    display(CASES.assign(label=CASES['result_label'].fillna('(not recorded)'))
            .groupby(['result_class', 'label']).size().rename('cases').reset_index()
            .sort_values(['result_class', 'cases'], ascending=[True, False]).reset_index(drop=True))
    print('Help answers - how each label is classified (cases; records in the window):')
    display(CASES.assign(label=CASES['help_label'].fillna('(not recorded)'))
            .groupby(['help_status', 'label']).size().rename('cases').reset_index()
            .sort_values(['help_status', 'cases'], ascending=[True, False]).reset_index(drop=True))
    _unm = contacts_all.loc[contacts_all['method_kind'].eq('unknown') & contacts_all['Wijze van contactpoging'].notna()
                            & contacts_all['channel'].ne('no contact - method not recorded'),
                            'Wijze van contactpoging'].unique()
    if len(_unm):
        _warn(f"unmapped contact-method label(s): {list(_unm)[:10]} - add them to method_kind().")
    if CASES['result_class'].eq('other (check mapping)').any():
        _warn("final-result label(s) mapped to 'other (check mapping)' - see the table in 1b.")
    _rs0 = CASES['reach_status'].value_counts().reindex(['reached', 'not reached', 'unknown', 'conflicting'], fill_value=0)
    print('\nReach status of all cases (shared definition, records in the window):')
    print(pd.DataFrame({'cases': _rs0, '%': (_rs0 / max(len(CASES), 1) * 100).round(1)}).to_string())
    if len(CASES):
        print('\nEffect of the time window on case status (rows = using ALL records, columns = window only):')
        display(pd.crosstab(CASES['reach_status_allrec'], CASES['reach_status'], margins=True, margins_name='total'))
        display(pd.crosstab(CASES['help_status_allrec'], CASES['help_status'], margins=True, margins_name='total'))
        print('  The effect on the contact GROUPS used in the comparisons is shown in section 9.')
    _section_done('1b. Shared Case Definitions & Data Checks')
except Exception as _err:
    _section_failed('1b. Shared Case Definitions & Data Checks', _err)


## 2. Dataset Profile

The real export is seen only once, and only the HTML report comes back. This section therefore
records a profile of the dataset — schema, exact category labels, value ranges,
date coverage and join quality — so the next version of the analysis can be written without
needing the raw data again. Category labels are listed for every column except those on a block
list of identifier and free-text columns (`SENSITIVE_COLS`).


### 2a. Schema

For each tab: columns, data types, missing-value percentage, and number of distinct values.


In [ ]:
_report_section('2a. Schema — tabs, columns, dtypes, missingness, cardinality')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    # Columns that must never have their raw values shown (identifiers / personal / free text)
    SENSITIVE_COLS = {
        'Hash', 'SignaalID', 'Meldingnummer', 'ID', 'Melder ID', 'Postcode',
        'Naam medewerker', 'Melder', 'Melder (submerk)', 'Onderwerp',
        'Toelichting bij de FollowUp', 'Verwijzing anders namelijk', 'Waarnaar is verwezen',
    }
    CATEGORICAL_MAX = 50   # only list distinct values for columns with at most this many

    def schema_table(df):
        return pd.DataFrame({
            'dtype':    df.dtypes.astype(str),
            'non_null': df.count(),
            'null_%':   (df.isnull().sum() / max(len(df), 1) * 100).round(1),
            'n_unique': df.nunique(dropna=True),
        })

    for name, df in all_tabs:
        print(f"\n══ {name}: {len(df):,} rows × {df.shape[1]} cols ══")
        display(schema_table(df))
    _section_done('2a. Schema — tabs, columns, dtypes, missingness, cardinality')
except Exception as _err:
    _section_failed('2a. Schema — tabs, columns, dtypes, missingness, cardinality', _err)


### 2b. Category Labels & Frequencies

The exact distinct values and their counts for every low-cardinality, non-identifier column —
pins down the precise spelling of contact methods, outcomes and Ja/Nee tokens for future code.


In [ ]:
_report_section('2b. Exact Category Labels & Frequencies')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    for name, df in all_tabs:
        shown_any = False
        for col in df.columns:
            if col in SENSITIVE_COLS:
                continue
            nu = df[col].nunique(dropna=True)
            if nu < 1 or nu > 1000:
                continue
            if not shown_any:
                print(f"\n══ {name} ══")
                shown_any = True
            if nu <= CATEGORICAL_MAX:
                vc = df[col].value_counts(dropna=False)
                vc.index = vc.index.map(lambda x: '(blank)' if pd.isna(x) else str(x))
                print(f"\n• {col}  ({nu} distinct):")
                print(vc.to_string())
            else:
                # 50 < distinct <= 1000: list only the most common values
                vc = df[col].value_counts().head(15)
                print(f"\n• {col}  ({nu} distinct — showing top 15):")
                print(vc.to_string())
    _section_done('2b. Exact Category Labels & Frequencies')
except Exception as _err:
    _section_failed('2b. Exact Category Labels & Frequencies', _err)


### 2c. Numeric Ranges & Date Coverage

Distribution of numeric columns (to design realistic bands) and, for each date column,
how well it parsed and what period it covers.


In [ ]:
_report_section('2c. Numeric Ranges & Date Coverage')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    for name, df in all_tabs:
        num = df.select_dtypes(include='number')
        num = num[[c for c in num.columns if c not in SENSITIVE_COLS]]
        if num.shape[1]:
            print(f"\n══ {name} — numeric columns ══")
            display(num.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))

    print("\n══ Date columns — coverage & parse success ══")
    date_cols = {'Signalen': ['Datum melding', 'Datum doorgezet', 'Startdatum contact'],
                 'Meldingen': ['Afgerond d.d.', 'Gezien d.d.']}
    tab_lookup = dict(all_tabs)
    for tab, cols in date_cols.items():
        if tab not in tab_lookup:
            continue
        df = tab_lookup[tab]
        for c in cols:
            if c in df.columns and pd.api.types.is_datetime64_any_dtype(df[c]):
                s = df[c]
                ok = s.notna().mean() * 100
                print(f"  {tab}.{c:<20} parsed={ok:5.1f}%  "
                      f"range={s.min().date() if s.notna().any() else 'n/a'} → "
                      f"{s.max().date() if s.notna().any() else 'n/a'}")

    print("\n══ Raw date formats (original text, before parsing) ══")
    if RAW_DATE_SAMPLES:
        for _col, _ex in RAW_DATE_SAMPLES.items():
            print(f"  {_col:<22} e.g. {_ex}")
    else:
        print("  All date columns were already proper date cells (no text parsing needed).")
    _section_done('2c. Numeric Ranges & Date Coverage')
except Exception as _err:
    _section_failed('2c. Numeric Ranges & Date Coverage', _err)


### 2d. Join Integrity & Structure

Checks how the tabs link together (orphan and duplicate rates) and the structures that decide
what is analysable: signals per household, contact attempts per case, and how many closed cases
have enough follow-up time.


In [ ]:
_report_section('2d. Join Integrity & Structural Distributions')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    print("── Join integrity ──")
    sig_with_mld = signalen['Meldingnummer'].notna().mean() * 100
    print(f"  Signalen with a Meldingnummer (linkable to Meldingen): {sig_with_mld:.1f}%")

    mld_keys = set(meldingen['Meldingnummer'].dropna())
    for tab in ['Tussenresultaat', 'Eindresultaat', 'Follow-Up resultaat']:
        if tab in extra_tabs and 'Meldingnummer' in extra_tabs[tab].columns:
            ids = extra_tabs[tab]['Meldingnummer'].dropna()
            match = ids.isin(mld_keys).mean() * 100 if len(ids) else 0
            print(f"  {tab}: {len(ids):,} rows, {match:.1f}% match a Meldingen case")

    if 'Melder ID' in signalen.columns:
        melder_ids = set(melders['ID'].dropna())
        m = signalen['Melder ID'].dropna()
        print(f"  Signalen.Melder ID matching Melders registry: "
              f"{m.isin(melder_ids).mean()*100:.1f}%")

    dup_mld = int(meldingen['Meldingnummer'].duplicated().sum())
    print(f"  Duplicate Meldingnummer in Meldingen: {dup_mld:,}")

    print("\n── Recidive structure (signals per household Hash) ──")
    sph = signalen.dropna(subset=['Hash']).groupby('Hash').size()
    if len(sph):
        print(f"  Households (unique Hash): {len(sph):,}")
        print(f"  Signals per household: mean={sph.mean():.2f}  median={sph.median():.0f}  p99={sph.quantile(.99):.0f}")
        multi = (sph > 1).mean() * 100
        print(f"  Households with >1 signal (recidive present): {multi:.1f}%")

    print("\n── Contact attempts per case ──")
    att = []
    for tab in ['Tussenresultaat', 'Eindresultaat']:
        if tab in extra_tabs:
            att.append(extra_tabs[tab][['Meldingnummer']])
    if att:
        apc = pd.concat(att).dropna().groupby('Meldingnummer').size()
        print(f"  Cases with ≥1 attempt: {len(apc):,}")
        print(f"  Attempts per case: mean={apc.mean():.2f}  median={apc.median():.0f}  p99={apc.quantile(.99):.0f}")

    print("\n── Follow-up availability for recidive windows ──")
    if 'Afgerond d.d.' in meldingen.columns and pd.api.types.is_datetime64_any_dtype(meldingen['Afgerond d.d.']):
        dmax = signalen['Datum melding'].max()
        closed_ct = ja_nee(meldingen['Afgerond'])
        for label, days in {'6 months': 182, '9 months': 273}.items():
            ok = (closed_ct.fillna(False) &
                  meldingen['Afgerond d.d.'].notna() &
                  ((meldingen['Afgerond d.d.'] + pd.Timedelta(days=days)) <= dmax)).sum()
            print(f"  Closed cases with ≥{label} follow-up: {ok:,}")

    print("\n── Cases per municipality (multilevel feasibility) ──")
    if 'Gemeente' in meldingen.columns:
        cpm = meldingen['Gemeente'].value_counts()
        print(f"  Municipalities: {cpm.size}  |  cases/municipality "
              f"mean={cpm.mean():.0f} median={cpm.median():.0f} min={cpm.min()} max={cpm.max()}")

    print("\n── One household per case? (Meldingnummer -> Hash) ──")
    _sl  = signalen.dropna(subset=['Meldingnummer', 'Hash'])
    _hpc = _sl.groupby('Meldingnummer')['Hash'].nunique()
    _multi = int((_hpc > 1).sum())
    print(f"  Cases with linked signals: {len(_hpc):,}")
    print(f"  ...mapping to >1 distinct household Hash: {_multi:,} "
          f"({_multi / max(len(_hpc), 1) * 100:.2f}%)")
    print("  OK - one household per case; case-level recidive logic holds." if _multi == 0
          else "  WARNING - some cases span multiple households; revisit the one-Hash-per-case assumption.")

    print("\n── Signalen -> Meldingen referential integrity ──")
    _sig_mld = signalen['Meldingnummer'].dropna()
    _orphan  = int((~_sig_mld.isin(mld_keys)).sum())
    print(f"  Signals carrying a Meldingnummer: {len(_sig_mld):,}")
    print(f"  ...whose Meldingnummer is NOT found in Meldingen (orphans): {_orphan:,} "
          f"({_orphan / max(len(_sig_mld), 1) * 100:.2f}%)")

    _j = signalen.dropna(subset=['Datum melding']).copy()
    _j['_yr'] = _j['Datum melding'].dt.year
    if _j['_yr'].nunique() > 1:
        _jt = _j.groupby('_yr').agg(signals=('Meldingnummer', 'size'),
                                    linkable=('Meldingnummer', lambda s: s.notna().sum()))
        _jt['linkable_%'] = (_jt['linkable'] / _jt['signals'] * 100).round(1)
        print("\n── Linkable signals per year (data-quality evolution) ──")
        print(_jt.to_string())
    _section_done('2d. Join Integrity & Structural Distributions')
except Exception as _err:
    _section_failed('2d. Join Integrity & Structural Distributions', _err)


### 2e. R0 — What does the system `Recidive` flag mean?

Verifies the semantics of the system-provided recurrence columns before anything relies on
them. **Step 1:** is `Recidive` exactly the union of `Recidive maand-1..6`? (agreement, kappa,
confusion matrix, mismatch-by-year for the 2021 boundary). **Step 2:** which rule generates the
flags — reconstructions from observed history for every candidate: {any reporter, same reporter
*type*, same individual reporter (`Melder ID`)} × {rolling window vs calendar month}, scored per
flag by agreement **and** by sensitivity/precision for the `Ja` class (most flags are `Nee`, so
agreement alone flatters a weak rule). The best-matching rule is a working hypothesis to confirm
with XXLLNC; §7a/§11 and `recidive_hist` are read accordingly. `Schuiver`/`Match` are
noted but out of scope.


In [ ]:
_report_section('2e. R0 - What does the system Recidive flag mean?')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _mcols = [c for c in signalen.columns if c.startswith('Recidive maand')]
    if 'Recidive' not in signalen.columns or not _mcols:
        print('Recidive columns not present in this export - R0 skipped.')
    else:
        _sys = ja_nee(signalen['Recidive']).fillna(False)
        _union = pd.Series(False, index=signalen.index)
        for _c in _mcols:
            _union = _union | ja_nee(signalen[_c]).fillna(False)
        n = len(signalen)
        tp = int((_sys & _union).sum());  tn = int((~_sys & ~_union).sum())
        fp = int((~_sys & _union).sum()); fn = int((_sys & ~_union).sum())
        agree = (tp + tn) / n * 100
        _po = (tp + tn) / n
        _pe = (_sys.mean() * _union.mean()) + ((1 - _sys.mean()) * (1 - _union.mean()))
        kappa = (_po - _pe) / (1 - _pe) if _pe < 1 else float('nan')

        print(f"System Recidive = Ja:        {int(_sys.sum()):,} ({_sys.mean()*100:.1f}%)")
        print(f"Union(maand-1..{len(_mcols)}) = Ja:     {int(_union.sum()):,} ({_union.mean()*100:.1f}%)")
        print(f"\nExact agreement: {agree:.2f}%   Cohen's kappa: {kappa:.3f}   (n={n:,})")
        print("\nConfusion matrix (rows = system Recidive, cols = union of maand flags):")
        print(pd.DataFrame([[tn, fp], [fn, tp]],
                           index=['system Nee', 'system Ja'],
                           columns=['union Nee', 'union Ja']).to_string())

        if agree >= 99.5:
            print("\nCONCLUSION (step 1): Recidive is (near-)exactly the union of the maand flags ->")
            print("a 6-MONTH WINDOWED recurrence flag. Read 7a/'11 targeting as 'repeats within")
            print("~6 months'; full history is controlled via prior_band.")
        elif agree >= 95:
            print("\nCONCLUSION (step 1): Recidive largely matches the union of the maand flags;")
            print("the remainder points to edge rules (window boundaries / same-day ties).")
        else:
            print("\nCONCLUSION (step 1): Recidive does NOT match the union of the maand flags -")
            print("it follows another rule; step 2 below tries to identify it.")

        _yr = signalen['Datum melding'].dt.year
        _mism = (_sys != _union)
        if _mism.any() and _yr.notna().any():
            _tb = pd.DataFrame({'year': _yr, 'mismatch': _mism}).dropna()
            print("\nMismatch rate by signal year (union vs system):")
            print(_tb.groupby('year')['mismatch']
                     .agg(n='size', mismatch_pct=lambda s: round(s.mean()*100, 2)).to_string())

        # ── Step 2: WHICH rule generates the flags? Reconstruct candidates ─────────
        # Candidates per flag: {any reporter | same reporter TYPE | same reporter (Melder ID)}
        #                      x {rolling window | calendar month}
        import bisect as _bi
        _sh = signalen.dropna(subset=['Hash', 'Datum melding']).copy()
        if 'Melder ID' in _sh.columns and 'Type melder' in melders.columns:
            _tymap = dict(zip(melders['ID'], melders['Type melder']))
            _sh['_ty'] = _sh['Melder ID'].map(_tymap)
        else:
            _sh['_ty'] = 'onbekend'
        _sh['_mo'] = _sh['Datum melding'].dt.year * 12 + _sh['Datum melding'].dt.month
        _sh['_rep'] = _sh['Melder ID'] if 'Melder ID' in _sh.columns else 'onbekend'

        _shs = _sh.sort_values('Datum melding')
        _hist    = _shs.groupby('Hash')['Datum melding'].apply(list).to_dict()
        _hist_ty = _shs.groupby(['Hash', '_ty'])['Datum melding'].apply(list).to_dict()
        _mons    = _sh.groupby('Hash')['_mo'].agg(set).to_dict()
        _mons_ty = _sh.groupby(['Hash', '_ty'])['_mo'].agg(set).to_dict()
        _hist_rep = _shs.groupby(['Hash', '_rep'])['Datum melding'].apply(list).to_dict()
        _mons_rep = _sh.groupby(['Hash', '_rep'])['_mo'].agg(set).to_dict()

        def _roll(arr, t, lo_d, hi_d):
            if not arr or pd.isna(t): return False
            lo = t - pd.Timedelta(days=hi_d); hi = t - pd.Timedelta(days=lo_d)
            return _bi.bisect_right(arr, hi) - _bi.bisect_left(arr, lo) > 0
        def _cal(ms, mo, k1, k2):
            if not ms: return False
            return any((mo - k) in ms for k in range(k1, k2 + 1))

        _H  = list(zip(_sh['Hash'], _sh['_ty'], _sh['Datum melding'], _sh['_mo'], _sh['_rep']))
        _cand = {}
        _cand[('Recidive', 'any/rolling 1-186d')]     = [_roll(_hist.get(h), t, 1, 186) for h, ty, t, mo, rp in _H]
        _cand[('Recidive', 'any/calendar M-1..6')]    = [_cal(_mons.get(h), mo, 1, 6) for h, ty, t, mo, rp in _H]
        _cand[('Recidive', 'same-type/rolling')]      = [_roll(_hist_ty.get((h, ty)), t, 1, 186) for h, ty, t, mo, rp in _H]
        _cand[('Recidive', 'same-type/calendar')]     = [_cal(_mons_ty.get((h, ty)), mo, 1, 6) for h, ty, t, mo, rp in _H]
        _cand[('Recidive', 'same-reporter/rolling')]  = [_roll(_hist_rep.get((h, rp)), t, 1, 186) for h, ty, t, mo, rp in _H]
        _cand[('Recidive', 'same-reporter/calendar')] = [_cal(_mons_rep.get((h, rp)), mo, 1, 6) for h, ty, t, mo, rp in _H]
        for _k in (1, 2, 3):
            _lab = f'maand-{_k}'
            _cand[(_lab, f'any/rolling {(_k-1)*30+1}-{_k*30+7}d')] = \
                [_roll(_hist.get(h), t, (_k-1)*30+1, _k*30+7) for h, ty, t, mo, rp in _H]
            _cand[(_lab, f'any/calendar M-{_k}')]  = [_cal(_mons.get(h), mo, _k, _k) for h, ty, t, mo, rp in _H]
            _cand[(_lab, f'same-type/calendar M-{_k}')] = [_cal(_mons_ty.get((h, ty)), mo, _k, _k) for h, ty, t, mo, rp in _H]
            _cand[(_lab, f'same-reporter/calendar M-{_k}')] = \
                [_cal(_mons_rep.get((h, rp)), mo, _k, _k) for h, ty, t, mo, rp in _H]

        _sysflags = {'Recidive': _sys.loc[_sh.index]}
        for _k in (1, 2, 3):
            _cn = f'Recidive maand-{_k}'
            if _cn in signalen.columns:
                _sysflags[f'maand-{_k}'] = ja_nee(signalen[_cn]).fillna(False).loc[_sh.index]

        _obs0 = _sh['Datum melding'].min()
        _mask = (_sh['Datum melding'] >= _obs0 + pd.Timedelta(days=186)).values
        if int(_mask.sum()) >= 200:
            print(f"\nStep 2 - which rule generates each flag? (agreement %, signals from "
                  f"{(_obs0 + pd.Timedelta(days=186)).date()} on; first 6 months excluded)")
            rows = []
            for (_flag, _rule), _vals in _cand.items():
                if _flag not in _sysflags: continue
                _sv = np.asarray(_sysflags[_flag].values[_mask], dtype=bool)
                _rv = np.asarray(_vals, dtype=bool)[_mask]
                _tp = int((_sv & _rv).sum())
                rows.append((_flag, _rule, round((_sv == _rv).mean()*100, 1),
                             round(_tp / max(int(_sv.sum()), 1) * 100, 1), round(_tp / max(int(_rv.sum()), 1) * 100, 1),
                             round(_sv.mean()*100, 1), round(_rv.mean()*100, 1)))
            _rt = pd.DataFrame(rows, columns=['system flag', 'candidate rule', 'agreement %',
                                              'sensitivity % (system Ja found)', 'precision % (rule Ja correct)',
                                              'system Ja %', 'reconstructed Ja %'])
            display(_rt)
            print("Best-matching rule per flag (by agreement - check sensitivity and precision as well):")
            for _flag, g in _rt.groupby('system flag'):
                _b = g.sort_values('agreement %', ascending=False).iloc[0]
                print(f"  {_flag:<10} -> {_b['candidate rule']:<30} (agreement {_b['agreement %']}%, "
                      f"sensitivity {_b['sensitivity % (system Ja found)']}%, "
                      f"precision {_b['precision % (rule Ja correct)']}%)")
            print("\nRead: most flags are Nee, so agreement alone can look high for a weak rule - a rule is a")
            print("likely definition only when agreement, sensitivity AND precision are all high.")
            print("'same-type' = same reporter TYPE (e.g. any energy supplier); 'same-reporter' = same Melder ID.")
            print("Confirm the definition with XXLLNC before relying on it.")
        else:
            print("\nStep 2 skipped - too few signals after the 6-month burn-in window.")

        # footnote: sibling system flags, out of R0 scope
        for _fc in ['Schuiver', 'Match']:
            if _fc in signalen.columns:
                _fv = ja_nee(signalen[_fc]).fillna(False)
                print(f"Note: system flag '{_fc}' = Ja for {_fv.mean()*100:.1f}% - not analysed here (out of scope).")
    _section_done('2e. R0 - What does the system Recidive flag mean?')
except Exception as _err:
    _section_failed('2e. R0 - What does the system Recidive flag mean?', _err)


### 2f. R1 — Recidive Fingerprint Census (what the intake screen shows and hides)

The six maand flags are the household-history "fingerprint" a caseworker sees at intake.
This census describes the incoming stream — how many signals arrive as new households,
freshly recurring, building, **saturated (all six Ja: the flag's ceiling)**, intermittent, or
**fading (about to reset to "newcomer")** — and quantifies the flag's two blind spots against
our own full observed history: **false newcomers** (flag = Nee but real history older than
6 months) and **saturation**. The bridge table gives the honest correction between the
system's ~6-month repeat rate and the true ever-repeat rate (relevant for §7a/§11).


In [ ]:
_report_section('2f. R1 - Recidive Fingerprint Census')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _mcols = [c for c in signalen.columns if c.startswith('Recidive maand')]
    if len(_mcols) < 6 or 'Recidive' not in signalen.columns:
        print('Recidive maand columns not present - R1 skipped.')
    else:
        _mcols = sorted(_mcols, key=lambda c: int(c.split('-')[-1]))
        _F = pd.DataFrame({c: (ja_nee(signalen[c]) == True) for c in _mcols})
        _depth = _F.sum(axis=1)
        _m1 = _F[_mcols[0]]
        _sysR = ja_nee(signalen['Recidive']) == True
        _yr = signalen['Datum melding'].dt.year

        print('Depth at arrival - number of maand-1..6 flags = Ja on the incoming signal:')
        _dd = _depth.value_counts().sort_index()
        print(pd.DataFrame({'signals': _dd, '%': (_dd / len(signalen) * 100).round(1)}).to_string())

        # ── fingerprint pattern (exhaustive & disjoint) ────────────────────────────
        _alive = pd.Series(True, index=signalen.index)
        _streak = pd.Series(0, index=signalen.index)
        for _c in _mcols:
            _alive = _alive & _F[_c]
            _streak = _streak + _alive.astype(int)
        _consec = _streak == _depth      # all Ja's form one unbroken run starting at maand-1

        pattern = pd.Series('', index=signalen.index, dtype=object)
        pattern[_depth == 0] = '0. no recent history (0/6)'
        pattern[(_depth >= 1) & (~_m1)] = '4. fading (gap since last signal)'
        pattern[(_depth == 1) & _m1] = '1. starting (maand-1 only)'
        pattern[_depth.between(2, 5) & _m1 & _consec] = '2. building (consecutive run)'
        pattern[_depth.between(2, 5) & _m1 & (~_consec)] = '3. intermittent (on-off rhythm)'
        pattern[_depth == 6] = '5. saturated (6/6 - ceiling)'

        print('\nFingerprint pattern of incoming signals:')
        _pc = pattern.value_counts().sort_index()
        print(pd.DataFrame({'signals': _pc, '%': (_pc / len(signalen) * 100).round(1)}).to_string())
        print("'fading' = will look like a NEWCOMER next month | 'saturated' = flag cannot")
        print("distinguish month-7 trouble from year-4 chronic | 'intermittent' often reflects")
        print('bi-monthly creditor reporting rhythm (see H2).')

        if _yr.nunique() > 1:
            _tmp = pd.DataFrame({'y': _yr, 'p': pattern}).dropna(subset=['y'])
            _pt = pd.crosstab(_tmp['y'].astype(int), _tmp['p'], normalize='index') * 100
            print('\nPattern mix per year (%):')
            print(_pt.round(1).to_string())

        # ── blind spots, measured against our own full observed history ───────────
        import bisect as _bi
        _GAP = 14   # same re-reporting convention as MIN_GAP_DAYS in section 9
        _sh = signalen.dropna(subset=['Hash', 'Datum melding'])
        _hist = _sh.sort_values('Datum melding').groupby('Hash')['Datum melding'].apply(list).to_dict()
        def _n_prior(h, t, cutoff_days):
            arr = _hist.get(h)
            if not arr or pd.isna(t):
                return 0
            return _bi.bisect_left(arr, t - pd.Timedelta(days=cutoff_days))
        _has_old = pd.Series([_n_prior(h, t, 186) > 0
                              for h, t in zip(_sh['Hash'], _sh['Datum melding'])], index=_sh.index)
        _has_any = pd.Series([_n_prior(h, t, _GAP) > 0
                              for h, t in zip(_sh['Hash'], _sh['Datum melding'])], index=_sh.index)
        _sysR2 = _sysR.loc[_sh.index]
        _sat2  = (_depth.loc[_sh.index] == 6)
        _obs0  = _sh['Datum melding'].min()
        _mask  = _sh['Datum melding'] >= _obs0 + pd.Timedelta(days=186)   # burn-in: history invisible before

        if int(_mask.sum()) >= 200:
            _mNee = _mask & (~_sysR2)
            _fn = (~_sysR2) & _has_old & _mask
            print('\nBlind spot 1 - FALSE NEWCOMERS (flag = Nee, but observed history older than 6 months):')
            print(f"  {int(_fn.sum()):,} of {int(_mNee.sum()):,} flag-Nee signals "
                  f"({int(_fn.sum()) / max(int(_mNee.sum()), 1) * 100:.1f}%) - the 6-month window has")
            print('  reset on households that are in fact repeat households.')
            print('\nBlind spot 2 - SATURATION (all six flags Ja):')
            print(f"  {int((_sat2 & _mask).sum()):,} of {int(_mask.sum()):,} signals "
                  f"({(_sat2 & _mask).sum() / max(int(_mask.sum()), 1) * 100:.1f}%) arrive at the flag's ceiling.")

            _bt = pd.DataFrame({'year': _yr.loc[_sh.index][_mask],
                                'flag Ja %': _sysR2[_mask].astype(float),
                                'observed ever-repeat %': _has_any[_mask].astype(float)}).dropna(subset=['year'])
            if _bt['year'].nunique() > 1:
                _g = (_bt.assign(year=_bt['year'].astype(int)).groupby('year').mean() * 100).round(1)
                _g['undercount (pp)'] = (_g['observed ever-repeat %'] - _g['flag Ja %']).round(1)
                print('\nBridge: system flag vs full-history repeat rate (after 6-month burn-in):')
                print(_g.to_string())
            else:
                print(f"\nBridge (after burn-in): flag Ja = {_sysR2[_mask].mean()*100:.1f}%  vs  "
                      f"observed ever-repeat = {_has_any[_mask].mean()*100:.1f}%")
            print('Use the observed column when the question is "was this household seen before";')
            print('the flag answers only "active in the last ~6 months" - read section 11 accordingly.')
        else:
            print('\nBlind-spot analysis skipped - too few signals after the 6-month burn-in window.')
    _section_done('2f. R1 - Recidive Fingerprint Census')
except Exception as _err:
    _section_failed('2f. R1 - Recidive Fingerprint Census', _err)


## 3. Signal Volume & Distribution

How many signals arrive per month, and which signal types are most common.


In [ ]:
_report_section('3. Signal Volume & Distribution')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    # Use only valid dates; convert to monthly Timestamps for a clean, fast x-axis.
    # (Garbage/NaT dates are dropped so a few bad rows can't blow up the plot range.)
    _valid_dates = signalen['Datum melding'].dropna()
    monthly = (_valid_dates.dt.to_period('M').dt.to_timestamp()
               .value_counts().sort_index().rename('signals'))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if len(monthly) > 0:
        monthly.plot(ax=axes[0], marker='o', color=COLORS[0], linewidth=2)
    axes[0].set_title('Signal Volume per Month')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Number of signals')

    type_counts = signalen['Type melding'].value_counts()
    type_counts.plot.barh(ax=axes[1], color=COLORS[1])
    axes[1].set_title('Signal Type (Type melding)')
    axes[1].set_xlabel('Count')

    plt.tight_layout()
    plt.show()

    print(f"Total signals      : {len(signalen):,}")
    print(f"Valid signal dates : {len(_valid_dates):,} "
          f"({len(_valid_dates)/max(len(signalen),1)*100:.1f}%)")
    if len(_valid_dates) > 0:
        print(f"Date range         : {_valid_dates.min().date()} → {_valid_dates.max().date()}")
    else:
        print("Date range         : no parseable dates in 'Datum melding'")
    print()
    print("Signal type counts:")
    print(type_counts.to_string())
    print()
    print("Signals per month (values behind the monthly volume chart):")
    print(monthly.to_string() if len(monthly) else "(no parseable dates)")
    print()
    print("Signals per year:")
    print(_valid_dates.dt.year.value_counts().sort_index().rename('signals').to_string())
    _section_done('3. Signal Volume & Distribution')
except Exception as _err:
    _section_failed('3. Signal Volume & Distribution', _err)


### 3a. Reporter Analysis

Which organisations (housing corporations, energy companies, health insurers…) send the signals,
and in what volume.


In [ ]:
_report_section('3a. Reporter (Melder) Analysis')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    sig_with_melder = signalen.merge(
        melders.rename(columns={'ID': 'Melder ID'}), on='Melder ID', how='left'
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    top_reporters = sig_with_melder['Melder'].value_counts().head(15)
    top_reporters.sort_values().plot.barh(ax=axes[0], color=COLORS[2])
    axes[0].set_title('Top 15 Reporters by Signal Volume')
    axes[0].set_xlabel('Number of signals')

    type_melder = sig_with_melder['Type melder'].value_counts()
    type_melder.plot.pie(ax=axes[1], autopct='%1.1f%%',
                         colors=sns.color_palette('Set2', len(type_melder)),
                         startangle=90)
    axes[1].set_title('Reporter Type Distribution (Type melder)')
    axes[1].set_ylabel('')

    plt.tight_layout()
    plt.show()

    print("Top 15 reporters by signal volume (values behind the left chart):")
    print(top_reporters.sort_values(ascending=False).rename('signals').to_string())
    print()
    print("Reporter type — signal counts:")
    print(
        type_melder.rename('signals')
        .to_frame()
        .assign(pct_of_total=(type_melder / len(signalen) * 100).round(1))
        .to_string()
    )
    _yr3 = sig_with_melder.dropna(subset=['Datum melding']).copy()
    _yr3['_yr'] = _yr3['Datum melding'].dt.year
    if _yr3['_yr'].nunique() > 1:
        _pv3 = (_yr3.pivot_table(index='Type melder', columns='_yr', values='Hash', aggfunc='count')
                .fillna(0).astype(int))
        print()
        print("Reporter type x year (signal counts):")
        print(_pv3.to_string())
    _section_done('3a. Reporter (Melder) Analysis')
except Exception as _err:
    _section_failed('3a. Reporter (Melder) Analysis', _err)


## 4. Signal → Report Conversion

A signal is "forwarded" (`Doorgezet = Ja`) when the municipality decides it deserves follow-up.
This measures the overall forwarding rate and whether it differs by signal type or reporter type.


In [ ]:
_report_section('4. Signal → Report Conversion')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    signalen['doorgezet_bool'] = ja_nee(signalen['Doorgezet'])
    signalen['has_report']     = signalen['Meldingnummer'].notna()

    total       = len(signalen)
    n_forwarded = int(signalen['doorgezet_bool'].sum())

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Overall pie
    doorgezet_counts = signalen['Doorgezet'].value_counts(dropna=False)
    doorgezet_counts.index = doorgezet_counts.index.fillna('(blank)')
    doorgezet_counts.plot.pie(ax=axes[0], autopct='%1.1f%%',
                              colors=[COLORS[1], COLORS[0], COLORS[3]], startangle=90)
    axes[0].set_title('Forwarded to Report?\n(Doorgezet)')
    axes[0].set_ylabel('')

    # By signal type
    type_conv = (signalen.groupby('Type melding')['doorgezet_bool']
                 .agg(forwarded='sum', total='count'))
    type_conv['rate_%'] = type_conv['forwarded'] / type_conv['total'] * 100
    type_conv['rate_%'].sort_values().plot.barh(ax=axes[1], color=COLORS[3])
    axes[1].axvline(n_forwarded / total * 100, color='red', linestyle='--',
                    linewidth=1.5, label=f'Overall {n_forwarded/total*100:.1f}%')
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Conversion Rate by Signal Type')
    axes[1].set_xlabel('% Forwarded')
    axes[1].legend(fontsize=9)

    # By reporter type
    if 'Type melder' in sig_with_melder.columns:
        sig_with_melder['doorgezet_bool'] = ja_nee(sig_with_melder['Doorgezet'])
        type_melder_conv = (sig_with_melder.groupby('Type melder')['doorgezet_bool']
                            .agg(forwarded='sum', total='count'))
        type_melder_conv['rate_%'] = (type_melder_conv['forwarded']
                                      / type_melder_conv['total'] * 100)
        type_melder_conv['rate_%'].sort_values().plot.barh(ax=axes[2], color=COLORS[4])
        axes[2].axvline(n_forwarded / total * 100, color='red', linestyle='--', linewidth=1.5)
        axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
        axes[2].set_title('Conversion Rate by Reporter Type')
        axes[2].set_xlabel('% Forwarded')

    plt.tight_layout()
    plt.show()

    print(f"Overall forwarding rate: {n_forwarded:,} / {total:,} = {n_forwarded/total*100:.1f}%")
    print()
    print("Conversion by signal type:")
    print(type_conv.sort_values('rate_%', ascending=False)
          .assign(**{'rate_%': type_conv['rate_%'].round(1)})
          .to_string())
    print()
    print("Forwarded vs not (Doorgezet) - counts behind the pie:")
    print(doorgezet_counts.rename('count').to_string())
    if 'Type melder' in sig_with_melder.columns:
        print()
        print("Conversion by reporter type (values behind the right chart):")
        print(type_melder_conv.sort_values('rate_%', ascending=False)
              .assign(**{'rate_%': type_melder_conv['rate_%'].round(1)}).to_string())

    _y4 = signalen.dropna(subset=['Datum melding']).copy()
    _y4['_yr'] = _y4['Datum melding'].dt.year
    _y4['_dz'] = ja_nee(_y4['Doorgezet']) == True
    if _y4['_yr'].nunique() > 1:
        _yt4 = _y4.groupby('_yr')['_dz'].agg(total='count', forwarded='sum')
        _yt4['rate_%'] = (_yt4['forwarded'] / _yt4['total'] * 100).round(1)
        print()
        print("Forwarding rate per year:")
        print(_yt4.to_string())

        if _y4['Type melding'].notna().any():
            _p4t = _y4.pivot_table(index='Type melding', columns='_yr', values='_dz', aggfunc='mean') * 100
            _n4t = _y4.pivot_table(index='Type melding', columns='_yr', values='_dz', aggfunc='count')
            print("\nForwarding rate by signal type per year (%, cells n>=30):")
            print(_p4t.where(_n4t >= 30).round(1).to_string())
        _ym4 = sig_with_melder.dropna(subset=['Datum melding']).copy()
        _ym4['_yr'] = _ym4['Datum melding'].dt.year
        _ym4['_dz'] = ja_nee(_ym4['Doorgezet']) == True
        if 'Type melder' in _ym4.columns and _ym4['Type melder'].notna().any():
            _p4m = _ym4.pivot_table(index='Type melder', columns='_yr', values='_dz', aggfunc='mean') * 100
            _n4m = _ym4.pivot_table(index='Type melder', columns='_yr', values='_dz', aggfunc='count')
            print("\nForwarding rate by reporter type per year (%, cells n>=30):")
            print(_p4m.where(_n4m >= 30).round(1).to_string())
    _section_done('4. Signal → Report Conversion')
except Exception as _err:
    _section_failed('4. Signal → Report Conversion', _err)


## 5. Contact Effectiveness

Once a case exists, the municipality tries to reach the client. This section measures reach —
overall, per contact method, and per time of day — using the shared definitions from §1b and
only dated records inside each case's window (signal → closure) for every rate, so all rates
use the same records.
Case-level reach is shown as reached / not reached / unknown / conflicting. Method reach rates
are shown separately for **outbound** methods (attempts with reach recorded); inbound callbacks
reach almost 100% by definition — they are responses, not outreach methods — and are listed
apart. (The `Bereikt` field in Meldingen is ~100% "N.v.t." for this municipality, so it is not used.)


In [ ]:
_report_section('5. Contact Effectiveness')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    # Shared definitions from 1b: contact records (contacts_all) and case status (CASES).
    # Meldingen 'Bereikt' is ~100% 'N.v.t.' for this municipality and is therefore not used.
    def _pct_ja(s):
        return (s == True).sum() / s.notna().sum() * 100 if s.notna().any() else np.nan

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    if len(contacts_all):
        contacts = contacts_all.copy()
        contacts['bereikt_bool'] = contacts['reached_flag']
        meldingen['reached_case'] = meldingen['Meldingnummer'].map(CASES['reach_status'].eq('reached')).fillna(False)

        _cs5 = (CASES.loc[CASES['n_attempts'] > 0, 'reach_status'].value_counts()
                .reindex(['reached', 'not reached', 'unknown', 'conflicting'], fill_value=0))
        _cs5p = _cs5[_cs5 > 0]
        if len(_cs5p):
            _cs5p.plot.pie(ax=axes[0], autopct='%1.1f%%', colors=sns.color_palette('Set2', len(_cs5p)), startangle=90)
        else:
            axes[0].text(0.5, 0.5, 'No case with a dated contact\nrecord in its window', ha='center', va='center',
                         transform=axes[0].transAxes, fontsize=11)
        axes[0].set_title('Client reached? (case level, shared definition)'); axes[0].set_ylabel('')

        _cw = contacts[contacts['in_window']]            # one filter for every rate below
        _out5 = _cw[_cw['method_kind'] == 'outbound']
        attempt_rate = _pct_ja(_out5['bereikt_bool'])
        method_reach = (_cw.groupby(['method_kind', 'Wijze van contactpoging'], dropna=False)
                        .agg(attempts=('_tab', 'size'), reach_recorded=('bereikt_bool', 'count'),
                             reach_rate=('bereikt_bool', _pct_ja)))
        _mr_out = (method_reach.xs('outbound', level='method_kind')
                   if 'outbound' in method_reach.index.get_level_values(0) else pd.DataFrame(columns=['reach_rate']))
        if _mr_out['reach_rate'].notna().any():
            _mr_out['reach_rate'].dropna().sort_values().plot.barh(ax=axes[1], color=COLORS[2])
            axes[1].axvline(attempt_rate, color='red', linestyle='--', linewidth=1.5, label='Outbound avg')
            axes[1].legend(fontsize=9)
        axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
        axes[1].set_title('Reach rate by OUTBOUND method\n(attempts with reach recorded)'); axes[1].set_xlabel('% Person Reached')

        dagdeel_reach = _out5.groupby('Dagdeel')['bereikt_bool'].agg(reach_rate=_pct_ja, n_recorded='count')
        if dagdeel_reach['reach_rate'].notna().any():
            dagdeel_reach['reach_rate'].dropna().sort_values().plot.barh(ax=axes[2], color=COLORS[4])
            axes[2].axvline(attempt_rate, color='red', linestyle='--', linewidth=1.5)
        axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
        axes[2].set_title('Outbound reach rate by time of day (Dagdeel)'); axes[2].set_xlabel('% Person Reached')
        plt.tight_layout(); plt.show()

        _n5 = int((CASES['n_attempts'] > 0).sum())
        print("Reach uses the shared definition (1b); Meldingen 'Bereikt' is unused (~100% 'N.v.t.').")
        print(f"Cases with >=1 contact record: {_n5:,}")
        for _s, _v in _cs5.items():
            print(f"  {_s:<12} {_v:>7,}  ({_v / max(_n5, 1) * 100:.1f}%)")
        print(f"Outbound attempt-level reach rate (attempts with reach recorded): {attempt_rate:.1f}%")
        print(f"  All rates use dated records inside each case's window; {int((~contacts['in_window']).sum()):,} "
              "records outside it are left out (see 1b).")
        print('  Reach rates are over attempts WITH reach recorded - letters and emails have none - so they')
        print('  are not response rates over all outreach.')
        print('\nAll contact methods - attempts, attempts with reach recorded, reach rate (by kind):')
        print('  outbound = municipality reaches out | inbound = client contacted the municipality')
        print('  (near 100% reached by definition - a response, not an outreach method) |')
        print('  admin = result entered without a contact attempt')
        display(method_reach.round(1))
        print('\nOutbound reach rate by time of day (Dagdeel) - values behind the right chart:')
        print(dagdeel_reach.round(1).to_string())

        _cy = _out5.dropna(subset=['Datum contactpoging'])
        _cy = _cy[_cy['bereikt_bool'].notna()].copy()
        _cy['_yr'] = _cy['Datum contactpoging'].dt.year
        _cy['_r'] = _cy['bereikt_bool'] == True
        if _cy['_yr'].nunique() > 1:
            _yt5 = _cy.groupby('_yr')['_r'].agg(attempts='count', reach='mean')
            _yt5['reach_%'] = (_yt5['reach'] * 100).round(1)
            print("\nOutbound reach rate per year (attempts with reach recorded):")
            print(_yt5.drop(columns='reach').to_string())
            _p5m = _cy.pivot_table(index='Wijze van contactpoging', columns='_yr', values='_r', aggfunc='mean') * 100
            _n5m = _cy.pivot_table(index='Wijze van contactpoging', columns='_yr', values='_r', aggfunc='count')
            _p5m = _p5m.where(_n5m >= 100).dropna(how='all')
            if len(_p5m):
                print("\nOutbound reach rate by method per year (%, cells with n>=100 attempts):")
                print(_p5m.round(1).to_string())
            _cd5 = _cy.dropna(subset=['Dagdeel'])
            if len(_cd5) >= 200:
                _p5d = _cd5.pivot_table(index='Dagdeel', columns='_yr', values='_r', aggfunc='mean') * 100
                _n5d = _cd5.pivot_table(index='Dagdeel', columns='_yr', values='_r', aggfunc='count')
                _p5d = _p5d.where(_n5d >= 100).dropna(how='all')
                if len(_p5d):
                    print("\nOutbound reach rate by time of day per year (%, cells with n>=100 attempts):")
                    print(_p5d.round(1).to_string())
    else:
        contacts = contacts_all.copy()
        meldingen['reached_case'] = False
        for ax in axes:
            ax.text(0.5, 0.5, 'No contact-attempt tabs\nfound in this export',
                    ha='center', va='center', transform=ax.transAxes, fontsize=11)
        plt.tight_layout(); plt.show()
        print("No contact-attempt tabs found in this export.")
    _section_done('5. Contact Effectiveness')
except Exception as _err:
    _section_failed('5. Contact Effectiveness', _err)


## 6. Client Response & Outcomes

Among clients who were reached: do they want help (`Wil de klant hulp`),
and what is the final result of the case (`Wat is het eindresultaat`)? A case is included only
when it counts as **reached** under the shared definition (§1b: an attempt with
`Persoon bereikt = Ja` or an inbound contact, without a conflicting "no contact" result).
Not-reached, unknown and conflicting cases are filtered out. Everything here is **per case**, from
the shared case table: the help answer and final result come only from dated records between the
signal and closure, so an answer recorded after closure is not shown. Within the reached cohort, `(not recorded)` means that the help decision or
final outcome itself is missing. The joint matrix then shows how final outcomes are distributed
within each recorded help response.


In [ ]:
_report_section('6. Client Response & Outcomes')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    if 'Eindresultaat' in extra_tabs:
        # One row per CASE from the shared, windowed case table (1b): help answer and final result
        # come only from dated records between the signal and closure. Only reached cases are shown.
        _rs6 = CASES['reach_status'].value_counts().reindex(
            ['reached', 'not reached', 'unknown', 'conflicting'], fill_value=0)
        print('Cases by reach status (shared definition, records in the window):')
        print(_rs6.rename('cases').to_string())
        print('  Only reached cases are analysed below; unknown and conflicting are not counted as not reached.')
        eind = (CASES[CASES['reach_status'].eq('reached')]
                .rename(columns={'help_label': 'Wil de klant hulp', 'result_label': 'Wat is het eindresultaat',
                                 'first_reach_date': 'Datum contactpoging'})
                .reset_index()[['Meldingnummer', 'Wil de klant hulp', 'Wat is het eindresultaat',
                                'Datum contactpoging']])
        eind['Datum contactpoging'] = pd.to_datetime(eind['Datum contactpoging'], errors='coerce')

        # Defensive consistency filter: a final result explicitly saying no contact does not belong
        # in reached-client response/outcome charts, even if another attempt says Ja.
        _no_contact6 = eind['Wat is het eindresultaat'].astype(str).str.contains(
            'geen contact kunnen', case=False, na=False)
        if int(_no_contact6.sum()):
            print(f'  Excluded {int(_no_contact6.sum()):,} contradictory explicit no-contact case(s).')
            eind = eind[~_no_contact6].copy()
        print(f'  Reached cases analysed: {len(eind):,}')
        print('  In the tables below, (not recorded) means a missing response/outcome among reached clients.')

        if len(eind) == 0:
            print('  No reached cases - help and final-result charts skipped.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            hulp_vc = eind['Wil de klant hulp'].value_counts(dropna=False)
            hulp_vc.index = hulp_vc.index.fillna('(not recorded)')
            hulp_vc.plot.pie(ax=axes[0], autopct='%1.1f%%',
                             colors=sns.color_palette('Set2', len(hulp_vc)), startangle=90)
            axes[0].set_title('Does the Client Want Help?\n(Wil de klant hulp)')
            axes[0].set_ylabel('')

            result_vc = eind['Wat is het eindresultaat'].value_counts(dropna=False)
            result_vc.index = result_vc.index.fillna('(not recorded)')
            result_vc.sort_values().plot.barh(ax=axes[1], color=COLORS[0])
            axes[1].set_title('Final Outcome (Wat is het eindresultaat)')
            axes[1].set_xlabel('Count')

            plt.tight_layout()
            plt.show()

            print("Does the client want help? (Wil de klant hulp) - counts behind the pie:")
            print(hulp_vc.rename('count').to_frame()
                  .assign(pct_=(hulp_vc / hulp_vc.sum() * 100).round(1)).to_string())
            print()
            print("Outcome breakdown:")
            print(
                result_vc.rename('count')
                .to_frame()
                .assign(pct_=(result_vc / result_vc.sum() * 100).round(1))
                .sort_values('count', ascending=False)
                .to_string()
            )

            # ── Joint help-response x final-outcome matrix (reached cohort only) ──────
            _help6 = eind['Wil de klant hulp'].fillna('(not recorded)').astype(str).rename('Wil de klant hulp')
            _result6 = (eind['Wat is het eindresultaat'].fillna('(not recorded)').astype(str)
                        .rename('Eindresultaat'))
            _mx6_n = pd.crosstab(_help6, _result6, dropna=False)
            _mx6_n = _mx6_n.loc[_mx6_n.sum(axis=1).sort_values(ascending=False).index,
                                 _mx6_n.sum(axis=0).sort_values(ascending=False).index]
            _mx6_pct = _mx6_n.div(_mx6_n.sum(axis=1), axis=0) * 100
            _mx6_counts = _mx6_n.copy()
            _mx6_counts['Total'] = _mx6_counts.sum(axis=1)
            _mx6_counts.loc['Total'] = _mx6_counts.sum(axis=0)
            print("\nJoint matrix: 'Wil de klant hulp' (rows) x final outcome (columns)")
            print('Counts (reached cases):')
            display(_mx6_counts)
            print('Row percentages: within each help response, how the final outcomes are distributed.')
            print('(not recorded) is a missing field among reached clients, not a not-reached category.')
            display(_mx6_pct.round(1))

            if _mx6_n.size:
                _annot6 = (_mx6_n.astype(str) + '\n(' + _mx6_pct.round(1).astype(str) + '%)')
                _w6 = min(24, max(10, 1.5 * len(_mx6_n.columns)))
                _h6 = min(16, max(4, 0.9 * len(_mx6_n.index)))
                fig, ax = plt.subplots(figsize=(_w6, _h6))
                sns.heatmap(_mx6_pct, annot=_annot6, fmt='', cmap='Blues', vmin=0, vmax=100,
                            linewidths=.5, cbar_kws={'label': '% within help response'}, ax=ax)
                ax.set_title('Final outcome by whether the reached client wants help')
                ax.set_xlabel('Final outcome (Wat is het eindresultaat)')
                ax.set_ylabel('Wants help (Wil de klant hulp)')
                ax.tick_params(axis='x', labelrotation=45)
                for _lab6 in ax.get_xticklabels():
                    _lab6.set_horizontalalignment('right')
                plt.tight_layout(); plt.show()

            _ey = eind.dropna(subset=['Datum contactpoging']).copy()
            _ey['_yr'] = _ey['Datum contactpoging'].dt.year
            if _ey['_yr'].nunique() > 1:
                _er6 = _ey['Wat is het eindresultaat'].fillna('(not recorded)').astype(str).str[:45]
                _p6r = pd.crosstab(_er6, _ey['_yr'], normalize='columns') * 100
                print("\nFinal result mix per year (% of reached cases, by year of first contact):")
                print(_p6r.round(1).to_string())
                _wh6 = _ey['Wil de klant hulp'].fillna('(not recorded)').astype(str).str[:35]
                _p6w = pd.crosstab(_wh6, _ey['_yr'], normalize='columns') * 100
                print("\n'Wil de klant hulp' mix per year (% of reached cases, by year of first contact):")
                print(_p6w.round(1).to_string())
    else:
        print("No 'Eindresultaat' tab found in this export.")
    _section_done('6. Client Response & Outcomes')
except Exception as _err:
    _section_failed('6. Client Response & Outcomes', _err)


## 7. Case Closure & Timing

Status overview of all cases (closed, reached, withdrawn, incorrect) and how long it takes
from the first signal to case closure. **Closed** (`Afgerond = Ja`) is administrative closure,
not proof that the debt was resolved; **reached** uses the shared definition (§1b).


In [ ]:
_report_section('7. Case Closure & Timing')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    meldingen['afgerond_bool']    = ja_nee(meldingen['Afgerond'])
    meldingen['ingetrokken_bool'] = ja_nee(meldingen['Ingetrokken'])
    meldingen['onterecht_bool']   = ja_nee(meldingen['Onterecht'])
    if 'reached_case' not in meldingen.columns:
        meldingen['reached_case'] = False

    earliest_signal = (signalen.dropna(subset=['Meldingnummer'])
                       .groupby('Meldingnummer')['Datum melding'].min()
                       .rename('Datum melding'))
    mel = meldingen.join(earliest_signal, on='Meldingnummer')
    mel['days_to_close'] = (mel['Afgerond d.d.'] - mel['Datum melding']).dt.days

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    n_mel = len(meldingen)
    status = pd.Series({
        'Closed\n(Afgerond)':       int(meldingen['afgerond_bool'].sum()),
        'Reached\n(shared def.)':   int(meldingen['reached_case'].sum()),
        'Withdrawn\n(Ingetrokken)': int(meldingen['ingetrokken_bool'].sum()),
        'Incorrect\n(Onterecht)':   int(meldingen['onterecht_bool'].sum()),
    })
    bars = axes[0].bar(status.index, status.values, color=COLORS[:4])
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                     f'{int(h):,}\n({h/max(n_mel,1)*100:.1f}%)',
                     ha='center', va='bottom', fontsize=9)
    axes[0].set_title(f'Report Status Overview  (n={n_mel:,})')
    axes[0].set_ylabel('Count')

    dtc = mel['days_to_close'].dropna()
    dtc = dtc[dtc >= 0]
    if len(dtc) > 1:
        dtc.hist(ax=axes[1], bins=min(30, len(dtc)), color=COLORS[2], edgecolor='white')
        axes[1].axvline(dtc.median(), color='red', linestyle='--', linewidth=1.5,
                        label=f'Median: {dtc.median():.0f}d')
        axes[1].axvline(dtc.mean(), color='orange', linestyle=':', linewidth=1.5,
                        label=f'Mean: {dtc.mean():.0f}d')
        axes[1].set_title('Days from Signal to Case Closure')
        axes[1].set_xlabel('Days'); axes[1].set_ylabel('Count'); axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'Insufficient data\nfor time-to-close',
                    ha='center', va='center', transform=axes[1].transAxes, fontsize=11)

    plt.tight_layout(); plt.show()

    print("Status summary (of all cases in the export):")
    for label, count in status.items():
        print(f"  {label.replace(chr(10), ' '):<30} {count:>6,}  ({count/max(n_mel,1)*100:.1f}%)")
    print("  Note: 'Closed' = administrative closure (Afgerond = Ja), not proof that the debt was")
    print("        resolved. 'Reached' uses the shared definition (1b). Days-to-close covers only")
    print("        cases with a linked signal in the observed window.")
    if len(dtc) > 1:
        print(f"\nTime to close (days) - percentiles behind the histogram:")
        print(f"  mean={dtc.mean():.0f}  p10={dtc.quantile(.1):.0f}  p25={dtc.quantile(.25):.0f}  "
              f"p50={dtc.median():.0f}  p75={dtc.quantile(.75):.0f}  p90={dtc.quantile(.9):.0f}")

    _myr = mel.dropna(subset=['days_to_close', 'Afgerond d.d.']).copy()
    _myr = _myr[_myr['days_to_close'] >= 0]
    _myr['_yr'] = _myr['Afgerond d.d.'].dt.year
    if _myr['_yr'].nunique() > 1:
        _yt7a = _myr.groupby('_yr')['days_to_close'].agg(cases='count', median_days='median').round(0)
        print("\nTime to close per close-year:")
        print(_yt7a.to_string())

    _ms7 = meldingen.dropna(subset=['Afgerond d.d.']).copy()
    _ms7['_yr'] = _ms7['Afgerond d.d.'].dt.year
    if _ms7['_yr'].nunique() > 1:
        _st7 = pd.DataFrame({
            'cases': _ms7.groupby('_yr').size(),
            'closed %': _ms7.groupby('_yr')['afgerond_bool'].apply(lambda s: (s == True).mean() * 100),
            'reached %': _ms7.groupby('_yr')['reached_case'].apply(lambda s: (s == True).mean() * 100),
            'withdrawn %': _ms7.groupby('_yr')['ingetrokken_bool'].apply(lambda s: (s == True).mean() * 100),
        })
        print("\nCase status per close-year:")
        print(_st7.round(1).to_string())
    _section_done('7. Case Closure & Timing')
except Exception as _err:
    _section_failed('7. Case Closure & Timing', _err)


### 7a. Recidive — Repeated Signals

A recidive signal means the same household signalled before. Shows what share of signals are
recidive (per month-prior flag) and whether repeat households are forwarded more or less often.


In [ ]:
_report_section('7a. Recidive — Repeated Signals')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    recidive_cols = [c for c in signalen.columns if c.startswith('Recidive maand')]

    if recidive_cols:
        rec_rates = {col.replace('Recidive ', ''): ja_nee(signalen[col]).mean() * 100
                     for col in recidive_cols}
        overall_rec = ja_nee(signalen['Recidive']).mean() * 100

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        pd.Series(rec_rates).plot.bar(ax=axes[0], color=COLORS[0])
        axes[0].axhline(overall_rec, color='red', linestyle='--', linewidth=1.5,
                        label=f'Overall: {overall_rec:.1f}%')
        axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
        axes[0].set_title('Recidive Rate by Month Prior')
        axes[0].set_ylabel('% of signals flagged as recidive')
        axes[0].tick_params(axis='x', rotation=0)
        axes[0].legend()

        rec_conv = (signalen
                    .groupby(ja_nee(signalen['Recidive']).rename('recidive'))
                    ['doorgezet_bool'].mean() * 100)
        rec_conv.index = rec_conv.index.map({True: 'Recidive', False: 'No Recidive'})
        rec_conv.plot.bar(ax=axes[1], color=[COLORS[1], COLORS[2]])
        axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
        axes[1].set_title('Forwarding Rate: Recidive vs No Recidive')
        axes[1].set_ylabel('% Forwarded to Report (Doorgezet)')
        axes[1].tick_params(axis='x', rotation=0)

        plt.tight_layout()
        plt.show()

        print(f"Overall recidive rate: {overall_rec:.1f}%")
        print(pd.Series(rec_rates).round(1).rename('recidive_%').to_string())
        print()
        print("Forwarding rate by recidive status (values behind the right chart):")
        print(rec_conv.round(1).rename('forwarded_%').to_frame().to_string())

        _ry = signalen.dropna(subset=['Datum melding']).copy()
        _ry['_yr'] = _ry['Datum melding'].dt.year
        _ry['_rec'] = ja_nee(_ry['Recidive']) == True
        if _ry['_yr'].nunique() > 1:
            _yt7 = (_ry.groupby('_yr')['_rec'].mean() * 100).round(1)
            print("\nRecidive rate per signal year (%):")
            print(_yt7.rename('recidive_%').to_string())

            _pm7 = {}
            for _mc in recidive_cols:
                _ry['_mb'] = ja_nee(_ry[_mc]) == True
                _pm7[_mc.replace('Recidive ', '')] = _ry.groupby('_yr')['_mb'].mean() * 100
            print("\nMaand-flag rates per year (%):")
            print(pd.DataFrame(_pm7).round(1).to_string())
            _ry['_dz7'] = ja_nee(_ry['Doorgezet']) == True
            _pf7 = _ry.pivot_table(index='_rec', columns='_yr', values='_dz7', aggfunc='mean') * 100
            _pf7.index = _pf7.index.map({True: 'Recidive', False: 'No recidive'})
            print("\nForwarding rate by recidive status per year (%):")
            print(_pf7.round(1).to_string())
    else:
        print("No recidive month columns found.")
    _section_done('7a. Recidive — Repeated Signals')
except Exception as _err:
    _section_failed('7a. Recidive — Repeated Signals', _err)


## 8. Signal-to-Case Conversion and Case Funnel

Two separate views, each with one unit:

- **(a) Signal conversion** (signal counts): signals received → forwarded (`Doorgezet = Ja`) →
  linked to a case number.
- **(b) Case funnel** (cases from forwarded signals in this export): case → at least one contact
  record → client reached → help accepted. Every stage is a subset of the one above and uses the
  shared definitions (§1b); reach-unknown and conflicting cases are shown, not counted as
  "not reached".

Administrative closure (`Afgerond = Ja`) is shown separately: it is not a funnel stage and not
proof that the debt was resolved. "Help accepted" includes a follow-up appointment, which is not
proof that help was received.

In [ ]:
_report_section('8. Signal-to-Case Conversion and Case Funnel')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _dz = ja_nee(signalen['Doorgezet']) == True
    n_signals   = len(signalen)
    n_forwarded = int(_dz.sum())
    n_linked    = int((_dz & signalen['Meldingnummer'].notna()).sum())
    print('(a) Signal conversion (signal counts):')
    print(f"  Signals received (observed period)          {n_signals:>7,}  (100.0%)")
    print(f"  Forwarded (Doorgezet = Ja)                  {n_forwarded:>7,}  ({n_forwarded / max(n_signals, 1) * 100:.1f}%)")
    print(f"  Forwarded AND linked to a case number       {n_linked:>7,}  ({n_linked / max(n_signals, 1) * 100:.1f}%)")

    cohort_mnums = set(signalen.loc[_dz, 'Meldingnummer'].dropna())
    _cc8 = CASES[CASES.index.isin(cohort_mnums)]
    n_cases   = len(_cc8)
    n_contact = int((_cc8['n_attempts'] > 0).sum())
    n_reached = int(_cc8['reach_status'].eq('reached').sum())
    n_accept  = int((_cc8['reach_status'].eq('reached') & _cc8['help_status'].eq('accepted')).sum())
    stages = ['Distinct cases\n(from forwarded signals)', 'With >=1 dated contact record\n(signal to closure)',
              'Client reached\n(shared definition)', 'Help accepted\n(reached; shared definition)']
    values = [n_cases, n_contact, n_reached, n_accept]

    palette = sns.color_palette('Blues_r', len(stages))
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(range(len(stages)), values, color=palette)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(n_cases, 1) * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:,}   ({val / max(n_cases, 1) * 100:.1f}% of cases)', va='center', fontsize=10)
    ax.set_yticks(range(len(stages))); ax.set_yticklabels(stages, fontsize=11)
    ax.set_xlabel('Cases')
    ax.set_title('(b) Case funnel - cases from forwarded signals in this export', fontsize=13, pad=12)
    ax.set_xlim(0, max(n_cases, 1) * 1.45); ax.invert_yaxis()
    plt.tight_layout(); plt.show()

    print('\n(b) Case funnel (each stage is a subset of the one above; % of cases):')
    for stage, val in zip(stages, values):
        print(f"  {stage.replace(chr(10), ' '):<50} {val:>7,}  ({val / max(n_cases, 1) * 100:.1f}%)")
    _rs8 = _cc8['reach_status'].value_counts()
    print(f"  (reach unknown: {int(_rs8.get('unknown', 0)):,} | conflicting records: {int(_rs8.get('conflicting', 0)):,}"
          " - not counted as reached or as not reached)")
    _cl8 = meldingen[meldingen['Meldingnummer'].isin(cohort_mnums)]
    _closed8 = int((ja_nee(_cl8['Afgerond']) == True).sum())
    print(f"\nAdministratively closed (Afgerond = Ja): {_closed8:,} of {n_cases:,} cases "
          f"({_closed8 / max(n_cases, 1) * 100:.1f}%) - closure is not proof that the debt was resolved.")
    print("'Help accepted' includes 'follow-up appointment made' and 'wants contact with VST' - an")
    print("appointment is not proof that help was received.")

    _fy = signalen.dropna(subset=['Datum melding']).copy()
    _fy['_yr'] = _fy['Datum melding'].dt.year
    if _fy['_yr'].nunique() > 1:
        _rows_y = []
        for _yy, _gy in _fy.groupby('_yr'):
            _dzy = ja_nee(_gy['Doorgezet']) == True
            _cy8 = CASES[CASES.index.isin(set(_gy.loc[_dzy, 'Meldingnummer'].dropna()))]
            _k = len(_cy8)
            _rows_y.append({'year': int(_yy), 'signals': len(_gy), 'forwarded %': round(_dzy.mean() * 100, 1),
                            'cases': _k,
                            'reached % of cases': round(_cy8['reach_status'].eq('reached').mean() * 100, 1) if _k else np.nan,
                            'reach unknown/conflicting %': round(_cy8['reach_status'].isin(['unknown', 'conflicting']).mean() * 100, 1) if _k else np.nan,
                            'help accepted % of cases': round((_cy8['reach_status'].eq('reached')
                                                               & _cy8['help_status'].eq('accepted')).mean() * 100, 1) if _k else np.nan})
        print("\nPer signal year:")
        print(pd.DataFrame(_rows_y).set_index('year').to_string())
    _section_done('8. Signal-to-Case Conversion and Case Funnel')
except Exception as _err:
    _section_failed('8. Signal-to-Case Conversion and Case Funnel', _err)


## 9. Effectiveness Framework — Is Contact Associated With Fewer Later Signals?

Sections 9–12 look at what happens **after** contact, not at reach itself. Two outcome
definitions are used, and they measure different things:

- **Main outcome (§9c): signalled again 2–12 months after the signal.** Every case gets the same
  clock — time zero is the case's signal date, contact and closure must be finished by day 60,
  and the outcome is *any* new signal of the household in day 61–365, split into *still reported*
  (the same problem ran on) and *came back* (a new episode after a quiet period). This is the
  preferred comparison of the contact groups.
- **Secondary outcome (§9a, §9b, §10–§12): a subsequent recorded signal more than 14 days after
  case closure** ("return" for short). It starts at closure, so it partly depends on *when* a case
  was closed relative to the creditors' monthly reports; it is kept for comparison with earlier
  reports. Cases without such a signal stay in the analysis until the data cutoff
  (administrative censoring) — they are not counted as failures or dropped.

Neither outcome proves that a debt was solved or that a new debt arose: a missing signal can
also mean moving away or leaving the reporting system (§13d), and a later signal can be the same
unpaid debt reported again. All comparisons are **associations**, not effects. §9d checks whether
results are comparable across calendar years before anything is pooled.

**Who is included (eligibility):** closed cases with a known household, linked to one household
only, with a signal date and a close date on or after it, and not closed as "invalid / not picked
up"; the exclusion flow is printed below. **Contact groups:** A = not reached, B = reached and
declined help, C = reached and accepted help (shared definitions, §1b). Cases with unknown or
conflicting reach, reached cases with no help answer or still hesitating, and households already
known to debt counselling or another team are counted but kept out of the A/B/C comparison.

**Time windows:** groups are assigned from dated contact records between the index signal and
closure (§1b); the effect of that rule on the groups is shown in the audit below.

**Calendar time (pre-specified):** every group comparison reports, in the same layout whatever the
results show: a pooled model (no calendar term), the **PRIMARY year-adjusted model** (calendar
year as categories), the same model within fixed two-year periods (2021–22, 2023–24, 2025–26) and
a household-clustered test of whether the group difference changes between periods.

Return percentages are **Kaplan–Meier estimates**: 100% means the estimated chance of no return
reached zero — not that every household, including censored ones, was seen to return.

**Observation end:** follow-up, censoring and eligibility end at `OBSERVATION_END` (setup cell) —
the date through which signal reporting is complete, to be confirmed by the data owner. Until it
is set, the latest signal in the file is used as a stand-in and this is printed below.

Many households have several cases, so Cox and logistic p-values are clustered by household.
With many tests in this notebook, isolated p-values below 0.05 are expected by chance alone —
read them as exploratory.

In [ ]:
_report_section('9. Effectiveness Framework — data preparation')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    import numpy as np
    import bisect
    from statsmodels.duration.survfunc import SurvfuncRight, survdiff
    from statsmodels.duration.hazard_regression import PHReg
    from scipy import stats as _st

    # EXPORT_DATE = end of observation, used for ALL follow-up, censoring and eligibility below
    # (sections 9-15). It is OBSERVATION_END (setup cell) when declared, otherwise the latest signal.
    _LAST_SIGNAL = signalen['Datum melding'].max()
    if globals().get('OBSERVATION_END') is not None:
        EXPORT_DATE = pd.Timestamp(OBSERVATION_END).normalize()
        print(f"Observation end: {EXPORT_DATE.date()} (declared OBSERVATION_END); latest signal in the file: "
              f"{_LAST_SIGNAL.date() if pd.notna(_LAST_SIGNAL) else 'n/a'}")
        if pd.notna(_LAST_SIGNAL) and _LAST_SIGNAL > EXPORT_DATE:
            print(f"  WARNING: {int((signalen['Datum melding'] > EXPORT_DATE).sum()):,} signal(s) dated after "
                  "OBSERVATION_END - check the setting.")
    elif globals().get('ANALYSIS_END') is not None:
        EXPORT_DATE = min(pd.Timestamp(ANALYSIS_END).normalize(), _LAST_SIGNAL.normalize()) \
                      if pd.notna(_LAST_SIGNAL) else pd.Timestamp(ANALYSIS_END).normalize()
        print(f"Observation end: {EXPORT_DATE.date()} - ANALYSIS_END (setup cell; the household-key change), or the "
              f"latest signal if the data end earlier (latest signal: "
              f"{_LAST_SIGNAL.date() if pd.notna(_LAST_SIGNAL) else 'n/a'}).")
    else:
        EXPORT_DATE = _LAST_SIGNAL
        print(f"Observation end: {EXPORT_DATE.date() if pd.notna(EXPORT_DATE) else 'n/a'} - FALLBACK: latest signal "
              "in the file, because OBSERVATION_END is not set.")
        _dc0 = globals().get('DATA_CUTOFF')
        if _dc0 is not None and pd.notna(EXPORT_DATE):
            print(f"  Gap to the declared extraction date (DATA_CUTOFF {pd.Timestamp(_dc0).date()}): "
                  f"{(pd.Timestamp(_dc0).normalize() - EXPORT_DATE).days} days - follow-up in that gap is ignored.")
        else:
            print('  No extraction date declared (DATA_CUTOFF), so the gap to it cannot be shown.')
        print('  Set OBSERVATION_END once the data owner confirms the date through which reporting is complete.')

    # A new signal arriving within this many days of case close is treated as re-reporting
    # of the same episode (creditors deliver signals in monthly batches), not a genuine return.
    MIN_GAP_DAYS = 14
    print(f"Minimum gap for a 'return' (secondary outcome): >{MIN_GAP_DAYS} days after case close")

    # Signals of one household up to this many days apart belong to the same problem period
    # ('episode'): creditors re-send an unpaid debt about monthly (median 31-33d, see 15a).
    # Used by the same-clock check (9c) and the episode analyses (13-15).
    EPISODE_GAP_DAYS = 45

    # ── Pre-specified calendar handling (fixed before any group comparison by period is examined) ──
    # Every group comparison reports, in the same layout and whatever the results show:
    #   pooled (no calendar term) | PRIMARY: year-adjusted (calendar year as categories) |
    #   the same model within each fixed period | a household-clustered group x period Wald test.
    # Periods = fixed two-year blocks from 2021 (Wgs start): 2021-2022, 2023-2024, 2025-2026, ...
    CAL_ANCHOR_YEAR, CAL_BLOCK_YEARS = 2021, 2
    def cal_period(year):
        y0 = CAL_ANCHOR_YEAR + CAL_BLOCK_YEARS * ((int(year) - CAL_ANCHOR_YEAR) // CAL_BLOCK_YEARS)
        return f'{y0}-{y0 + CAL_BLOCK_YEARS - 1}'
    print(f"Calendar handling (pre-specified): PRIMARY = year-adjusted; pooled, per-period "
          f"({CAL_BLOCK_YEARS}-year blocks from {CAL_ANCHOR_YEAR}) and interaction results are always reported too.")

    # ── Clean age (raw column contains impossible values e.g. -7973, 125) ─────────
    _age = pd.to_numeric(signalen['Leeftijd'], errors='coerce')
    signalen['Leeftijd_clean'] = _age.where(_age.between(18, 99))

    # ── Signal-level helper flags ─────────────────────────────────────────────────
    signalen['doorgezet_bool'] = ja_nee(signalen['Doorgezet'])
    signalen['recidive_hist']  = ja_nee(signalen['Recidive'])   # built-in backward-looking flag

    # reporter type onto signals (reporter registry deduplicated in 1b)
    if 'Melder ID' in signalen.columns and 'Type melder' in melders.columns:
        _mel = melders.rename(columns={'ID': 'Melder ID'})[['Melder ID', 'Type melder']]
        signalen_x = signalen.merge(_mel, on='Melder ID', how='left', validate='many_to_one')
    else:
        signalen_x = signalen.copy(); signalen_x['Type melder'] = np.nan

    # debt & age bands (ascii labels - safe for regression formulas)
    _DEBT_LABELS = ['<200', '200-700', '700-2k', '>2k']
    _debt_col = next((c for c in ['Achterstandsbedrag', 'Termijnbedrag'] if c in signalen_x.columns), None)
    if _debt_col:
        signalen_x['debt_band'] = pd.cut(pd.to_numeric(signalen_x[_debt_col], errors='coerce').clip(lower=0),
            bins=[-1, 200, 700, 2000, 1e12], labels=_DEBT_LABELS)
    else:
        print('  NOTE: no debt amount column - debt band is missing for every case.')
        signalen_x['debt_band'] = pd.Categorical([np.nan] * len(signalen_x), categories=_DEBT_LABELS)
    signalen_x['age_band'] = pd.cut(signalen_x['Leeftijd_clean'],
        bins=[18, 30, 45, 60, 100], right=False, labels=['18-29', '30-44', '45-59', '60+'])

    # ── Case -> originating (earliest) signal ROW + its covariates ────────────────
    _slink = signalen_x.dropna(subset=['Meldingnummer']).sort_values('Datum melding', kind='stable')
    _first = (_slink.drop_duplicates('Meldingnummer', keep='first')
              .rename(columns={'Datum melding': 'signal_date'}))

    # ── Contact-derived case features: the shared case table from 1b ──────────────
    contacts = contacts_all.copy()
    contacts['_reached'] = contacts['reached_flag']
    _case_feats = (CASES[['n_attempts', 'n_outbound', 'reach_status', 'first_reach_date', 'reach_channel',
                          'help_label', 'help_status', 'help_date', 'result_label', 'result_class',
                          'n_after_close', 'reach_status_allrec', 'help_status_allrec', 'result_class_allrec']]
                   .rename(columns={'reach_channel': 'reach_method', 'help_label': 'wil_hulp',
                                    'result_label': 'eindresultaat'})
                   .reset_index())

    # ── Closed cases -> one eligibility reason per case ───────────────────────────
    _mc = meldingen[['Meldingnummer', 'Afgerond d.d.']].copy()
    _mc['afgerond'] = ja_nee(meldingen['Afgerond'])
    _n_cases_all = len(_mc)
    _mc = _mc[(_mc['afgerond'] == True) & _mc['Afgerond d.d.'].notna()].rename(columns={'Afgerond d.d.': 'close_date'})
    _fcols = [c for c in ['Meldingnummer', 'Hash', 'Melder ID', 'signal_date', 'doorgezet_bool', 'recidive_hist',
                          'Type melder', 'debt_band', 'age_band', 'Leeftijd_clean'] if c in _first.columns]
    _all = _mc.merge(_first[_fcols], on='Meldingnummer', how='left', validate='one_to_one', indicator=True)
    _all = _all.merge(_case_feats, on='Meldingnummer', how='left', validate='one_to_one')
    for _c, _fill in [('result_class', 'not recorded'), ('result_class_allrec', 'not recorded'),
                      ('reach_status', 'unknown'), ('reach_status_allrec', 'unknown'),
                      ('help_status', 'unknown'), ('help_status_allrec', 'unknown')]:
        _all[_c] = _all[_c].fillna(_fill)

    # next signal from the same Hash strictly after close_date + MIN_GAP_DAYS (secondary outcome)
    _allsig = signalen.dropna(subset=['Hash', 'Datum melding'])
    _sig_by_hash = _allsig.sort_values('Datum melding').groupby('Hash')['Datum melding'].apply(list).to_dict()
    def _next_sig(h, c):
        arr = _sig_by_hash.get(h)
        if not arr or pd.isna(c):
            return pd.NaT
        i = bisect.bisect_right(arr, c + pd.Timedelta(days=MIN_GAP_DAYS))
        return arr[i] if i < len(arr) else pd.NaT
    _all['next_signal_date'] = pd.to_datetime(pd.Series([_next_sig(h, c) for h, c in zip(_all['Hash'], _all['close_date'])],
                                                        index=_all.index, dtype=object), errors='coerce')
    for _dc in ['close_date', 'signal_date']:
        _all[_dc] = pd.to_datetime(_all[_dc], errors='coerce')
    # event=0: no return observed -> kept as signal-free through the cutoff, administratively censored
    _all['event'] = _all['next_signal_date'].notna().astype(int)
    _all['duration'] = (_all['next_signal_date'].fillna(EXPORT_DATE) - _all['close_date']).dt.days

    _REASONS = ['not linked to a signal', 'household (Hash) unknown', 'linked to more than one household',
                'signal date missing / closed before the signal', 'closed on or after the data cutoff',
                "closed as 'invalid / not picked up'"]
    _all['eligibility'] = np.select(
        [_all['_merge'].eq('left_only'), _all['Hash'].isna(), _all['Meldingnummer'].isin(AMBIGUOUS_CASES),
         _all['signal_date'].isna() | (_all['close_date'] < _all['signal_date']), ~(_all['duration'] > 0),
         _all['result_class'].eq('invalid / not picked up')],
        _REASONS, default='eligible')

    # ── Contact groups (shared definitions, 1b) ───────────────────────────────────
    ARM_LABEL = {'A_not_reached': 'A - not reached', 'B_refused': 'B - help declined',
                 'C_accepted': 'C - help accepted',
                 'U_reach_unknown': 'U1 - reach unknown / conflicting',
                 'U_help_unknown': 'U2 - reached, help answer unknown / hesitating',
                 'X_in_care': 'X - already known to counselling / other team',
                 'B1_resolving': 'B1 - declined, final result says resolving',
                 'B2_difficulty': 'B2 - declined, other final results'}
    ARMS_ABC = ['A_not_reached', 'B_refused', 'C_accepted']
    def _arm_rule(result_cls, reach_st, help_st):
        return np.select([result_cls.eq('already in care'), reach_st.eq('not reached'), ~reach_st.eq('reached'),
                          help_st.eq('accepted'), help_st.eq('declined')],
                         ['X_in_care', 'A_not_reached', 'U_reach_unknown', 'C_accepted', 'B_refused'],
                         default='U_help_unknown')
    _all['arm'] = _arm_rule(_all['result_class'], _all['reach_status'], _all['help_status'])
    _all['arm_allrec'] = _arm_rule(_all['result_class_allrec'], _all['reach_status_allrec'], _all['help_status_allrec'])
    _all['group'] = np.where(_all['eligibility'].eq('eligible'), _all['arm'].map(ARM_LABEL),
                             'excluded: ' + _all['eligibility'])

    eff = _all[_all['eligibility'].eq('eligible')].drop(columns=['_merge']).copy()
    for _bc in ['debt_band', 'age_band']:
        if _bc in eff.columns and hasattr(eff[_bc], 'cat'):
            eff[_bc] = eff[_bc].cat.remove_unused_categories()
    eff['reached'] = eff['reach_status'].eq('reached')          # True only with evidence of contact
    for _dc in ['first_reach_date', 'help_date']:
        eff[_dc] = pd.to_datetime(eff[_dc], errors='coerce')
    eff['days_to_reach'] = (eff['first_reach_date'] - eff['signal_date']).dt.days   # window: never negative
    eff['close_year'] = eff['close_date'].dt.year
    _RESOLVING = {'paid', 'payment arrangement', 'self-resolving', 'quick fix'}
    eff['arm_clean'] = np.where(eff['arm'].eq('B_refused'),
                                np.where(eff['result_class'].isin(_RESOLVING), 'B1_resolving', 'B2_difficulty'),
                                eff['arm'])

    # ── Signal-level survival frame (for A4: non-forwarded signals never become cases) ──
    _ss = signalen_x.dropna(subset=['Hash', 'Datum melding']).copy()
    _ss['signal_date'] = pd.to_datetime(_ss['Datum melding'], errors='coerce')
    _ss = _ss.dropna(subset=['signal_date'])
    def _next_after(h, t):
        arr = _sig_by_hash.get(h)
        if not arr:
            return pd.NaT
        j = bisect.bisect_right(arr, t + pd.Timedelta(days=MIN_GAP_DAYS))
        return arr[j] if j < len(arr) else pd.NaT
    _ss['next_date'] = pd.to_datetime(pd.Series([_next_after(h, t) for h, t in zip(_ss['Hash'], _ss['signal_date'])],
                                                index=_ss.index, dtype=object), errors='coerce')
    _ss['event'] = _ss['next_date'].notna().astype(int)
    _ss['duration'] = (_ss['next_date'].fillna(EXPORT_DATE) - _ss['signal_date']).dt.days
    sigsurv = _ss[_ss['duration'] > 0].copy()
    for _bc in ['debt_band', 'age_band']:
        if _bc in sigsurv.columns and hasattr(sigsurv[_bc], 'cat'):
            sigsurv[_bc] = sigsurv[_bc].cat.remove_unused_categories()

    # ── Observed prior history (possible now that the export spans 2021-2026) ─────
    def _prior_sigs(h, t):
        arr = _sig_by_hash.get(h)
        if not arr or pd.isna(t):
            return 0
        return bisect.bisect_left(arr, t - pd.Timedelta(days=MIN_GAP_DAYS))
    eff['prior_sigs'] = [_prior_sigs(h, t) for h, t in zip(eff['Hash'], eff['signal_date'])]
    eff['prior_band'] = pd.cut(eff['prior_sigs'], bins=[-1, 0, 2, 5, 1e9],
                               labels=['0', '1-2', '3-5', '6+'])
    eff['prior_band'] = eff['prior_band'].cat.remove_unused_categories()

    # ── Shared survival helpers ───────────────────────────────────────────────────
    def cuminc_at(dur, evt, day, min_known=10):
        # Kaplan-Meier cumulative incidence at `day` (an ESTIMATE). n/a when fewer than `min_known`
        # cases have a known status by `day` (event by then, or still observed at `day`), or when the
        # curve cannot be read at `day`. 0% (no events) and 100% (estimated survival reached 0 by
        # `day`) are valid estimates; 100% does not mean every censored case was seen to return.
        dur = np.asarray(dur, float); evt = np.asarray(evt, int)
        ok = ~np.isnan(dur); dur, evt = dur[ok], evt[ok]
        if len(dur) == 0:
            return np.nan
        ev_by = (evt == 1) & (dur <= day)
        if int((ev_by | (dur >= day)).sum()) < min_known:
            return np.nan
        if not ev_by.any():
            return 0.0
        sf = SurvfuncRight(dur, evt)
        idx = np.searchsorted(sf.surv_times, day, side='right') - 1
        s_day = 1.0 if idx < 0 else float(sf.surv_prob[idx])
        if s_day <= 0:
            return 1.0                         # Kaplan-Meier survival estimate reached 0 by `day`
        if not (dur >= day).any() and ((evt == 0) & (dur < day)).any():
            return np.nan                      # nobody left at `day` and the curve has not reached 0
        return 1 - s_day

    def cuminc_str(dur, evt, day, width=7):
        v = cuminc_at(dur, evt, day)
        return f'{v*100:{width}.1f}%' if pd.notna(v) else f'{"n/a":>{width}} '

    def _ph_report(r, status):
        # EXPLORATORY time-trend check of a Cox model - not a formal proportional-hazards test:
        # correlation of Schoenfeld residuals with the rank of event time, events only. No p-values
        # (they would ignore that one household's cases are related).
        try:
            sch = np.asarray(r.schoenfeld_residuals, float)
            dur = np.asarray(r.model.endog, float)
            ev = (np.asarray(status, int) == 1) & ~np.isnan(sch).any(axis=1)
            _rk = _st.rankdata(dur[ev])
            rows = [(name, _st.pearsonr(sch[ev, j], _rk)[0]) for j, name in enumerate(r.model.exog_names)]
            _t = pd.DataFrame(rows, columns=['term', 'rho (residual vs time rank)']).set_index('term')
            print('  Exploratory time-trend check (not a formal test; no p-values): rho near 0 = little sign')
            print('  that the HR changes over follow-up; a larger |rho| = read that HR as an average over time.')
            display(_t.round(3))
        except Exception as _e:
            print(f'  Exploratory time-trend check not available: {_e}')

    def _cox_fit(formula, data, status, groups=None, strata=None, ph_check=False, return_cov=False):
        # Cox fit -> HR table (household-clustered sandwich CIs when groups are given).
        # Gates - any failure RAISES, so callers print 'not estimable' and show NO table:
        #   a design that is not identifiable (constant or collinear columns; within strata when
        #   stratified), a statsmodels convergence / Hessian warning during the fit, diverging
        #   (|log HR| > 15) or non-finite estimates or standard errors.
        from statsmodels.tools import sm_exceptions as _smx
        kw = {'strata': np.asarray(strata)} if strata is not None else {}
        mod = PHReg.from_formula(formula, data=data, status=status, **kw)
        X = np.asarray(mod.exog, float)
        if X.shape[1] == 0:
            raise ValueError('no estimable terms')
        Xc = (X - pd.DataFrame(X).groupby(np.asarray(strata)).transform('mean').values if strata is not None
              else X - X.mean(axis=0))
        if np.linalg.matrix_rank(Xc) < X.shape[1]:
            raise ValueError('design not identifiable (constant or collinear terms)')
        with warnings.catch_warnings(record=True) as _w:
            warnings.simplefilter('always')
            r = mod.fit()
        _bad = [w for w in _w if issubclass(w.category, (_smx.ConvergenceWarning, _smx.HessianInversionWarning))]
        if _bad:
            raise ValueError(f'did not converge ({str(_bad[0].message)[:80]})')
        b = np.asarray(r.params, float); cov = np.asarray(r.cov_params(), float)
        if groups is not None:
            sr = np.asarray(r.model.score_residuals(r.params), float)
            bad = np.isnan(sr).any(axis=1)
            if bad.any():
                # statsmodels leaves residuals undefined for cases never at risk when an event happened
                # in their stratum; they carry no information (contribution 0). Anything else is an error.
                dur = np.asarray(r.model.endog, float); st = np.asarray(status, int)
                stv = np.asarray(strata) if strata is not None else np.zeros(len(dur))
                first_ev = pd.Series(dur[st == 1]).groupby(stv[st == 1]).min()
                fe = pd.Series(stv).map(first_ev).astype(float).fillna(np.inf).values
                if not np.all(dur[bad] < fe[bad]):
                    raise ValueError('unexpected undefined score residuals')
                sr[bad] = 0.0
            U = pd.DataFrame(sr).groupby(np.asarray(groups)).sum().values
            cov = cov @ (U.T @ U) @ cov
        se = np.sqrt(np.diag(cov))
        if not (np.all(np.isfinite(b)) and np.all(np.isfinite(se))) or np.abs(b).max() > 15:
            raise ValueError('estimates diverge or are not finite (separation / collinearity)')
        tab = pd.DataFrame({'HR': np.exp(b), 'CI2.5%': np.exp(b - 1.96 * se),
                            'CI97.5%': np.exp(b + 1.96 * se), 'p': 2 * _st.norm.sf(np.abs(b / se))},
                           index=r.model.exog_names)
        if ph_check:
            _ph_report(r, status)
        if return_cov:
            return tab, b, cov, list(r.model.exog_names)
        return tab

    def cox_hr(data, formula, covars, event_col='event', dur_col='duration', cluster_col='Hash', ph_check=False):
        # Cox HRs; CIs/p-values clustered by household (cluster_col). No table when not estimable.
        d = data.dropna(subset=[dur_col, event_col] + covars).copy()
        for _c in covars:
            if hasattr(d[_c], 'cat'):
                d[_c] = d[_c].cat.remove_unused_categories()
        if len(d) < 40 or int(d[event_col].sum()) < 5:
            print('  (Cox skipped - too few cases/events)'); return None
        try:
            _groups = d[cluster_col].values if cluster_col and cluster_col in d.columns else None
            return _cox_fit(formula, d, d[event_col].values, groups=_groups, ph_check=ph_check).round(3)
        except Exception as _e:
            print(f'  Cox not estimable: {_e}'); return None

    def _logit_fit(formula, data, cluster=None):
        # Logistic fit -> (model, None) or (None, reason). No silent fallbacks: a model that does not
        # converge, has diverging coefficients (separation) or non-finite standard errors is reported
        # as not estimable instead of being refitted another way.
        import statsmodels.formula.api as smf
        try:
            kw = ({'cov_type': 'cluster', 'cov_kwds': {'groups': pd.factorize(data[cluster])[0]}}
                  if cluster is not None else {})
            m = smf.logit(formula, data=data).fit(disp=0, maxiter=100, **kw)
        except Exception as _e:
            return None, f'{type(_e).__name__}: {_e}'
        if not m.mle_retvals.get('converged', True):
            return None, 'did not converge'
        if np.abs(np.asarray(m.params, float)).max() > 15:
            return None, 'coefficients diverge (perfect or quasi-separation)'
        if not np.all(np.isfinite(np.asarray(m.bse, float))):
            return None, 'standard errors not finite (separation or collinearity)'
        return m, None

    def _wald(b, cov, idx):
        # joint Wald test of the coefficients in idx (household-clustered covariance)
        bi = np.asarray(b, float)[idx]; Vi = np.asarray(cov, float)[np.ix_(idx, idx)]
        k = int(np.linalg.matrix_rank(Vi))
        stat = float(bi @ np.linalg.pinv(Vi) @ bi)
        return stat, k, float(_st.chi2.sf(stat, k))

    _ARM_T = 'C(arm, Treatment(reference="A_not_reached"))'
    _ARM_ROWS = [('B_refused', 'B vs A (declined vs not reached)'), ('C_accepted', 'C vs A (accepted vs not reached)')]

    def _fit_any(kind, rhs, d, lhs, event_col):
        # -> (params Series, cov DataFrame), household-clustered; raises when not estimable
        if kind == 'logit':
            m, why = _logit_fit(f'{lhs} ~ {rhs}', d, cluster='Hash')
            if m is None:
                raise ValueError(why)
            return m.params, m.cov_params()
        _, b, cov, names = _cox_fit(f'{lhs} ~ {rhs}', d, d[event_col].values, groups=d['Hash'].values,
                                    return_cov=True)
        return pd.Series(b, index=names), pd.DataFrame(cov, index=names, columns=names)

    def compare_by_calendar(kind, data, lhs, covars_f, covars, year_col, title, event_col='event', ph_primary=False):
        # Pre-specified layout, always reported in full: pooled | PRIMARY year-adjusted | each period |
        # group x period interaction (household-clustered). kind: 'cox' (lhs = duration) or 'logit'.
        est = 'HR' if kind == 'cox' else 'OR'
        print(f'\n{title} - {est} by group (ref = A not reached); household-clustered 95% CIs; associations, not effects')
        d = data[data['arm'].isin(ARMS_ABC)].dropna(subset=[lhs, year_col] + covars).copy()
        if kind == 'cox':
            d = d.dropna(subset=[event_col])
        if len(d) < 200 or d['arm'].nunique() < 2:
            print('  Too few cases for the group comparison - skipped.'); return None
        d['_year'] = d[year_col].astype(int).astype(str)
        d['_period'] = d[year_col].astype(int).map(cal_period)
        base = _ARM_T + (' + ' + covars_f if covars_f else '')
        specs = [('pooled (no calendar)', d, base), ('PRIMARY: year-adjusted', d, base + ' + C(_year)')]
        specs += [(f'period {p}', d[d['_period'] == p], base) for p in sorted(d['_period'].unique())]
        n_lab = 'events' if kind == 'cox' else 'outcome = 1'
        cols, primary = {}, None
        for name, sub, rhs in specs:
            sub = sub.copy()
            for c in covars:
                if hasattr(sub[c], 'cat'):
                    sub[c] = sub[c].cat.remove_unused_categories()
            n1 = int(sub[event_col].sum()) if kind == 'cox' else int(sub[lhs].sum())
            col = {'cases': f'{len(sub):,}', n_lab: f'{n1:,}'}
            try:
                if len(sub) < 200 or n1 < 30 or (kind == 'logit' and len(sub) - n1 < 30):
                    raise ValueError('too few cases/events')
                prm, cov = _fit_any(kind, rhs, sub, lhs, event_col)
                for code, label in _ARM_ROWS:
                    term = f'{_ARM_T}[T.{code}]'
                    if term in prm.index:
                        b, se = float(prm[term]), float(np.sqrt(cov.loc[term, term]))
                        col[label] = (f"{np.exp(b):.2f} ({np.exp(b - 1.96 * se):.2f}-{np.exp(b + 1.96 * se):.2f}) "
                                      f"p={2 * _st.norm.sf(abs(b / se)):.3f}")
                    else:
                        col[label] = 'n/a (no cases in this group)'
                if name.startswith('PRIMARY'):
                    primary = (prm, cov, sub, rhs)
            except Exception as _e:
                print(f'  {name}: not estimable ({_e})')
                for _, label in _ARM_ROWS:
                    col[label] = 'not estimable'
            cols[name] = col
        display(pd.DataFrame(cols).reindex([lab for _, lab in _ARM_ROWS] + ['cases', n_lab]))
        try:
            if d['_period'].nunique() < 2:
                raise ValueError('only one period')
            prm, cov = _fit_any(kind, f'{_ARM_T} * C(_period)' + (' + ' + covars_f if covars_f else ''), d, lhs, event_col)
            idx = [i for i, t in enumerate(prm.index) if ':' in t and 'arm' in t and '_period' in t]
            if not idx:
                raise ValueError('no interaction terms')
            stat, k, p = _wald(prm.values, cov.values, idx)
            print(f'  Group x period interaction (household-clustered Wald test): chi2={stat:.2f}, df={k}, p={p:.3f}')
            print('  (small p = the group difference is not the same in every period - read the per-period')
            print('   columns; a large p does not prove that it is the same)')
        except Exception as _e:
            print(f'  Group x period interaction: not estimable ({_e})')
        print('  PRIMARY (pre-specified) = year-adjusted; every column is reported each run, whatever it shows.')
        if primary is not None:
            prm, cov, sub_p, rhs_p = primary
            se = np.sqrt(np.diag(cov.values))
            full = pd.DataFrame({est: np.exp(prm.values), 'CI2.5%': np.exp(prm.values - 1.96 * se),
                                 'CI97.5%': np.exp(prm.values + 1.96 * se),
                                 'p': 2 * _st.norm.sf(np.abs(prm.values / se))}, index=prm.index)
            full = full.drop(index='Intercept', errors='ignore')
            full = full[~full.index.str.startswith('C(_year)')]
            print('  PRIMARY model - all terms (calendar-year terms not listed):')
            display(full.round(3))
            if kind == 'cox' and ph_primary:
                try:
                    _cox_fit(f'{lhs} ~ {rhs_p}', sub_p, sub_p[event_col].values, ph_check=True)
                except Exception as _e:
                    print(f'  Exploratory time-trend check not available: {_e}')
        return cols

    # ── Summary: eligibility flow, joint audit, effect of the time window ─────────
    print('\nEligibility flow (closed cases -> effectiveness frame):')
    print(f"  {'cases in Meldingen':<52} {_n_cases_all:>7,}")
    _rem = len(_all)
    print(f"  {'closed, with a close date':<52} {_rem:>7,}")
    for _r in _REASONS:
        _rem -= int((_all['eligibility'] == _r).sum())
        print(f"  {'minus ' + _r:<52} {_rem:>7,}")
    if len(_all):
        print('\nCase-level audit - every closed case by group or exclusion reason, final-result class and')
        print('reach status (dated records between signal and closure; totals reconcile with the flow):')
        display(_all.groupby(['group', 'result_class', 'reach_status']).size().rename('cases').reset_index()
                .sort_values(['group', 'cases'], ascending=[True, False]).reset_index(drop=True))
        _el = _all[_all['eligibility'].eq('eligible')]
        if len(_el):
            print('Effect of the time window on group assignment (eligible cases; rows = group if ALL records')
            print('were used, columns = group from dated records between signal and closure - the rule used):')
            display(pd.crosstab(_el['arm_allrec'].map(ARM_LABEL), _el['arm'].map(ARM_LABEL),
                                margins=True, margins_name='total'))
            _chg = int((_el['arm_allrec'] != _el['arm']).sum())
            print(f"  cases whose group changes because of the time window: {_chg:,} of {len(_el):,} "
                  f"({_chg / len(_el) * 100:.1f}%)")

    print(f"\nCases in the effectiveness frame: {len(eff):,}")
    if len(eff):
        print(f"  returned (event, secondary outcome): {int(eff['event'].sum()):,} ({eff['event'].mean()*100:.1f}%)")
        print(f"  no return observed through cutoff:   {int((eff['event'] == 0).sum()):,}"
              "  (retained as signal-free; administratively censored)")
        print(f"  median days from case close to the data cutoff (time available to see a return): "
              f"{(EXPORT_DATE - eff['close_date']).dt.days.median():.0f}")
        print(f"  median days to return or cutoff: {eff['duration'].median():.0f}"
              "  (short because most returns come quickly)")
        print("  contact groups (A/B/C are compared; U and X are counted but kept out of the comparison):")
        for _c in ['A_not_reached', 'B_refused', 'C_accepted', 'U_reach_unknown', 'U_help_unknown', 'X_in_care']:
            print(f"    {ARM_LABEL[_c]:<50} {int((eff['arm'] == _c).sum()):>7,}")
        eff['_close_yr'] = eff['close_date'].dt.year
        if eff['_close_yr'].nunique() > 1:
            print("  90-day return by case-close year (secondary outcome; Kaplan-Meier estimates; see 9d):")
            for _yy, _gy in eff.groupby('_close_yr'):
                if len(_gy) >= 50:
                    print(f"    {int(_yy)}: {cuminc_str(_gy['duration'], _gy['event'], 90, 5)}  (n={len(_gy):,})")
    else:
        print("  No eligible closed cases - check the export (needs Meldingnummer on signals + close dates).")
    _section_done('9. Effectiveness Framework — data preparation')
except Exception as _err:
    _section_failed('9. Effectiveness Framework — data preparation', _err)


### 9a. Experiment A1+A2 — Does contact help even when help is refused?

Uses the **secondary outcome** (signal >14 days after case closure); the main, same-clock
comparison is §9c. Closed cases are split into three groups: **A** not reached, **B** reached but
declined help, **C** reached and accepted help (shared definitions, §1b; unknown statuses are
left out). Each group gets a time-to-return curve plus the pre-specified calendar-aware
comparison (pooled, PRIMARY year-adjusted, per period, interaction) and an exploratory time-trend
check of the primary Cox model.

The key contrast is **B vs A**: if households that declined help still return less than households
never reached, that would be consistent with contact *itself* having value — a "wake-up"
effect — but A and B differ in more than contact. **A2** asks how much of that
comes from households that declined but whose final result says they were resolving it (B1:
paid, payment arrangement, self-resolving, quick fix). B1 is identified from the *final result*,
which the contact itself may have influenced, so A2 shows **both** versions (all who declined,
and without B1). They are two versions of the comparison — not bounds on an effect, and A
differs from B in more ways than contact.


In [ ]:
_report_section('9. Contact Decomposition - Does Contact Reduce Return?')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run

    _arms = ['A_not_reached', 'B_refused', 'C_accepted']
    _present = [a for a in _arms if (eff['arm'] == a).sum() > 0]

    if len(eff) == 0 or int(eff['event'].sum()) == 0 or len(_present) < 2:
        print('No return events or fewer than two observed arms to compare.')
    else:
        fig, ax = plt.subplots(figsize=(11, 6))
        for a in _present:
            g = eff[eff['arm'] == a]
            sf = SurvfuncRight(g['duration'].values, g['event'].values)
            ax.step(sf.surv_times, (1 - sf.surv_prob) * 100, where='post',
                    linewidth=2, label=f'{ARM_LABEL[a]} (n={len(g)})')
        ax.set_xlim(0, 730)
        ax.set_xlabel('Days since case close'); ax.set_ylabel('Cumulative return %')
        ax.set_title('Return after case close, by contact arm')
        ax.legend()
        plt.tight_layout(); plt.show()

        _HZ = [30, 90, 180, 270, 365, 540, 730]
        print('Cumulative return % by arm (values behind the curves):')
        print('  ' + 'arm'.ljust(20) + ''.join(f'{h}d'.rjust(8) for h in _HZ) + '       n')
        for a in _present:
            g = eff[eff['arm'] == a]
            print('  ' + ARM_LABEL[a].ljust(20)
                  + ''.join(cuminc_str(g['duration'], g['event'], h) for h in _HZ)
                  + f'{len(g):8d}')
        print('  (n/a = fewer than 10 cases with a known status at that horizon, or not estimable;')
        print('   percentages are Kaplan-Meier estimates)')
        _small = [ARM_LABEL[a] for a in _present if (eff['arm'] == a).sum() < 50]
        if _small:
            print(f"  (small arm{'s' if len(_small) > 1 else ''} — interpret with caution: {', '.join(_small)})")

        try:
            _sub = eff[eff['arm'].isin(_present)]
            _chi, _p = survdiff(_sub['duration'].values, _sub['event'].values, _sub['arm'].values)
            print(f'\nLog-rank across arms: chi2={_chi:.2f}, p={_p:.4f}')
            print('  (log-rank treats every case as independent - prefer the clustered Cox p below)')
        except Exception as _e:
            print(f'Log-rank failed: {_e}')

        # HR < 1 = fewer / slower returns than A. Calendar handling is pre-specified (see 9).
        compare_by_calendar('cox', eff, 'duration', 'recidive_hist + C(prior_band) + C(debt_band) + C(age_band)',
                            ['recidive_hist', 'prior_band', 'debt_band', 'age_band'], 'close_year',
                            'Secondary outcome (return after case close, Cox)', ph_primary=True)

        # ── A2: declined help, with and without B1 ────────────────────────────────
        # B1 (final result says resolving) is identified from the FINAL result, which the contact
        # itself may have influenced. Two versions of the comparison - not bounds on an effect.
        print('\nA2 - Declined help, with and without B1 (B1 = final result says resolving):')
        _a2 = [(eff['arm'] == 'A_not_reached', 'A - not reached (reference)'),
               (eff['arm'] == 'B_refused', 'B - all who declined'),
               (eff['arm_clean'] == 'B2_difficulty', 'B2 - declined, other final results'),
               (eff['arm'] == 'C_accepted', 'C - help accepted')]
        for _mask, _lab in _a2:
            g = eff[_mask]
            if len(g) > 0:
                print(f'  {_lab:<40} 90d return={cuminc_str(g["duration"], g["event"], 90, 5)} (n={len(g)})')
        print('  B1 is defined from the final result, which contact may have influenced. Neither row is')
        print('  an effect estimate, and A differs from B in more ways than contact (associations only).')

        if eff['close_date'].dt.year.nunique() > 1:
            _yrows = []
            for _yy, _gy in eff.groupby(eff['close_date'].dt.year):
                _r = {'year': int(_yy)}
                for a in _present:
                    _gg = _gy[_gy['arm'] == a]
                    _v = cuminc_at(_gg['duration'], _gg['event'], 90) if len(_gg) > 0 else np.nan
                    _r[ARM_LABEL[a]] = round(_v * 100, 1) if pd.notna(_v) else np.nan
                _yrows.append(_r)
            print('\n90-day return % by arm and close year (all observed groups; NaN = insufficient follow-up):')
            print(pd.DataFrame(_yrows).set_index('year').to_string())
    _section_done('9. Contact Decomposition - Does Contact Reduce Return?')
except Exception as _err:
    _section_failed('9. Contact Decomposition - Does Contact Reduce Return?', _err)


### 9b. Chained Cases — Contact Method × Final Result Patterns

This extends the contact-arm comparison by combining the method that first achieved contact
with the final result of that case. It first estimates return after each method–result
combination, then orders repeated closed cases within each household and describes which
method/result combinations tend to be followed by which combinations in the next case.

`reach_method` is the **first successful** contact (an outbound channel, or "callback after …"
for an inbound contact); not-reached cases are shown as `(not reached)` and cases with unknown
or conflicting reach as `(reach unknown)`. A chained next case must start
more than `MIN_GAP_DAYS` after the current case closes, matching the return definition in §9.
All observed combinations are retained and their sample sizes are shown. These are pattern and
association estimates—not causal effects—because method choice and final result depend on case
difficulty, household circumstances, and municipal workflow.


In [ ]:
_report_section('9b. Chained Cases - Contact Method x Final Result Patterns')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run

    if len(eff) == 0:
        print('No closed, linked cases available for the chained-case analysis.')
    else:
        _chain9 = eff.sort_values(['Hash', 'signal_date', 'close_date']).copy()
        _method_raw9 = (_chain9['reach_method'] if 'reach_method' in _chain9.columns
                        else pd.Series(np.nan, index=_chain9.index))
        _result_raw9 = (_chain9['eindresultaat'] if 'eindresultaat' in _chain9.columns
                        else pd.Series(np.nan, index=_chain9.index))
        _chain9['_method_state'] = (_method_raw9.fillna('(method not recorded)')
                                    .astype(str).replace(r'^\s*$', '(method not recorded)', regex=True))
        _chain9.loc[_chain9['reach_status'].eq('not reached'), '_method_state'] = '(not reached)'
        _chain9.loc[_chain9['reach_status'].isin(['unknown', 'conflicting']), '_method_state'] = '(reach unknown)'
        _chain9['_result_state'] = (_result_raw9.fillna('(not recorded)')
                                    .astype(str).replace(r'^\s*$', '(not recorded)', regex=True))
        _chain9['case_order'] = _chain9.groupby('Hash').cumcount() + 1
        _chain9['next_case_signal_date'] = _chain9.groupby('Hash')['signal_date'].shift(-1)
        _chain9['next_method_state'] = _chain9.groupby('Hash')['_method_state'].shift(-1)
        _chain9['next_result_state'] = _chain9.groupby('Hash')['_result_state'].shift(-1)
        _chain9['days_to_next_case_start'] = (
            _chain9['next_case_signal_date'] - _chain9['signal_date']).dt.days
        _chain9['days_close_to_next_case'] = (
            _chain9['next_case_signal_date'] - _chain9['close_date']).dt.days

        # Joint association with the next signal, retaining every observed method-result combination.
        _joint9_rows = []
        for (_meth9, _res9), _g9 in _chain9.groupby(['_method_state', '_result_state'], dropna=False):
            _v90 = cuminc_at(_g9['duration'], _g9['event'], 90)
            _v365 = cuminc_at(_g9['duration'], _g9['event'], 365)
            _joint9_rows.append({
                'contact method': _meth9, 'final result': _res9,
                'cases': len(_g9), 'households': _g9['Hash'].nunique(),
                'return events': int(_g9['event'].sum()),
                'observed through 90d': int((_g9['duration'] >= 90).sum()),
                'return_90d_%': _v90 * 100 if pd.notna(_v90) else np.nan,
                'observed through 365d': int((_g9['duration'] >= 365).sum()),
                'return_365d_%': _v365 * 100 if pd.notna(_v365) else np.nan,
            })
        _joint9 = (pd.DataFrame(_joint9_rows)
                   .sort_values(['return_90d_%', 'cases'], ascending=[False, False], na_position='last'))
        print('Joint method x final-result association with a subsequent signal:')
        print('  Return percentages use the same censoring-aware estimator as §9a.')
        print('  n/a means that the fixed horizon is not sufficiently observed; the group is still retained.')
        display(_joint9.round({'return_90d_%': 1, 'return_365d_%': 1}))

        # Heatmap: method x result, with every combination's case count visible.
        _ret9 = _joint9.pivot(index='contact method', columns='final result', values='return_90d_%')
        _num9 = _joint9.pivot(index='contact method', columns='final result', values='cases').fillna(0)
        print('\nCase counts behind the method x final-result matrix:')
        display(_num9.astype(int))
        if _ret9.size and _ret9.notna().any().any():
            _ann9 = _ret9.copy().astype(object)
            for _i9 in range(len(_ret9.index)):
                for _j9 in range(len(_ret9.columns)):
                    _n9 = int(_num9.iloc[_i9, _j9])
                    _v9 = _ret9.iloc[_i9, _j9]
                    _ann9.iloc[_i9, _j9] = (f'{_v9:.1f}%\n(n={_n9})'
                                              if pd.notna(_v9) else '')
            _w9 = min(26, max(11, 1.6 * len(_ret9.columns)))
            _h9 = min(18, max(4, 0.9 * len(_ret9.index)))
            fig, ax = plt.subplots(figsize=(_w9, _h9))
            sns.heatmap(_ret9, mask=_ret9.isna(), annot=_ann9, fmt='', cmap='YlOrRd',
                        vmin=0, vmax=100, linewidths=.5,
                        cbar_kws={'label': '90-day cumulative return %'}, ax=ax)
            for _i9 in range(len(_ret9.index)):
                for _j9 in range(len(_ret9.columns)):
                    if pd.isna(_ret9.iloc[_i9, _j9]) and int(_num9.iloc[_i9, _j9]) > 0:
                        ax.text(_j9 + .5, _i9 + .5, f'n/a\n(n={int(_num9.iloc[_i9, _j9])})',
                                ha='center', va='center', fontsize=8, color='gray')
            ax.set_title('90-day return by successful contact method and final result')
            ax.set_xlabel('Final result of current case')
            ax.set_ylabel('First successful contact method')
            ax.tick_params(axis='x', labelrotation=45)
            for _lab9 in ax.get_xticklabels():
                _lab9.set_horizontalalignment('right')
            plt.tight_layout(); plt.show()

        # Case chains: only a genuinely later case under the same >14-day return convention.
        _has_next9 = _chain9['next_case_signal_date'].notna()
        _valid_next9 = _has_next9 & (_chain9['days_close_to_next_case'] > MIN_GAP_DAYS)
        _seq9 = _chain9[_valid_next9].copy()
        print(f'\nPotential consecutive closed-case pairs: {int(_has_next9.sum()):,}')
        print(f'Valid post-close chains (next case starts >{MIN_GAP_DAYS}d after close): {len(_seq9):,}')
        print(f'Concurrent/nearby pairs excluded from chain transitions: {int((_has_next9 & ~_valid_next9).sum()):,}')

        def _transition9(data, current, nxt, label):
            _tn9 = pd.crosstab(data[current], data[nxt], dropna=False)
            if _tn9.size == 0:
                print(f'\nNo {label} transitions available.')
                return
            _tp9 = _tn9.div(_tn9.sum(axis=1), axis=0) * 100
            print(f'\n{label} transition counts (current case rows -> next case columns):')
            display(_tn9)
            print(f'{label} transition row percentages (each current-state row sums to 100%):')
            display(_tp9.round(1))

        if len(_seq9):
            _transition9(_seq9, '_method_state', 'next_method_state', 'Contact-method')
            _transition9(_seq9, '_result_state', 'next_result_state', 'Final-result')

            _paths9 = (_seq9.groupby(['_method_state', '_result_state',
                                             'next_method_state', 'next_result_state'], dropna=False)
                       .agg(transitions=('Hash', 'size'), households=('Hash', 'nunique'),
                            median_days_between_case_starts=('days_to_next_case_start', 'median'))
                       .reset_index()
                       .rename(columns={'_method_state': 'current method',
                                        '_result_state': 'current result',
                                        'next_method_state': 'next method',
                                        'next_result_state': 'next result'})
                       .sort_values(['transitions', 'current method', 'current result'],
                                    ascending=[False, True, True]))
            print('\nComplete method/result paths across consecutive cases (all observed paths):')
            display(_paths9.round({'median_days_between_case_starts': 1}))
        else:
            print('No valid post-close case chains are available.')

        print('\nInterpretation: recurring patterns identify combinations worth investigating, not effects.')
        print('Method choice and final result are selected by case difficulty and workflow; late chains')
        print('are also administratively censored near the export date.')
    _section_done('9b. Chained Cases - Contact Method x Final Result Patterns')
except Exception as _err:
    _section_failed('9b. Chained Cases - Contact Method x Final Result Patterns', _err)


### 9c. Same-Clock Check — outcome measured from the signal date (MAIN OUTCOME)

This is the **main comparison** of the contact groups (A/B/C, shared definitions §1b).

In this export almost every signal becomes its own case, cases close about a month after their
signal, and creditors re-send an unpaid debt about monthly. "Return after case close" (§9a)
therefore depends partly on *when in the monthly cycle* a case was closed: if one group's cases
close a few days later, the next monthly report falls inside or outside the 14-day buffer, and
the groups get different "return" figures even when nothing about the households differs.

This check gives every case the **same clock**: time zero is the case's **signal date**, contact
and closure must be finished by **day 60**, and the outcome window is **day 61–365**. Each case
falls into exactly one group:
- **quiet** — no signal at all in day 61–365;
- **still reported** — reports never paused for more than 45 days, so the problem ran on past
  day 60 (same rule as the §13 episodes);
- **came back** — reports stopped by day 60, then a new episode started before day 365.

Only cases with the full year observed and closed by day 60 are included; their groups come from
dated records between the signal and closure (so before day 60). Two logistic outcomes compare
the groups — any signal in day 61–365, and, among cases whose reports had stopped by day 60,
coming back — each in the pre-specified calendar layout (pooled, PRIMARY year-adjusted, per
period, interaction; household-clustered).

In [ ]:
_report_section('9c. Same-Clock Check - outcome measured from the signal date')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    CLOCK_LANDMARK = 60    # contact and case closure must be finished by this day after the signal
    CLOCK_END      = 365   # outcome window: day CLOCK_LANDMARK+1 .. CLOCK_END after the signal

    # Quick check behind the issue: do the arms close at different times after their signal?
    _dc9 = eff.assign(days_to_close=(eff['close_date'] - eff['signal_date']).dt.days)
    _dc9 = _dc9[_dc9['days_to_close'] >= 0]
    _dt9 = _dc9.groupby('arm')['days_to_close'].agg(cases='count', p25=lambda s: s.quantile(.25),
                                                    median='median', p75=lambda s: s.quantile(.75))
    _dt9.index = [ARM_LABEL.get(i, i) for i in _dt9.index]
    print('Days from signal to case close, by arm (a different closing time shifts "return after close"):')
    display(_dt9.round(0))

    def _run_end(arr, t0, gap):
        # last signal of the uninterrupted run that starts at t0 (each next signal <= gap days later)
        cur = t0
        j = bisect.bisect_right(arr, cur)
        while j < len(arr) and (arr[j] - cur).days <= gap:
            cur = arr[j]; j += 1
        return cur

    _c9 = eff[eff['arm'].isin(ARMS_ABC)].dropna(subset=['Hash', 'signal_date', 'close_date']).copy()
    _c9['days_to_close'] = (_c9['close_date'] - _c9['signal_date']).dt.days
    _n9 = len(_c9)
    _c9 = _c9[_c9['signal_date'] + pd.Timedelta(days=CLOCK_END) <= EXPORT_DATE]
    _n9full = len(_c9)
    _c9 = _c9[_c9['days_to_close'].between(0, CLOCK_LANDMARK)]
    print(f'\nCases in the same-clock check: {len(_c9):,} of {_n9:,} closed cases in groups A/B/C')
    print(f'  left out: {_n9 - _n9full:,} without a full year of data after the signal, '
          f'{_n9full - len(_c9):,} closed after day {CLOCK_LANDMARK} (or close date before the signal)')

    if len(_c9) < 200:
        print('Too few cases with a full year of follow-up - same-clock check skipped.')
    else:
        _any9, _still9 = [], []
        for _h, _t0 in zip(_c9['Hash'], _c9['signal_date']):
            arr = _sig_by_hash.get(_h, [])
            lo = _t0 + pd.Timedelta(days=CLOCK_LANDMARK); hi = _t0 + pd.Timedelta(days=CLOCK_END)
            _any9.append(bisect.bisect_right(arr, hi) - bisect.bisect_right(arr, lo) > 0)
            _still9.append(_run_end(arr, _t0, EPISODE_GAP_DAYS) > lo)
        _any9 = np.array(_any9, dtype=bool); _still9 = np.array(_still9, dtype=bool)
        _c9['any_signal'] = _any9.astype(int)
        _c9['clock_state'] = np.select([~_any9, _still9], ['quiet', 'still reported'], default='came back')

        _st9 = pd.crosstab(_c9['arm'].map(ARM_LABEL), _c9['clock_state'], normalize='index') * 100
        _st9 = _st9.reindex(columns=['quiet', 'still reported', 'came back'], fill_value=0)
        _st9['cases'] = _c9['arm'].map(ARM_LABEL).value_counts()
        print(f'\nWhat happened in day {CLOCK_LANDMARK + 1}-{CLOCK_END} after the signal, by arm (% of cases):')
        display(_st9.round(1))
        _c9['_syr'] = _c9['signal_date'].dt.year
        _py9 = _c9.groupby(['_syr', 'arm'])['any_signal'].agg(cases='size', pct='mean').reset_index()
        _py9['pct'] = (_py9['pct'] * 100).round(1)
        print(f'\nAny signal in day {CLOCK_LANDMARK + 1}-{CLOCK_END} by signal year and group (%, then cases) -')
        print('read per period if the calendar check (9d) shows a break:')
        print(_py9.pivot(index='_syr', columns='arm', values='pct').rename(columns=ARM_LABEL).to_string())
        print(_py9.pivot(index='_syr', columns='arm', values='cases').rename(columns=ARM_LABEL).to_string())
        print('quiet = no signal | still reported = reports never paused >45 days, the problem ran on')
        print(f'past day {CLOCK_LANDMARK} | came back = reports stopped by day {CLOCK_LANDMARK}, then a new episode started')

        _cov9f = 'recidive_hist + C(prior_band) + C(debt_band) + C(age_band)'
        _cov9 = ['recidive_hist', 'prior_band', 'debt_band', 'age_band']
        compare_by_calendar('logit', _c9, 'any_signal', _cov9f, _cov9, '_syr',
                            f'MAIN OUTCOME - any signal in day {CLOCK_LANDMARK + 1}-{CLOCK_END} after the signal')
        _c9['came_back'] = (_c9['clock_state'] == 'came back').astype(int)
        compare_by_calendar('logit', _c9[_c9['clock_state'] != 'still reported'], 'came_back', _cov9f, _cov9,
                            '_syr', f'Came back (cases whose reports had stopped by day {CLOCK_LANDMARK})')
        print('\nOR < 1 = fewer cases with that outcome than "not reached" (A), with the same clock for')
        print('everyone. If the arm differences shrink or flip compared with 9a, part of 9a was')
        print('closing-time timing rather than a difference between the households.')
    _section_done('9c. Same-Clock Check - outcome measured from the signal date')
except Exception as _err:
    _section_failed('9c. Same-Clock Check - outcome measured from the signal date', _err)


### 9d. Calendar Comparability Check

The 90-day "return after close" rate changes sharply between years (July 2026 report: about 30%
for cases closed in 2021–2022, about 80% in 2023–2024, then 51% and 40%), in all contact groups
at once. The group comparisons in §9a and §9c already report pooled, year-adjusted (primary),
per-period and interaction results; this section helps explain the calendar pattern itself. It is
**diagnostic only** — it can narrow down, not establish, the cause:

1. **Monthly closure cohorts** with a complete 90-day window (secondary outcome), next to the
   same-clock outcome by signal month (main outcome). A sharp step in one month points to a
   process change; a gradual drift to changing conditions or case mix.
2. **By year, group and reporter type**, with case counts.
3. **Days from closure to the next signal**, especially around the 14-day exclusion threshold.
4. **Alternative exclusion windows** (7/14/30/60 days) by year and group.
5. **Where the counted return comes from**: the same reporter, the same reporter type, or
   another type.
6. **Data checks**: contact records dated after closure, help answers recorded after closure,
   cases linked to several households, and signals on the last export date.

If a break remains unexplained, results should be reported per period rather than as one
pooled number. The data owner can also be asked whether delivery schedules, case creation or
household linking changed around the break.

In [ ]:
_report_section('9d. Calendar Comparability Check')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _H9 = 90
    _cal = eff.copy()
    _cal['close_m'] = _cal['close_date'].dt.to_period('M')
    _cal['full90'] = _cal['close_date'] + pd.Timedelta(days=_H9) <= EXPORT_DATE
    _cal['ret90'] = ((_cal['event'] == 1) & (_cal['duration'] <= _H9)).astype(int)
    _cf = _cal[_cal['full90']].copy()
    print(f'Cases with a complete {_H9}-day window after closure: {len(_cf):,} of {len(_cal):,}')
    if len(_cf) < 100:
        print('Too few cases with a complete window - calendar check skipped.')
    else:
        # 1. monthly series: secondary outcome by close month, main outcome by signal month
        _ms = _cf.groupby('close_m')['ret90'].agg(cases='size', ret90='mean')
        _ms['ret90_%'] = (_ms['ret90'] * 100).round(1)
        _ms = _ms.drop(columns='ret90')
        _same = None
        if isinstance(globals().get('_c9'), pd.DataFrame) and 'any_signal' in _c9.columns and len(_c9):
            _same = (_c9.assign(sig_m=_c9['signal_date'].dt.to_period('M'))
                     .groupby('sig_m')['any_signal'].agg(cases='size', pct='mean'))
            _same['pct'] = (_same['pct'] * 100).round(1)
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(_ms.index.to_timestamp(), _ms['ret90_%'], marker='.', linewidth=1.5, color=COLORS[0],
                label=f'signal >{MIN_GAP_DAYS}d after close, within {_H9}d (by close month; secondary)')
        if _same is not None and len(_same):
            ax.plot(_same.index.to_timestamp(), _same['pct'], marker='.', linewidth=1.5, color=COLORS[1],
                    label=f'any signal day {CLOCK_LANDMARK + 1}-{CLOCK_END} after the signal (by signal month; main)')
        ax.set_ylabel('% of cases'); ax.set_ylim(0, 100); ax.legend(fontsize=9)
        ax.set_title('9d - Monthly cohorts: secondary vs main outcome')
        plt.tight_layout(); plt.show()
        print('Monthly closure cohorts (complete windows only):')
        print(_ms.to_string())
        _jump = _ms['ret90_%'].rolling(3).mean().diff(3).abs()
        if _jump.notna().any():
            print(f"Largest change between consecutive 3-month windows: around {_jump.idxmax()} "
                  f"({_jump.max():.1f} percentage points)")
        if _same is not None and len(_same):
            print('\nSame-clock outcome by signal month (main outcome, groups A/B/C):')
            print(_same.rename(columns={'pct': 'any_signal_%'}).to_string())

        # 2. by close year x group, and close year x reporter type (counts follow each table)
        _cf['_yr'] = _cf['close_date'].dt.year
        _g9 = _cf[_cf['arm'].isin(ARMS_ABC)]
        print(f'\n{_H9}-day return by close year and group (%, complete windows; cases below):')
        print((_g9.pivot_table(index='_yr', columns='arm', values='ret90', aggfunc='mean') * 100)
              .rename(columns=ARM_LABEL).round(1).to_string())
        print(_g9.pivot_table(index='_yr', columns='arm', values='ret90', aggfunc='count')
              .rename(columns=ARM_LABEL).to_string())
        _p3 = _cf.pivot_table(index='_yr', columns='Type melder', values='ret90', aggfunc='mean') * 100
        _n3 = _cf.pivot_table(index='_yr', columns='Type melder', values='ret90', aggfunc='count')
        print(f'\n{_H9}-day return by close year and reporter type of the case (%, cells n>=30; cases below):')
        print(_p3.where(_n3 >= 30).round(1).to_string())
        print(_n3.fillna(0).astype(int).to_string())

        # 3. days from closure to the household's NEXT signal (no exclusion window)
        def _next_any(h, c):
            arr = _sig_by_hash.get(h)
            if not arr:
                return pd.NaT
            i = bisect.bisect_right(arr, c)
            return arr[i] if i < len(arr) else pd.NaT
        _cf['next_any'] = pd.to_datetime(pd.Series([_next_any(h, c) for h, c in zip(_cf['Hash'], _cf['close_date'])],
                                                   index=_cf.index, dtype=object), errors='coerce')
        _dd9 = (_cf['next_any'] - _cf['close_date']).dt.days
        _cf['next_bin'] = pd.cut(_dd9, bins=[0, 7, 14, 21, 30, 45, 90, np.inf],
                                 labels=['1-7', '8-14 (not counted)', '15-21', '22-30', '31-45', '46-90', '>90'])
        _cf['next_bin'] = _cf['next_bin'].cat.add_categories(['none in data']).fillna('none in data')
        print('\nDays from case closure to the next signal of the household (% of cases per close year;')
        print(f'signals within {MIN_GAP_DAYS} days of closure are not counted as a return):')
        print((pd.crosstab(_cf['_yr'], _cf['next_bin'], normalize='index') * 100).round(1).to_string())
        print('Share of cases whose next signal falls in day 10-18, around the threshold (%):')
        print((_dd9.between(10, 18).groupby(_cf['_yr']).mean() * 100).round(1).rename('% day 10-18').to_string())

        # 4. alternative exclusion windows by close year and group
        _t4 = []
        for _gp in ([7, 14, 30, 60] if len(_g9) else []):
            _nx = []
            for _h, _c in zip(_g9['Hash'], _g9['close_date']):
                arr = _sig_by_hash.get(_h) or []
                i = bisect.bisect_right(arr, _c + pd.Timedelta(days=_gp))
                _nx.append(arr[i] if i < len(arr) else pd.NaT)
            _nxs = pd.to_datetime(pd.Series(_nx, index=_g9.index, dtype=object), errors='coerce')
            _r4 = ((_nxs - _g9['close_date']).dt.days <= _H9).astype(float)
            _tt = (_r4.groupby([_g9['close_date'].dt.year, _g9['arm']]).mean().unstack() * 100)
            _tt.columns = [ARM_LABEL[c] for c in _tt.columns]
            _tt.insert(0, 'gap', f'>{_gp}d')
            _tt.index.name = 'close year'
            _t4.append(_tt)
        if _t4:
            print(f'\n{_H9}-day return under alternative exclusion windows, by close year and group (%):')
            print(pd.concat(_t4).round(1).to_string())
        else:
            print('\nAlternative exclusion windows: no cases in groups A/B/C - skipped.')

        # 5. where does the counted return come from?
        _sr9 = signalen_x.dropna(subset=['Hash', 'Datum melding']).sort_values('Datum melding', kind='stable')
        _rep_cols = [c for c in ['Melder ID', 'Type melder'] if c in _sr9.columns]
        _rep = {h: (g['Datum melding'].tolist(),
                    g['Melder ID'].tolist() if 'Melder ID' in _rep_cols else [np.nan] * len(g),
                    g['Type melder'].tolist() if 'Type melder' in _rep_cols else [np.nan] * len(g))
                for h, g in _sr9.groupby('Hash')}
        _mids = _cf['Melder ID'] if 'Melder ID' in _cf.columns else pd.Series(np.nan, index=_cf.index)
        _src = []
        for _h, _c, _mid, _ty in zip(_cf['Hash'], _cf['close_date'], _mids, _cf['Type melder']):
            rec = _rep.get(_h)
            if not rec:
                _src.append('no return within 90d'); continue
            dts, mids, tys = rec
            i = bisect.bisect_right(dts, _c + pd.Timedelta(days=MIN_GAP_DAYS))
            if i >= len(dts) or (dts[i] - _c).days > _H9:
                _src.append('no return within 90d')
            elif pd.notna(_mid) and mids[i] == _mid:
                _src.append('same reporter')
            elif pd.notna(_ty) and tys[i] == _ty:
                _src.append('same reporter type, other reporter')
            else:
                _src.append('other reporter type')
        _cf['next_src'] = _src
        print(f'\nWhere the counted return (>{MIN_GAP_DAYS}d, within {_H9}d) comes from, by close year (% of cases):')
        print((pd.crosstab(_cf['_yr'], _cf['next_src'], normalize='index') * 100).round(1).to_string())

        # 6. data checks
        print('\nData checks:')
        _ac9 = eff['n_after_close'].fillna(0) > 0
        print(f"  cases with contact records dated after closure: {int(_ac9.sum()):,} of {len(eff):,} "
              f"({_ac9.mean() * 100:.1f}%) - those records are not used for reach, help or final result")
        if _ac9.any():
            print((_ac9.groupby(eff['close_date'].dt.year).mean() * 100).round(1).rename('% of cases').to_string())
        _gc = eff['arm_allrec'] != eff['arm']
        print(f"  cases whose group would differ if records outside the window were used: {int(_gc.sum()):,} "
              f"({_gc.mean() * 100:.1f}%; by close year below)")
        if _gc.any():
            print((_gc.groupby(eff['close_date'].dt.year).mean() * 100).round(1).rename('% of cases').to_string())
        print(f"  cases linked to more than one household (left out): {len(AMBIGUOUS_CASES):,}")
        _last_day = signalen['Datum melding'] == _LAST_SIGNAL
        print(f"  signals dated on the latest signal date ({_LAST_SIGNAL.date()}): {int(_last_day.sum()):,} "
              f"from {signalen.loc[_last_day, 'Hash'].nunique():,} household(s) - their current episode is still "
              "running at the cutoff (a large number suggests a batch-delivery day)")
    _section_done('9d. Calendar Comparability Check')
except Exception as _err:
    _section_failed('9d. Calendar Comparability Check', _err)


## 10. Experiments A3 + A4 + A5 — Method, Forwarding, Resolution

Three questions about what happens *after* the funnel:

- **A3:** return (secondary outcome) by the first successful contact — an outbound channel, or
  "callback after …" when the client made contact (the label states what prompted it; callbacks
  are responses, not outreach methods). Methods are chosen per case, so differences are
  associations, not a ranking of effectiveness.
- **A4:** do forwarded signals (`Doorgezet = Ja`) lead to fewer new signals than non-forwarded ones?
  Signals that arrive while the household already has another open case are often not forwarded
  and are followed by more signals anyway, so A4 is also shown for signals *without* another open case.
- **A5:** which final case results (paid, payment arrangement, no contact…) predict *not* returning?


In [ ]:
_report_section('10. Reach vs Effectiveness & Resolution Quality')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run

    # ── A4 - forwarding -> next signal (signal-level; non-forwarded signals never become cases) ──
    print('A4 - Forwarding (Doorgezet) -> next signal  [signal-level Cox HR, ref = not forwarded]:')
    hr = cox_hr(sigsurv, 'duration ~ doorgezet_bool + recidive_hist', ['doorgezet_bool', 'recidive_hist'])
    if hr is not None:
        display(hr)

    # Signals arriving while the household already has ANOTHER open case are often not forwarded
    # (the municipality is already involved) and are followed by more signals anyway, which can
    # make 'not forwarded' look worse. Repeat A4 among signals without another open case.
    _cs4 = (signalen.dropna(subset=['Meldingnummer', 'Hash', 'Datum melding'])
            .groupby('Meldingnummer').agg(Hash=('Hash', 'first'), c_start=('Datum melding', 'min'))
            .reset_index().rename(columns={'Meldingnummer': 'c_mnr'})
            .merge(meldingen[['Meldingnummer', 'Afgerond d.d.']].rename(columns={'Meldingnummer': 'c_mnr'}),
                   on='c_mnr', how='left'))
    _cs4['c_end'] = _cs4['Afgerond d.d.'].fillna(EXPORT_DATE)      # still open -> open until cutoff
    _pr4 = (sigsurv[['Hash', 'signal_date', 'Meldingnummer']].assign(_row=np.arange(len(sigsurv)))
            .merge(_cs4[['Hash', 'c_mnr', 'c_start', 'c_end']], on='Hash'))
    _open4 = _pr4.loc[(_pr4['c_mnr'] != _pr4['Meldingnummer']) & (_pr4['c_start'] < _pr4['signal_date'])
                      & (_pr4['signal_date'] <= _pr4['c_end']), '_row']
    sigsurv['open_case'] = np.isin(np.arange(len(sigsurv)), _open4.unique())
    _fw4 = (sigsurv['doorgezet_bool'] == True).groupby(sigsurv['open_case']).mean() * 100
    print(f"\n  Signals arriving while another case of the household is open: "
          f"{sigsurv['open_case'].mean()*100:.1f}%")
    print(f"  Forwarding rate: {_fw4.get(True, np.nan):.1f}% with another open case vs "
          f"{_fw4.get(False, np.nan):.1f}% without")
    print('A4 among signals WITHOUT another open case (cleaner comparison):')
    hr = cox_hr(sigsurv[~sigsurv['open_case']], 'duration ~ doorgezet_bool + recidive_hist',
                ['doorgezet_bool', 'recidive_hist'])
    if hr is not None:
        display(hr)

    # ── A3 - does the best-reaching method also best prevent return? ──────────────
    print('\nA3 - Return by the first successful contact (outbound channel, or callback with its stated prompt):')
    if 'reach_method' in eff.columns:
        _m = eff[eff['reached'] == True].dropna(subset=['reach_method'])
        _method_groups = list(_m.groupby('reach_method'))
        _rows = []
        for meth, g in _method_groups:
            _rows.append((str(meth)[:45], len(g), cuminc_at(g['duration'], g['event'], 90) * 100,
                          cuminc_at(g['duration'], g['event'], 365) * 100))
        if _rows:
            _mt = pd.DataFrame(_rows, columns=['method', 'n', 'return_90d_%', 'return_365d_%']).sort_values('return_90d_%')
            display(_mt.round(1))

            # Same survival-derived cumulative-return graph as §9a, now stratified by reach method.
            fig, ax = plt.subplots(figsize=(11, 6))
            for _i, (meth, g) in enumerate(_method_groups):
                sf = SurvfuncRight(g['duration'].values, g['event'].values)
                ax.step(sf.surv_times, (1 - sf.surv_prob) * 100, where='post',
                        linewidth=2, color=COLORS[_i % len(COLORS)],
                        label=f'{str(meth)[:45]} (n={len(g)})')
            ax.set_xlim(0, 730)
            ax.set_xlabel('Days since case close'); ax.set_ylabel('Cumulative return %')
            ax.set_title('Return after case close, by method that achieved contact')
            ax.legend(fontsize=9)
            plt.tight_layout(); plt.show()

            fig, ax = plt.subplots(figsize=(11, 5))
            ax.barh(_mt['method'], _mt['return_90d_%'], color=COLORS[0])
            ax.set_xlabel('90-day return %  (lower = fewer later signals - an association, not effectiveness)')
            ax.set_title('A3 - Return by method that achieved contact')
            plt.tight_layout(); plt.show()
            print('Methods are chosen per case (difficulty, availability), so this ordering is an association.')
            print("'callback after ...' rows are client responses (the label states what prompted them),")
            print('not outreach methods.')
            if _m['close_date'].dt.year.nunique() > 1:
                _yr3 = []
                for (_mm, _yy), _gg in _m.groupby(['reach_method', _m['close_date'].dt.year]):
                    _v = cuminc_at(_gg['duration'], _gg['event'], 90)
                    _yr3.append((str(_mm)[:34], int(_yy), len(_gg),
                                 round(_v * 100, 1) if pd.notna(_v) else np.nan))
                if _yr3:
                    _d3 = pd.DataFrame(_yr3, columns=['method', 'year', 'n', 'ret90'])
                    _p3 = _d3.pivot(index='method', columns='year', values='ret90')
                    _n3 = _d3.pivot(index='method', columns='year', values='n')
                    print('\nA3 per year - 90d return by reaching method (all observed groups):')
                    print(_p3.to_string())
                    print('\nCase counts behind the method-by-year estimates:')
                    print(_n3.to_string())
        else:
            print('  No reached case has a recorded successful contact method.')
    else:
        print('  primary_method unavailable.')

    # ── A5 - does a good resolution predict not returning? ────────────────────────
    print('\nA5 - 90-day return by final result (Eindresultaat, n>=30):')
    if 'eindresultaat' in eff.columns:
        _rows = []
        for res, g in eff.dropna(subset=['eindresultaat']).groupby('eindresultaat'):
            if len(g) >= 30:
                _rows.append((str(res)[:55], len(g), cuminc_at(g['duration'], g['event'], 90) * 100,
                              cuminc_at(g['duration'], g['event'], 365) * 100))
        if _rows:
            display(pd.DataFrame(_rows, columns=['eindresultaat', 'n', 'return_90d_%', 'return_365d_%'])
                    .sort_values('return_90d_%').round(1))
        else:
            print('  No result category with >=30 cases.')
    else:
        print('  eindresultaat unavailable.')
    _section_done('10. Reach vs Effectiveness & Resolution Quality')
except Exception as _err:
    _section_failed('10. Reach vs Effectiveness & Resolution Quality', _err)


## 11. Experiment B — Who Returns? (Targeting)

Profiles the repeat-signal rate by reporter type, debt band, age band and minor children,
to answer *who should be prioritised*. Uses the source system's own `Recidive` flag as ground
truth; the full 2021–2026 history in the export (§13 episodes) provides a consistent
cross-check. Groups with fewer than 30 signals are hidden. Note: the flag looks *backward* (was
the household reported in the previous ~6 months?) and is counted per **signal**, so households
with many signals weigh more — it describes repeat reporting, not a household-level prediction
of future return.


In [ ]:
_report_section('11. Targeting - Who Returns? (built-in Recidive flag)')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    MIN_CELL = 30
    obsB = signalen_x.copy()
    obsB['recidive_hist'] = ja_nee(obsB['Recidive'])
    _ov = obsB['recidive_hist'].mean() * 100
    print(f"Overall built-in recidive rate: {_ov:.1f}%  (n={int(obsB['recidive_hist'].notna().sum()):,})")

    _dims = [('Type melder', 'Reporter type'), ('debt_band', 'Debt band'),
             ('age_band', 'Age band'), ('Minderjarige kinderen', 'Minor children')]

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    for ax, (col, label) in zip(axes.flatten(), _dims):
        if col not in obsB.columns:
            ax.text(0.5, 0.5, f'{label}\nnot available', ha='center', va='center', transform=ax.transAxes); continue
        g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
        g = g[g['n'] >= MIN_CELL].sort_values('rate')
        if len(g) == 0:
            ax.text(0.5, 0.5, f'{label}\ntoo few', ha='center', va='center', transform=ax.transAxes); continue
        ax.barh(g.index.astype(str), g['rate'] * 100, color=COLORS[1])
        ax.axvline(_ov, color='red', linestyle='--', linewidth=1.5, label=f'overall {_ov:.1f}%')
        ax.xaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_title(f'Recidive by {label}'); ax.set_xlabel('Repeat-signal %'); ax.legend(fontsize=8)
    plt.suptitle('Targeting - repeat-signal rate by household profile', y=1.01)
    plt.tight_layout(); plt.show()

    print('\nHighest repeat-signal rate per dimension (backward-looking flag, counted per signal -')
    print('households with many signals weigh more; not a prediction of future return):')
    for col, label in _dims:
        if col not in obsB.columns:
            continue
        g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
        g = g[g['n'] >= MIN_CELL].sort_values('rate', ascending=False)
        if len(g):
            print(f'  {label:<16}: {g.index[0]}  ->  {g.iloc[0]["rate"]*100:.1f}%  (n={int(g.iloc[0]["n"]):,})')

    print('\nFull recidive breakdown by dimension (values behind the bars, n>=30):')
    for col, label in _dims:
        if col not in obsB.columns:
            continue
        g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
        g = g[g['n'] >= MIN_CELL].sort_values('rate', ascending=False)
        if len(g):
            g['recidive_%'] = (g['rate'] * 100).round(1)
            print(f'\n- {label}:')
            print(g.drop(columns='rate').to_string())

    _yc = obsB.dropna(subset=['Datum melding']).copy()
    _yc['_yr'] = _yc['Datum melding'].dt.year
    if _yc['_yr'].nunique() > 1:
        _yc['_rec'] = ja_nee(_yc['Recidive']) == True
        for _col, _label in _dims:
            if _col not in _yc.columns:
                continue
            _pv11 = _yc.pivot_table(index=_col, columns='_yr', values='_rec',
                                    aggfunc='mean', observed=True) * 100
            _nv11 = _yc.pivot_table(index=_col, columns='_yr', values='_rec',
                                    aggfunc='count', observed=True)
            _pv11 = _pv11.where(_nv11 >= 30).dropna(how='all')
            if len(_pv11):
                print(f'\nRecidive % by {_label} per year (cells n>=30):')
                print(_pv11.round(1).to_string())
    _section_done('11. Targeting - Who Returns? (built-in Recidive flag)')
except Exception as _err:
    _section_failed('11. Targeting - Who Returns? (built-in Recidive flag)', _err)


## 12. Experiments C1–C4 — Return Timing, Contact Effort & Recurrence Patterns

Is recorded contact effort associated with subsequent return?
Four questions about *when* and *how much*:

- **C3:** when do households return after closure — first weeks or a long tail?
- **C1:** is *faster* contact associated with a lower chance of returning?
- **C2:** is the *number* of outbound contact attempts (dated, signal to closure) associated with
  returning?
- **C4:** how many signals does a household send — one-off or chronic returners?

For C1, the descriptive table defines **Fast** as contact at or below the observed median number
of days to reach and **Slow** as contact above that median. These are text labels, not 0/1
codes. The adjusted Cox model uses the actual number of days to reach as a continuous variable.


In [ ]:
_report_section('12. Return Timing, Contact Effort & Recurrence Patterns')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run

    # ── C3 - overall recurrence timing ────────────────────────────────────────────
    if len(eff) and int(eff['event'].sum()) >= 10:
        sf = SurvfuncRight(eff['duration'].values, eff['event'].values)
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.step(sf.surv_times, (1 - sf.surv_prob) * 100, where='post', color=COLORS[2], linewidth=2)
        ax.set_xlim(0, 730); ax.set_xlabel('Days since case close'); ax.set_ylabel('Cumulative return %')
        ax.set_title('C3 - When do households return? (overall)')
        plt.tight_layout(); plt.show()
        _HZ = [30, 90, 180, 270, 365, 540, 730]
        print("Overall cumulative return % (values behind the curve):")
        print('  ' + ''.join(f'{h}d'.rjust(8) for h in _HZ))
        print('  ' + ''.join(cuminc_str(eff['duration'], eff['event'], h) for h in _HZ))
        print('  (n/a = fewer than 10 cases with a known status at that horizon, or not estimable;')
        print('   percentages are Kaplan-Meier estimates)')
    else:
        print('C3 - Not enough events for a recurrence curve.')

    # ── C1 - speed to contact ─────────────────────────────────────────────────────
    print('\nC1 - Speed-to-contact -> return:')
    if 'days_to_reach' in eff.columns and eff['days_to_reach'].notna().any():
        _r = eff[(eff['reached'] == True) & eff['days_to_reach'].notna()].copy()
        if len(_r) >= 40:
            _med = _r['days_to_reach'].median()
            _r['speed_grp'] = np.where(_r['days_to_reach'] <= _med, 'Fast', 'Slow')
            print(f'  Descriptive split at the observed median: {_med:g} days')
            print(f'    Fast = days_to_reach <= {_med:g}; Slow = days_to_reach > {_med:g}')
            print('    Fast/Slow are text labels, not binary 0/1 values.')
            _speed_rows = []
            for grp in ['Fast', 'Slow']:
                g = _r[_r['speed_grp'] == grp]
                _criterion = (f'<= {_med:g} days' if grp == 'Fast' else f'> {_med:g} days')
                _speed_rows.append((grp, _criterion, len(g), g['days_to_reach'].median(),
                                    cuminc_at(g['duration'], g['event'], 90) * 100,
                                    cuminc_at(g['duration'], g['event'], 365) * 100))
            display(pd.DataFrame(_speed_rows, columns=['group', 'definition', 'n',
                                                       'median_days_to_reach',
                                                       'return_90d_%', 'return_365d_%']).round(1))
            print('\n  Adjusted Cox model: days_to_reach enters as a continuous variable, not as Fast=0/1.')
            print('  Its hazard ratio is interpreted per one additional day before successful contact.')
            hr = cox_hr(_r, 'duration ~ days_to_reach + recidive_hist', ['days_to_reach', 'recidive_hist'])
            if hr is not None:
                display(hr)
        else:
            print('  Too few reached cases with contact timing.')
    else:
        print('  days_to_reach unavailable.')

    # ── C2 - contact-attempt intensity (descriptive association) ──────────────────
    print('\nC2 - Number of OUTBOUND contact attempts (dated, signal to closure) -> return:')
    if 'n_outbound' in eff.columns and eff['n_outbound'].notna().any():
        _a = eff.dropna(subset=['n_outbound']).copy()
        print(f"  (cases with no outbound attempt in the window: {int((_a['n_outbound'] == 0).sum()):,} - not in a band)")
        _a['att_band'] = pd.cut(_a['n_outbound'], bins=[0, 1, 3, 6, 1e9], labels=['1', '2-3', '4-6', '7+'])
        _rows = []
        for b, g in _a.groupby('att_band'):
            if len(g) >= 30:
                _rows.append((str(b), len(g), cuminc_at(g['duration'], g['event'], 90) * 100,
                              cuminc_at(g['duration'], g['event'], 365) * 100))
        if _rows:
            display(pd.DataFrame(_rows, columns=['outbound attempts', 'n', 'return_90d_%', 'return_365d_%']).round(1))
        else:
            print('  Too few cases per attempt band.')
    else:
        print('  n_outbound unavailable.')

    # ── C4 - recurrent signals per household ──────────────────────────────────────
    print('\nC4 - Repeated signals per household:')
    _sph = signalen.dropna(subset=['Hash']).groupby('Hash').size()
    if len(_sph):
        print(f'  households={len(_sph):,}  mean={_sph.mean():.2f}  median={_sph.median():.0f}  '
              f'p99={_sph.quantile(.99):.0f}  with >1 signal: {(_sph > 1).mean()*100:.1f}%')
    _section_done('12. Return Timing, Contact Effort & Recurrence Patterns')
except Exception as _err:
    _section_failed('12. Return Timing, Contact Effort & Recurrence Patterns', _err)


## 13. Module 1 — Within-Household Comparisons (E1, E2, E3, G4)

Round-1 results showed selection bias: worse-off households are reached and accept help more
often, so between-household comparisons are misleading. This module compares households
**with themselves** across their own episodes, and stress-tests the outcome definition.
The prep cell below collapses each household's signals into **episodes** (signals within
`EPISODE_GAP_DAYS` = 45 days of each other are the same episode) — the shared unit for §13–§15.
Creditors re-send the same unpaid debt about every month (median 31–33 days, §15a), so a 14-day
rule would turn one unresolved debt into a new "episode" every month. The case-return outcome
in §9–§12 keeps `MIN_GAP_DAYS` = 14; §15a shows how sensitive that outcome is to the gap.


In [ ]:
_report_section('13. Module 1 - Within-Household Comparisons: preparation')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    # ── Episode table: collapse each household's signals into episodes ────────────
    _t = signalen_x.dropna(subset=['Hash', 'Datum melding']).sort_values(['Hash', 'Datum melding']).copy()
    _t['_prev'] = _t.groupby('Hash')['Datum melding'].shift()
    # Creditors re-send the SAME unpaid debt about monthly (median 31-33d, see 15a). With a 14-day
    # rule each monthly re-report became a new 'episode'; signals up to EPISODE_GAP_DAYS apart are
    # now one episode. (MIN_GAP_DAYS = 14 still defines a case 'return' in sections 9-12.)
    # (EPISODE_GAP_DAYS = 45 is set in section 9, so 9c and sections 13-15 share one rule.)
    _dgap = (_t['Datum melding'] - _t['_prev']).dt.days
    _t['_newep'] = (_t['_prev'].isna()) | (_dgap > EPISODE_GAP_DAYS)
    _t['epnum'] = _t.groupby('Hash')['_newep'].cumsum()

    epi = (_t.groupby(['Hash', 'epnum'])
           .agg(start=('Datum melding', 'min'), end=('Datum melding', 'max'),
                n_signals=('SignaalID', 'count'))
           .reset_index())
    _types = (_t.dropna(subset=['Type melder']).sort_values('Datum melding')
              .groupby(['Hash', 'epnum'])['Type melder'].first().rename('ep_type'))
    epi = epi.merge(_types, on=['Hash', 'epnum'], how='left').sort_values(['Hash', 'epnum'])
    epi['next_start'] = epi.groupby('Hash')['start'].shift(-1)
    epi['gap_days']  = (epi['next_start'] - epi['end']).dt.days
    epi['gap_event'] = epi['next_start'].notna().astype(int)
    epi['gap_dur']   = np.where(epi['gap_event'] == 1, epi['gap_days'],
                                (EXPORT_DATE - epi['end']).dt.days)
    # Full episode inventory: an episode whose last signal is on the export date has no follow-up
    # yet (gap_dur = 0) but is KEPT - counts, classes, transitions and H1 targets use every episode.
    # Survival analyses use epi_surv (episodes with follow-up > 0 days).
    epi['ongoing_at_cutoff'] = (epi['gap_event'] == 0) & (epi['gap_dur'] <= 0)

    # Creditor types per episode and 'new type' transitions (G1b, G3). For every episode after the
    # first: NEW type = it contains a reporter type never seen in any EARLIER signal of the household
    # (types first seen within the same episode count for that episode); same type again = all its
    # types were also in the previous episode; otherwise known other type. Uses ALL types per episode.
    _tt = (_t.dropna(subset=['Type melder']).groupby(['Hash', 'epnum'])['Type melder']
           .agg(lambda s: frozenset(s)).rename('ep_types'))
    epi = epi.merge(_tt, on=['Hash', 'epnum'], how='left')
    _kinds = {}
    for _h, g in epi.sort_values(['Hash', 'epnum']).groupby('Hash'):
        _seen, _prev = set(), None
        for _en, _ty in zip(g['epnum'], g['ep_types']):
            _E = _ty if isinstance(_ty, frozenset) else frozenset()
            if _prev is not None and _E:
                _kinds[(_h, _en)] = ('NEW type (cascade)' if _E - _seen
                                     else 'same type again' if _E <= _prev else 'known other type')
            _seen |= _E
            _prev = _E if _E else _prev
    epi['transition_kind'] = [_kinds.get((h, e)) for h, e in zip(epi['Hash'], epi['epnum'])]
    epi_surv = epi[epi['gap_dur'] > 0].copy()

    _eph = epi.groupby('Hash').size()
    print(f"Episodes: {len(epi):,} across {epi['Hash'].nunique():,} households "
          f"(signals <= {EPISODE_GAP_DAYS}d apart merged into one episode)")
    print(f"Episodes whose last signal is ON the export date (kept; no follow-up yet): {int(epi['ongoing_at_cutoff'].sum()):,}")
    print(f"Episodes whose end is not yet confirmed (last signal within {EPISODE_GAP_DAYS} days of the export "
          f"date; kept, censored in survival analyses): "
          f"{int(((epi['gap_event'] == 0) & (epi['gap_dur'] <= EPISODE_GAP_DAYS)).sum()):,}")
    print('Episode-gap sensitivity (how many episodes each merge rule produces):')
    for _gp in sorted({MIN_GAP_DAYS, 30, EPISODE_GAP_DAYS, 60}):
        _ne = int((_t['_prev'].isna() | (_dgap > _gp)).sum())
        print(f"  gap {_gp:>2}d: {_ne:,} episodes ({_ne / max(_t['Hash'].nunique(), 1):.2f} per household)"
              + ('   <- used' if _gp == EPISODE_GAP_DAYS else ''))
    print(f"Episodes per household: mean={_eph.mean():.2f}  median={_eph.median():.0f}  p99={_eph.quantile(.99):.0f}")
    _eyr = epi['start'].dt.year.value_counts().sort_index()
    if len(_eyr) > 1:
        print("Episodes started per year:")
        print(_eyr.rename('episodes').to_string())

    # ── Creditor breadth at case start (shared by E1b and G2) ─────────────────────
    _ts2 = signalen_x.dropna(subset=['Hash', 'Datum melding', 'Type melder']).sort_values('Datum melding')
    _by_h = {h: list(zip(g['Datum melding'].tolist(), g['Type melder'].tolist()))
             for h, g in _ts2.groupby('Hash')}
    def _breadth(h, t):
        lst = _by_h.get(h)
        if not lst or pd.isna(t):
            return np.nan
        return len({ty for d0, ty in lst if d0 <= t})
    eff['breadth'] = [_breadth(h, t) for h, t in zip(eff['Hash'], eff['signal_date'])]
    eff['breadth_band'] = pd.cut(eff['breadth'], bins=[0, 1, 2, 99], labels=['1', '2', '3+'])

    # ── Case-level episode numbering for within-household models ──────────────────
    eff_sorted = eff.sort_values(['Hash', 'close_date']).copy()
    eff_sorted['case_epnum'] = eff_sorted.groupby('Hash').cumcount() + 1
    eff_multi = eff_sorted[eff_sorted.groupby('Hash')['Hash'].transform('size') >= 2].copy()
    print(f"Households with >=2 closed cases (usable for within-household designs): "
          f"{eff_multi['Hash'].nunique():,} ({len(eff_multi):,} cases)")
    _section_done('13. Module 1 - Within-Household Comparisons: preparation')
except Exception as _err:
    _section_failed('13. Module 1 - Within-Household Comparisons: preparation', _err)


### 13a. E1 — Within-Household Comparison

Compares the **same household** across its own episodes: does the episode where it was
reached (or accepted help) show a lower risk of a new signal than its other episodes?
Stratified Cox with household as stratum removes stable differences between households — not
changes over time in need, calendar period, earlier help, or selection into later cases. Only
"discordant" households (exposure varies across their episodes) contribute; cases with unknown
reach or help status are left out, and CIs are clustered by household.

**Check (last case dropped):** a household only gets a later case if it came back, so its
earlier cases almost always end in a return and only its *last* case can stay signal-free.
If contact practice changed over the years, that alone can make "reached" look protective.
E1 is therefore re-run without each household's last case. This removes that pattern, but the
check still uses only households that came back, so it is a sensitivity check, not a fix.

**E1 on episodes:** almost every signal is its own case, so consecutive "cases" are mostly
consecutive months of the *same* debt. E1 is therefore also run with the §13 episodes (problem
periods, 45-day rule) as the unit: exposure = reached / accepted help in the episode's *first*
case — the actual first case, open or closed; if that case is not in the analysis frame, the
episode's exposure is unknown and it is left out (long-running problems get no extra chances);
outcome = time from the episode's last signal to the next episode — a clock that does not depend
on when a case was closed. It is shown with and without each household's last episode.

**Read the episode-level results first.** The case-level models are affected by closing-time
timing (§9c): in tests on simulated data where contact had no effect, they still showed a clear
"benefit" of being reached (HR about 0.75), while the episode-level model did not.

**E1b** repeats the model adjusted for episode-level severity (episode number, prior
signals, creditor breadth, debt band, year); stable household traits are already absorbed
by the stratum. If the effect moves toward 1, the unadjusted result was partly severity.


In [ ]:
_report_section('13a. E1 - Within-Household Comparison')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run

    def _within_cox(d, expo, label):
        d = d.dropna(subset=[expo])           # unknown exposure (reach / help unknown) is left out
        _disc = d.groupby('Hash')[expo].nunique()
        dd = d[d['Hash'].isin(_disc[_disc > 1].index)].copy()
        n_hh = dd['Hash'].nunique()
        print(f"{label}: discordant households = {n_hh:,}  "
              f"(episodes {len(dd):,}, events {int(dd['event'].sum()):,})")
        if len(dd) < 100 or dd['event'].sum() < 30:
            print('  Too few discordant episodes/events - model skipped.')
            return
        try:
            # household strata + household-clustered CIs (one household's episodes are not independent)
            display(_cox_fit(f'duration ~ {expo}', dd, dd['event'].values,
                             groups=dd['Hash'].values, strata=dd['Hash'].values).round(3))
        except Exception as _e:
            print(f'  stratified Cox not estimable: {_e}')

    if len(eff_multi) == 0:
        print('No households with multiple closed cases - E1 skipped.')
    else:
        print('Inclusion flow (who this analysis describes):')
        print(f'  households with >=1 closed, linked case: {eff["Hash"].nunique():,}')
        print(f'  with >=2 closed cases:                   {eff_multi["Hash"].nunique():,}  '
              f'({len(eff_multi):,} cases)')
        print('  (only households whose exposure VARIES across episodes identify the effect;')
        print('   their counts are printed per exposure below)\n')
        eff_multi['reached_f'] = eff_multi['reach_status'].map({'reached': 1.0, 'not reached': 0.0})
        eff_multi['accepted']  = np.where(eff_multi['arm'].eq('C_accepted'), 1.0,
                                          np.where(eff_multi['arm'].isin(['A_not_reached', 'B_refused']), 0.0, np.nan))
        _within_cox(eff_multi, 'reached_f', 'Exposure = reached in that episode')
        print()
        _within_cox(eff_multi, 'accepted', 'Exposure = help accepted in that episode')
        print('\nHR < 1 = the same household has fewer / later new signals after exposed episodes (association).')
        print('NOTE: these case-level models are affected by closing-time timing (see 9c) - in tests on')
        print('simulated data where contact had no effect they still showed a clear "benefit" (HR ~0.75).')
        print('Read the episode-level results further below first.')

        # ── Check: drop each household's LAST closed case ─────────────────────────
        # A later case only exists if the household came back, so earlier cases nearly always end
        # in a return and only the last case can stay signal-free (censored). Dropping the last
        # case removes that artefact; a similar HR here supports the result above.
        _lastn = eff_multi.groupby('Hash')['case_epnum'].transform('max')
        eff_nolast = eff_multi[eff_multi['case_epnum'] < _lastn].copy()
        eff_nolast = eff_nolast[eff_nolast.groupby('Hash')['Hash'].transform('size') >= 2]
        print("\nCheck - same models WITHOUT each household's last case (removes the 'only the last case")
        print("can stay signal-free' pattern, but still uses only households that came back):")
        _within_cox(eff_nolast, 'reached_f', 'Exposure = reached (last case dropped)')
        _within_cox(eff_nolast, 'accepted', 'Exposure = help accepted (last case dropped)')
        print('A similar HR means this one pattern does not explain the result; it cannot rule out other')
        print('selection (only households that came back are used).')

        # ── E1b: adjusted for episode-level (time-varying) severity ───────────────
        print('\nE1b - same model, adjusted for how severe each episode was (episode number,')
        print('prior signals, creditor breadth, debt band, year). Stable household traits are')
        print('already absorbed by the household stratum, so only episode-varying ones enter.')
        _dsc = eff_multi.dropna(subset=['reached_f']).groupby('Hash')['reached_f'].nunique()
        db = eff_multi[eff_multi['Hash'].isin(_dsc[_dsc > 1].index)]
        db = db.dropna(subset=['reached_f', 'duration', 'event', 'debt_band']).copy()
        db['yr'] = (db['close_date'].dt.year - int(db['close_date'].dt.year.min())) if len(db) else 0
        if hasattr(db['debt_band'], 'cat'):
            db['debt_band'] = db['debt_band'].cat.remove_unused_categories()
        if len(db) >= 120 and db['event'].sum() >= 40:
            _terms = ['reached_f']
            for _tv in ['case_epnum', 'prior_sigs', 'breadth', 'yr']:
                if _tv in db.columns and db[_tv].nunique() > 1:
                    _terms.append(_tv)
            if db['debt_band'].nunique() > 1:
                _terms.append('C(debt_band)')
            try:
                display(_cox_fit('duration ~ ' + ' + '.join(_terms), db, db['event'].values,
                                 groups=db['Hash'].values, strata=db['Hash'].values).round(3))
                print('Compare reached_f with the unadjusted model above: if it moves toward 1,')
                print('part of the unadjusted association reflects episode severity.')
            except Exception as _e:
                print(f'  adjusted model not estimable: {_e}')
        else:
            print(f'  Too few complete discordant episodes for the adjusted model (n={len(db)}).')

        # ── E1 on EPISODES (problem periods) instead of monthly cases ─────────────
        # Almost every signal is its own case, so consecutive 'cases' are mostly consecutive months
        # of the same debt. Unit here = section-13 episode: exposure = reached (accepted) in the
        # episode's first case; outcome = time from the episode's last signal to the next episode,
        # a clock that does not depend on when a case was closed.
        # Map EVERY case (open or closed, in the frame or not) to its episode, so 'first case' really
        # is the episode's first case. Exposure is known only when that case is in the effectiveness
        # frame with a known status; otherwise the episode is left out (exposure unknown).
        # 'Reached in ANY case' would favour long episodes - more cases, more chances to be reached -
        # and leaned towards 'contact helps' in tests on simulated data where contact had no effect.
        _allc = (_first[['Meldingnummer', 'Hash', 'signal_date']].dropna(subset=['Hash', 'signal_date'])
                 .sort_values('signal_date'))
        _em1 = pd.merge_asof(_allc, epi[['Hash', 'epnum', 'start', 'end']].sort_values('start'),
                             left_on='signal_date', right_on='start', by='Hash', direction='backward')
        _em1 = _em1[_em1['epnum'].notna() & (_em1['signal_date'] <= _em1['end'])].copy()
        _em1['epnum'] = _em1['epnum'].astype(int)
        _ex1 = (_em1.drop_duplicates(['Hash', 'epnum'], keep='first')
                .merge(eff_sorted[['Meldingnummer', 'reach_status', 'arm']], on='Meldingnummer', how='left'))
        _ex1['reached_f'] = _ex1['reach_status'].map({'reached': 1.0, 'not reached': 0.0})
        _ex1['accepted'] = np.where(_ex1['arm'].eq('C_accepted'), 1.0,
                                    np.where(_ex1['arm'].isin(['A_not_reached', 'B_refused']), 0.0, np.nan))
        ep_e1 = (epi_surv.merge(_ex1[['Hash', 'epnum', 'reached_f', 'accepted']], on=['Hash', 'epnum'], how='inner')
                   .rename(columns={'gap_dur': 'duration', 'gap_event': 'event'}))
        ep_multi = ep_e1[ep_e1.groupby('Hash')['Hash'].transform('size') >= 2].copy()
        print(f'\nE1 on EPISODES (problem periods, {EPISODE_GAP_DAYS}-day rule) instead of monthly cases:')
        print(f'  households with >=2 episodes: {ep_multi["Hash"].nunique():,} ({len(ep_multi):,} episodes;'
              ' episodes whose first case has an unknown status are left out per exposure)')
        _within_cox(ep_multi, 'reached_f', 'Exposure = reached in the first case of that episode')
        _within_cox(ep_multi, 'accepted', 'Exposure = help accepted in the first case of that episode')
        _lastep = ep_multi.groupby('Hash')['epnum'].transform('max')
        ep_nolast = ep_multi[ep_multi['epnum'] < _lastep]
        ep_nolast = ep_nolast[ep_nolast.groupby('Hash')['Hash'].transform('size') >= 2]
        print("Check without each household's last episode:")
        _within_cox(ep_nolast, 'reached_f', 'Exposure = reached (last episode dropped)')
        _within_cox(ep_nolast, 'accepted', 'Exposure = help accepted (last episode dropped)')
        print('Outcome = time from the end of one problem period to the next; HR < 1 = the same')
        print('household stays away longer after periods in which it was reached / accepted help (association).')
    _section_done('13a. E1 - Within-Household Comparison')
except Exception as _err:
    _section_failed('13a. E1 - Within-Household Comparison', _err)


### 13b. E2 — State Dependence vs Chronic Vulnerability

Does each episode raise the risk of the next (escalation — early intervention can break a
spiral), or do a fixed group of chronic households generate most signals (needs a different
service model)? Looked at two ways — hazard of the next episode by episode number, and
within-household inter-episode gap trends — neither of which separates the two explanations
cleanly.

In the Cox table, episode **1** is the reference category. The formula term
`Treatment(reference="1")` is statistical reference-category coding only; it does not mean
that episode 1 or any household received a treatment. Each later row compares the rate of a
next episode after that episode number with the rate after the first observed episode.


In [ ]:
_report_section('13b. E2 - State Dependence vs Chronic Vulnerability')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _e = epi_surv.copy()
    _e['ep_band'] = pd.cut(_e['epnum'], bins=[0, 1, 2, 3, 1e9], labels=['1', '2', '3', '4+'])

    print('Cumulative probability of a NEXT episode within 180d, by episode number:')
    for b, g in _e.groupby('ep_band'):
        if len(g) >= 30:
            print(f'  after episode {b:>2}: 180d={cuminc_str(g["gap_dur"], g["gap_event"], 180, 5)}  '
                  f'365d={cuminc_str(g["gap_dur"], g["gap_event"], 365, 5)}  (n={len(g):,})')

    d = _e.dropna(subset=['ep_band']).copy()
    _bandok = [b for b, g in d.groupby('ep_band') if len(g) >= 30 and g['gap_event'].sum() >= 5]
    d = d[d['ep_band'].isin(_bandok)]
    d['ep_band'] = d['ep_band'].cat.remove_unused_categories()
    if len(d) >= 200 and d['gap_event'].sum() >= 50 and d['ep_band'].nunique() >= 2:
        try:
            # clustered by household: one household's episodes are not independent evidence
            _fit = _cox_fit('gap_dur ~ C(ep_band, Treatment(reference="1"))', d,
                            d['gap_event'].values, groups=d['Hash'].values)
            _hr = _fit['HR'].values; _pv = _fit['p'].values
            _ci = np.log(_fit[['CI2.5%', 'CI97.5%']].values)
            _cats = [str(n).split('[T.')[-1].rstrip(']') for n in _fit.index]
            def _e2_hr_text(h):
                if np.isclose(h, 1):
                    return 'approximately the same hazard as after episode 1'
                direction = 'higher' if h > 1 else 'lower'
                return f'{abs(h - 1) * 100:.1f}% {direction} hazard of a next episode'
            _e2tab = pd.DataFrame({
                'comparison': ['vs after episode 1'] * len(_hr),
                'HR': _hr,
                'CI2.5%': np.exp(_ci[:, 0]),
                'CI97.5%': np.exp(_ci[:, 1]),
                'p': _pv,
                'plain-language estimate': [_e2_hr_text(h) for h in _hr],
            }, index=[f'After episode {c}' for c in _cats])
            _e2ref = pd.DataFrame({
                'comparison': ['reference category'], 'HR': [1.0],
                'CI2.5%': [np.nan], 'CI97.5%': [np.nan], 'p': [np.nan],
                'plain-language estimate': ['baseline for all comparisons'],
            }, index=['After episode 1 (reference)'])
            _e2tab = pd.concat([_e2ref, _e2tab])
            print('\nGap-time Cox - hazard of a next episode by episode number:')
            print('  Reference = after episode 1 (the first OBSERVED episode in these data).')
            print('  "Treatment(reference=1)" means reference-category coding, not an intervention.')
            print('  HR > 1 = next episode tends to occur sooner; HR < 1 = it tends to occur later.')
            print('  A 95% confidence interval containing 1 means no clear difference from episode 1.')
            print('  CIs and p-values are clustered by household (episodes of one household are not independent).')
            display(_e2tab.round(3))
            print('NOTE: rising HRs can reflect escalation OR chronic households accumulating episodes;')
            print('the within-household gap trend below gives a rough indication - it cannot separate them.')
        except Exception as _e2:
            print(f'Gap-time Cox failed: {_e2}')
    else:
        print('Too few episodes for the gap-time model.')

    # within-household gap trend (needs >=2 completed gaps = >=3 episodes)
    _g = epi.dropna(subset=['gap_days'])
    _cnt = _g.groupby('Hash').size()
    short = long_ = flat = 0
    for _h in _cnt[_cnt >= 2].index:
        gg = _g[_g['Hash'] == _h].sort_values('epnum')['gap_days'].values
        if gg[-1] < gg[0] * 0.75:   short += 1
        elif gg[-1] > gg[0] * 1.33: long_ += 1
        else:                       flat += 1
    tot = short + long_ + flat
    if tot >= 30:
        print(f"\nWithin-household gap trends ({tot:,} households with >=2 completed gaps):")
        print(f"  gaps shortening (escalating):  {short:,} ({short/tot*100:.1f}%)")
        print(f"  gaps lengthening (recovering): {long_:,} ({long_/tot*100:.1f}%)")
        print(f"  roughly stable:                {flat:,} ({flat/tot*100:.1f}%)")
        print('Mostly shortening gaps fit escalation; stable/mixed gaps fit chronic heterogeneity -')
        print('a rough indication only.')
    else:
        print('\nToo few households with >=3 episodes for the gap-trend test.')
    _section_done('13b. E2 - State Dependence vs Chronic Vulnerability')
except Exception as _err:
    _section_failed('13b. E2 - State Dependence vs Chronic Vulnerability', _err)


### 13c. E3 — Trajectory Around Help Acceptance (exploratory)

**Exploratory analysis — not causal proof.** Does a household's own signal rate bend after it
accepts help? Groups are fixed at each household's **first reached case with a known help
answer**: accepted there vs declined there — a household that declined first and accepted later
stays in the comparison group, so group membership never depends on the future. t0 is the
**first successful contact** of that case, a proxy for the decision moment (the printed median
gap to the recorded help answer shows how close the proxy is); it falls back to the close date
when no contact date is recorded. Windows without any household are shown as n/a, not 0. Because help is
accepted at the household's worst moment, a post-t0 decline is expected from regression to the
mean alone (the "Ashenfelter dip"); the defense is the **reached-but-refused comparison group
aligned at the same moment** — the comparison of interest is whether the acceptors' curve bends
*more* (consistent with, not proof of, an effect of help).

Each window above only uses households whose window fits inside the data, so early and late
windows contain different households. A **balanced check** repeats the comparison with only the
households observable over the whole ±12 months, so every window describes the same group.


In [ ]:
_report_section('13c. E3 - Trajectory Around Help Acceptance')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _BINS = [(-24, -12), (-12, -6), (-6, -3), (-3, 0), (0, 3), (3, 6), (6, 12), (12, 24)]
    _OBS_START = signalen['Datum melding'].min()

    def _relative_rates(t0map, label):
        rows = []
        for a, b in _BINS:
            n_hh = 0; n_sig = 0
            for _h, _t0 in t0map.items():
                w0 = _t0 + pd.Timedelta(days=int(a * 30.44))
                w1 = _t0 + pd.Timedelta(days=int(b * 30.44))
                if w1 > EXPORT_DATE or w0 < _OBS_START:
                    continue                      # only households whose window is fully observed
                arr = _sig_by_hash.get(_h, [])
                n_sig += bisect.bisect_left(arr, w1) - bisect.bisect_left(arr, w0)
                n_hh += 1
            rows.append((f'{a}..{b}m', n_hh, n_sig / n_hh / (b - a) if n_hh else np.nan))
        return pd.DataFrame(rows, columns=['window', 'n_households', f'{label} signals/hh/month'])

    # Groups fixed at each household's FIRST reached case with a known help answer (B or C), so
    # membership never depends on later behaviour. t0 = first successful contact of that case.
    _bc3 = eff_sorted[eff_sorted['arm'].isin(['B_refused', 'C_accepted'])].sort_values(['Hash', 'signal_date'])
    _fb3 = _bc3.drop_duplicates('Hash', keep='first').copy()
    _t0raw = pd.to_datetime(_fb3['first_reach_date'], errors='coerce')
    _n_fb = int(_t0raw.isna().sum())
    _fb3['t0'] = _t0raw.fillna(_fb3['close_date'])
    _acc = _fb3.loc[_fb3['arm'] == 'C_accepted'].set_index('Hash')['t0']
    _ref = _fb3.loc[_fb3['arm'] == 'B_refused'].set_index('Hash')['t0']
    print(f"Households by their first reached case: accepted {len(_acc):,} | declined (comparison) {len(_ref):,}")
    print(f"t0 anchor: first successful contact ({_n_fb} case(s) without one fell back to close date)")
    _lag3 = (pd.to_datetime(_fb3['help_date'], errors='coerce') - _t0raw).dt.days
    if _lag3.notna().any():
        print(f"Days from first contact to the recorded help answer: median {_lag3.median():.0f} "
              "(t0 is a proxy for the decision moment)")

    if len(_acc) < 20:
        print('Too few help-acceptor households - E3 skipped.')
    else:
        ta = _relative_rates(_acc.to_dict(), 'acceptors')
        tr = _relative_rates(_ref.to_dict(), 'refusers')
        tab = ta.merge(tr, on='window', suffixes=('', '_r'))
        print('\nSignal rate around the aligned moment (t0 = first successful contact of that case):')
        display(tab.round(3))
        fig, ax = plt.subplots(figsize=(10, 5))
        _x = range(len(_BINS))
        ax.plot(_x, ta.iloc[:, 2], marker='o', linewidth=2, color=COLORS[0], label=f'help accepted (n<= {len(_acc)})')
        ax.plot(_x, tr.iloc[:, 2], marker='s', linewidth=2, color=COLORS[1], label=f'declined at first reached case (n<= {len(_ref)})')
        ax.axvline(3.5, color='red', linestyle='--', alpha=0.6)
        ax.set_xticks(list(_x)); ax.set_xticklabels([w for w, _, _ in zip(ta['window'], _x, _x)])
        ax.set_xlabel('months relative to t0'); ax.set_ylabel('signals per household per month')
        ax.set_title('E3 - Own trajectory around help acceptance vs aligned refusers')
        ax.legend()
        plt.tight_layout(); plt.show()
        print('Both curves peaking before t0 is expected: households are contacted at their worst moment')
        print('(the "Ashenfelter dip"). If the acceptor curve falls more than the comparison curve after t0,')
        print('that is consistent with help mattering - not proof: households that accept may differ in')
        print('ways that are not measured.')

        # ── Balanced check: the SAME households in every window (+-12 months) ─────
        _BAL = [(-12, -6), (-6, -3), (-3, 0), (0, 3), (3, 6), (6, 12)]
        def _balanced(t0map):
            keep = {h: t for h, t in t0map.items()
                    if t - pd.Timedelta(days=int(12 * 30.44)) >= _OBS_START
                    and t + pd.Timedelta(days=int(12 * 30.44)) <= EXPORT_DATE}
            rates = []
            for a, b in _BAL:
                n_sig = 0
                for _h, _t0 in keep.items():
                    arr = _sig_by_hash.get(_h, [])
                    n_sig += (bisect.bisect_left(arr, _t0 + pd.Timedelta(days=int(b * 30.44)))
                              - bisect.bisect_left(arr, _t0 + pd.Timedelta(days=int(a * 30.44))))
                rates.append(round(n_sig / max(len(keep), 1) / (b - a), 3))
            return rates, len(keep)
        _ra, _nb_acc = _balanced(_acc.to_dict())
        _rr, _nb_ref = _balanced(_ref.to_dict())
        if _nb_acc >= 20 and _nb_ref >= 20:
            print(f'\nBalanced check - the same households in every window (+-12 months): '
                  f'{_nb_acc:,} acceptor and {_nb_ref:,} refuser households')
            display(pd.DataFrame({'window': [f'{a}..{b}m' for a, b in _BAL],
                                  'acceptors signals/hh/month': _ra,
                                  'refusers signals/hh/month': _rr}))
            print('Same shape as the table above = the pattern is not caused by which households are included.')
        else:
            print('\nBalanced check skipped - too few households observable over +-12 months.')
    _section_done('13c. E3 - Trajectory Around Help Acceptance')
except Exception as _err:
    _section_failed('13c. E3 - Trajectory Around Help Acceptance', _err)


### 13d. G4 — Silent Success vs Disappearance from Observation

"No new signal" is not proof of success — the household may simply have left the observation
system. This cell profiles non-returners and counts, per creditor type, households that
**stopped this type while still reported by others** (consistent with resolving that debt — not
proof), households that **stopped this type earlier but are now gone everywhere**, and
households that **stopped with all types at once** (resolved *or* left the reporting system —
ambiguous). Output: descriptive counts, not a success rate. The non-returner profile uses only cases with at least one
year of follow-up (a case closed last week has simply not had time to return).


In [ ]:
_report_section('13d. G4 - Silent Success vs Disappearance')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    # Only cases with at least one year of follow-up: a case closed last week has simply not had
    # time to return yet and would be mixed in with real non-returners.
    _fu1 = eff[(EXPORT_DATE - eff['close_date']).dt.days >= 365]
    _nr = _fu1[~((_fu1['event'] == 1) & (_fu1['duration'] <= 365))]
    print(f"Closed cases with >=1 year of follow-up: {len(_fu1):,} of {len(eff):,}")
    print(f"  ...no return within 1 year: {len(_nr):,} ({len(_nr)/max(len(_fu1),1)*100:.1f}%)")
    print('  (cases closed less than a year before the export are left out - too early to tell)')
    if len(_nr) >= 30:
        print('\nProfile of cases with no return within 1 year:')
        for _col, _lab in [('eindresultaat', 'Final result'), ('reach_status', 'Reach status'),
                           ('recidive_hist', 'Prior recidive flag')]:
            if _col in _nr.columns:
                vc = _nr[_col].astype(str).value_counts().head(6)
                print(f'- {_lab}:')
                print((vc / len(_nr) * 100).round(1).rename('%').to_string()); print()

    # selective vs total exit per creditor type
    _EXIT = 180
    _ts = signalen_x.dropna(subset=['Hash', 'Datum melding', 'Type melder'])
    _last_all  = _ts.groupby('Hash')['Datum melding'].max()
    _hh_gone   = (EXPORT_DATE - _last_all).dt.days >= _EXIT
    _lt = _ts.groupby(['Hash', 'Type melder'])['Datum melding'].max().reset_index()
    _lt['hh_last']  = _lt['Hash'].map(_last_all)
    _lt['selective'] = (_lt['hh_last'] - _lt['Datum melding']).dt.days >= _EXIT
    _lt['hh_gone']   = _lt['Hash'].map(_hh_gone)
    rows = []
    for _ty, g in _lt.groupby('Type melder'):
        if len(g) < 30:
            continue
        rows.append((str(_ty)[:28], len(g), int((g['selective'] & ~g['hh_gone']).sum()),
                     int((g['selective'] & g['hh_gone']).sum()), int((~g['selective'] & g['hh_gone']).sum())))
    if rows:
        print(f"Exits per creditor type (exit = no signal of that type for >={_EXIT}d):")
        display(pd.DataFrame(rows, columns=['type', 'household-type pairs',
                                            'stopped this type, still reported by others',
                                            'stopped this type earlier, now gone everywhere',
                                            'stopped with all types at once (now gone)']))
        print('Stopped this type while still reported by others = consistent with resolving that debt, not proof.')
        print('Gone everywhere = resolved OR left the reporting system (ambiguous).')
    else:
        print('Too few household-type pairs for the exit analysis.')
    _section_done('13d. G4 - Silent Success vs Disappearance')
except Exception as _err:
    _section_failed('13d. G4 - Silent Success vs Disappearance', _err)


## 14. Module 2 — Trajectory & Early-Warning Science (G1, G2, G3, G5)

Tests the founding premise of Vroegsignalering (are some signals genuinely *early*?) and maps
the recurring population. Uses the episode table from §13. All "first" statements mean *first
observed since 2021* (Wgs start) — earlier history is not visible.


### 14a. G1 — Debt Cascade: Transitions Between Creditor Types

Households in trouble choose what to stop paying first. The within-household transition matrix
P(next episode type | current type) reveals a payment-priority ladder; the first-mover index
shows which types open a household's observed history. Types that systematically come first
are candidates for early-warning signals.

**G1b** classifies every episode after the first using **all** reporter types in each episode
(not only the first): *NEW type* = a type never seen in any earlier signal of the household;
*same type again* = all its types were in the previous episode; *known other type* otherwise.
Types are reporter **types** (e.g. any energy supplier), not individual reporters.


In [ ]:
_report_section('14a. G1 - Debt Cascade: Transitions Between Creditor Types')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _seq = epi.dropna(subset=['ep_type']).sort_values(['Hash', 'epnum']).copy()
    _seq['next_type'] = _seq.groupby('Hash')['ep_type'].shift(-1)
    _tr = _seq.dropna(subset=['next_type'])
    if len(_tr) < 100:
        print(f'Too few episode transitions ({len(_tr)}) for a stable matrix.')
    else:
        mat = pd.crosstab(_tr['ep_type'], _tr['next_type'], normalize='index') * 100
        _rowcnt = _tr['ep_type'].value_counts()
        mat = mat.loc[[i for i in mat.index if _rowcnt.get(i, 0) >= 30]]
        print(f"Episode-to-episode transitions: {len(_tr):,}  (rows with n>=30 shown)")
        print('P(next episode type | current type), % per row:')
        display(mat.round(1))

        fig, ax = plt.subplots(figsize=(9, max(4, 0.8 * len(mat))))
        im = ax.imshow(mat.values, cmap='Blues', aspect='auto')
        ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels([str(c)[:18] for c in mat.columns], rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(len(mat.index)));  ax.set_yticklabels([str(i)[:18] for i in mat.index], fontsize=8)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(j, i, f'{mat.values[i, j]:.0f}', ha='center', va='center', fontsize=8,
                        color='white' if mat.values[i, j] > 50 else 'black')
        ax.set_title('G1 - P(next episode type | current type) %')
        plt.tight_layout(); plt.show()

        _first_ty1 = _seq.groupby('Hash')['ep_type'].first()
        fm = _first_ty1.value_counts(); base = _seq['ep_type'].value_counts()
        tab = pd.DataFrame({'first_episodes': fm, 'all_episodes': base}).fillna(0)
        tab['first_mover_index'] = ((fm / fm.sum()) / (base / base.sum())).round(2)
        tab = tab[tab['all_episodes'] >= 30].sort_values('first_mover_index', ascending=False)
        print('First-mover index (>1 = over-represented as the FIRST observed type of a household):')
        display(tab)
        print('NOTE: "first" = first observed since 2021 (Wgs start); pre-2021 history is censored.')

        # ── G1b: same-type recurrence vs cross-type cascade (all types per episode, 13 prep) ──
        _ek = epi.sort_values(['Hash', 'epnum']).copy()
        _ek['current'] = _ek.groupby('Hash')['ep_type'].shift(1)          # first type of the previous episode
        _ek['year'] = _ek.groupby('Hash')['end'].shift(1).dt.year
        _dd = (_ek.dropna(subset=['transition_kind'])[['current', 'transition_kind', 'year']]
               .rename(columns={'transition_kind': 'transition'}))
        if len(_dd) >= 100:
            _ov = _dd['transition'].value_counts(normalize=True) * 100
            print('\nG1b - what kind of transition is recurrence? (all transitions):')
            print(_ov.round(1).rename('%').to_string())
            _bt = pd.crosstab(_dd['current'], _dd['transition'], normalize='index') * 100
            _cnt2 = _dd['current'].value_counts()
            _bt = _bt.loc[[i for i in _bt.index if _cnt2.get(i, 0) >= 30]]
            if len(_bt):
                print('\nPer current type (% of its transitions, n>=30):')
                display(_bt.round(1))
            _yy2 = _dd.dropna(subset=['year'])
            if len(_yy2) >= 100 and _yy2['year'].nunique() > 1:
                _ytr = pd.crosstab(_yy2['year'].astype(int), _yy2['transition'], normalize='index') * 100
                print('\nTransition mix per year (%, year of the earlier episode):')
                print(_ytr.round(1).to_string())
            print("'same type again' = unresolved issue or reporting rhythm (see H2);")
            print("'NEW type (cascade)' = a reporter TYPE never seen in any earlier signal of the household -")
            print('consistent with spreading distress (types, not individual creditors).')
        else:
            print('\nG1b - too few transitions for the decomposition.')
    _section_done('14a. G1 - Debt Cascade: Transitions Between Creditor Types')
except Exception as _err:
    _section_failed('14a. G1 - Debt Cascade: Transitions Between Creditor Types', _err)


### 14b. G2 — Breadth vs Depth

Round 1 found debt *amount* barely predicts recurrence. Hypothesis: the **number of distinct
creditor types** a household has accumulated is the real severity marker. Both enter one model
head-to-head; if breadth dominates, the number of creditor types is the stronger marker of later
signals in these data (an association, not a triage rule). The model
also adjusts for prior history (`prior_band`): breadth grows with how long a household has been
observed, so without it breadth could simply stand in for a longer history.


In [ ]:
_report_section('14b. G2 - Breadth vs Depth')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    # breadth is computed in the §13 prep cell (shared with E1b); recompute only if missing
    if 'breadth_band' not in eff.columns:
        _ts2 = signalen_x.dropna(subset=['Hash', 'Datum melding', 'Type melder']).sort_values('Datum melding')
        _by_h = {h: list(zip(g['Datum melding'].tolist(), g['Type melder'].tolist()))
                 for h, g in _ts2.groupby('Hash')}
        def _breadth(h, t):
            lst = _by_h.get(h)
            if not lst or pd.isna(t):
                return np.nan
            return len({ty for d0, ty in lst if d0 <= t})
        eff['breadth'] = [_breadth(h, t) for h, t in zip(eff['Hash'], eff['signal_date'])]
        eff['breadth_band'] = pd.cut(eff['breadth'], bins=[0, 1, 2, 99], labels=['1', '2', '3+'])

    print('90-day return by creditor breadth at case start:')
    for b, g in eff.dropna(subset=['breadth_band']).groupby('breadth_band'):
        if len(g) >= 30:
            print(f'  {b:>3} creditor type(s): {cuminc_str(g["duration"], g["event"], 90, 5)}  (n={len(g):,})')

    print('\nHead-to-head Cox (breadth and debt amount in ONE model, adjusted for prior history):')
    if eff['breadth_band'].dropna().nunique() < 2:
        hr = None
        print('  Creditor breadth does not vary in this dataset - Cox skipped.')
    else:
        # prior_band: breadth grows with how long a household has been observed, so without it
        # breadth could simply stand in for 'longer history'
        hr = cox_hr(eff, 'duration ~ C(breadth_band) + C(debt_band) + recidive_hist + C(prior_band)',
                    ['breadth_band', 'debt_band', 'recidive_hist', 'prior_band'])
    if hr is not None:
        display(hr)
        print('If breadth HRs are large while debt-band HRs stay ~1, the number of creditor types is the')
        print('stronger marker of later signals in these data (an association, not a triage rule).')
    _section_done('14b. G2 - Breadth vs Depth')
except Exception as _err:
    _section_failed('14b. G2 - Breadth vs Depth', _err)


### 14c. G3 — Trajectory Typology

Maps the recurring population. Base classes by volume — **one-off** (1 episode), **episodic**
(2–3), **chronic** (4+) — plus four independent behavioural flags per household:
**escalating** (gaps shortening), **resolving** (gaps lengthening or inactive ≥1 year),
**monthly cadence** (administrative re-reporting pattern) and **cascading** (a later episode
introduced a new creditor type). The stakeholder headline: what share of all signal volume
comes from the most chronic 5–10% of households?


In [ ]:
_report_section('14c. G3 - Trajectory Typology')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _hh = epi.groupby('Hash').agg(n_ep=('epnum', 'max'), last=('end', 'max'),
                                  n_types=('ep_type', 'nunique')).reset_index()
    _hh['cls'] = np.where(_hh['n_ep'] == 1, 'one-off',
                  np.where(_hh['n_ep'] <= 3, 'episodic (2-3)', 'chronic (4+)'))
    _hh['inactive_365d'] = (EXPORT_DATE - _hh['last']).dt.days >= 365

    # gap-trend flag (same rule as E2)
    _g = epi.dropna(subset=['gap_days'])
    _trend = {}
    for _h, gg in _g.groupby('Hash'):
        v = gg.sort_values('epnum')['gap_days'].values
        if len(v) >= 2:
            _trend[_h] = ('escalating' if v[-1] < v[0] * 0.75
                          else 'resolving' if v[-1] > v[0] * 1.33 else 'stable')
    _hh['gap_trend'] = _hh['Hash'].map(_trend)

    # monthly-cadence flag (same rule as 14e): >=50% of signal gaps 25-35d, <=2 creditor types
    _sg = signalen_x.dropna(subset=['Hash', 'Datum melding']).sort_values(['Hash', 'Datum melding']).copy()
    _sg['_gap'] = _sg.groupby('Hash')['Datum melding'].diff().dt.days
    _cad = _sg.dropna(subset=['_gap']).groupby('Hash')['_gap'].apply(lambda s: (s.between(25, 35)).mean() >= 0.5)
    _nty = signalen_x.dropna(subset=['Hash']).groupby('Hash')['Type melder'].nunique()
    _hh['flag_monthly_cadence'] = _hh['Hash'].map(_cad).fillna(False) & (_hh['Hash'].map(_nty) <= 2)
    _hh['flag_escalating'] = _hh['gap_trend'] == 'escalating'
    _hh['flag_resolving']  = (_hh['gap_trend'] == 'resolving') | ((_hh['n_ep'] >= 2) & _hh['inactive_365d'])
    _casc = set(epi.loc[epi['transition_kind'].eq('NEW type (cascade)'), 'Hash'])
    _hh['flag_cascading']  = _hh['Hash'].isin(_casc)   # a later episode brought a type never seen before

    _prof = signalen_x.dropna(subset=['Hash']).groupby('Hash').agg(
        n_sig=('SignaalID', 'count'),
        children=('Minderjarige kinderen', lambda s: (s.astype(str).str.lower() == 'ja').any()))
    _hh = _hh.merge(_prof, on='Hash', how='left')

    print('Trajectory classes x behavioural flags (flags are independent, not exclusive):')
    rows = []
    for c, g in _hh.groupby('cls'):
        rows.append((c, len(g), round(len(g)/len(_hh)*100, 1), round(g['n_sig'].mean(), 1),
                     round(g['children'].mean()*100, 1),
                     round(g['flag_escalating'].mean()*100, 1),
                     round(g['flag_resolving'].mean()*100, 1),
                     round(g['flag_monthly_cadence'].mean()*100, 1),
                     round(g['flag_cascading'].mean()*100, 1)))
    tab = pd.DataFrame(rows, columns=['class', 'households', '% of hh', 'mean signals', '% children',
                                      '% escalating', '% resolving', '% monthly cadence', '% cascading'])
    display(tab)
    print('escalating = inter-episode gaps shortening | resolving = gaps lengthening or inactive >=1y')
    print('monthly cadence = >=50% of gaps 25-35d from <=2 creditors (administrative re-reporting)')
    print('cascading = a later episode introduced a new creditor type (multi-creditor spread)')

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(tab['class'], tab['% of hh'], color=COLORS[0])
    for _b, v in zip(ax.patches, tab['% of hh']):
        ax.text(_b.get_width() + 0.5, _b.get_y() + _b.get_height()/2, f'{v}%', va='center', fontsize=9)
    ax.set_xlabel('% of households'); ax.set_title('G3 - Trajectory classes')
    plt.tight_layout(); plt.show()

    _sc = _prof['n_sig'].sort_values(ascending=False)
    _tot = _sc.sum()
    for pct in (5, 10):
        k = max(int(len(_sc) * pct / 100), 1)
        print(f"Top {pct:>2}% households ({k:,}) generate {_sc.head(k).sum()/_tot*100:.1f}% of all signals")

    _first_ep_yr = epi.groupby('Hash')['start'].min().dt.year
    _hh['_cohort'] = _hh['Hash'].map(_first_ep_yr)
    _cc = _hh.dropna(subset=['_cohort']).copy()
    if _cc['_cohort'].nunique() > 1:
        _ct = pd.crosstab(_cc['_cohort'].astype(int), _cc['cls'], normalize='index') * 100
        print('\nClass shares by first-episode year (cohort view, %):')
        print(_ct.round(1).to_string())
        print('(later cohorts have had less time to accumulate episodes - compare with care)')
    _section_done('14c. G3 - Trajectory Typology')
except Exception as _err:
    _section_failed('14c. G3 - Trajectory Typology', _err)


### 14d. G5 — Early-Warning Value per Signal Partner

One scorecard per partner type: how often it opens a household's history, how often another
episode follows, how fast, how reachable its cases are, and how often the recorded final result
is payment or a payment arrangement. Caveat: municipal handling differs per partner (forwarding 76–99%), so reach and
resolution mix partner signal quality with municipal triage. "Followed by a next episode" is
censoring-aware (within 365 days; episodes near the export count only for the time observed).


In [ ]:
_report_section('14d. G5 - Early-Warning Value per Signal Partner')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _first_ty = epi.dropna(subset=['ep_type']).sort_values(['Hash', 'epnum']).groupby('Hash')['ep_type'].first()
    _fm_share = _first_ty.value_counts(normalize=True) * 100
    rows = []
    for _ty in epi['ep_type'].dropna().unique():
        ep_t = epi[epi['ep_type'] == _ty]
        if len(ep_t) < 30:
            continue
        # censoring-aware: an episode near the export has not had a full year to be followed yet
        _es = epi_surv[epi_surv['ep_type'] == _ty]
        followed = cuminc_at(_es['gap_dur'], _es['gap_event'], 365) * 100
        med_gap  = ep_t.loc[ep_t['gap_event'] == 1, 'gap_days'].median()   # among followed episodes
        ge = eff[eff['Type melder'] == _ty]
        _gk = ge[ge['reach_status'].isin(['reached', 'not reached'])]
        reach = _gk['reach_status'].eq('reached').mean() * 100 if len(_gk) >= 30 else np.nan
        resol = ge['result_class'].isin(['paid', 'payment arrangement']).mean() * 100 if len(ge) >= 30 else np.nan
        rows.append((str(_ty)[:26], len(ep_t), round(_fm_share.get(_ty, 0), 1), round(followed, 1),
                     med_gap, round(reach, 1) if pd.notna(reach) else np.nan,
                     round(resol, 1) if pd.notna(resol) else np.nan))
    if rows:
        tab = pd.DataFrame(rows, columns=['partner type', 'episodes', '% households: first type',
                                          '% followed by next episode within 365d',
                                          'median days to next (followed episodes)',
                                          'reach % (its cases, known status)', '% paid/arrangement'])
        display(tab.sort_values('% households: first type', ascending=False))
        print('A type that often comes first AND is often followed by another episode is a CANDIDATE')
        print('early-warning signal - a pattern to investigate, not proof.')
        print('"% followed" is censoring-aware (Kaplan-Meier, within 365 days).')
        print('Caveat: reach/resolution reflect municipal triage per partner as well as the partner itself.')
    else:
        print('Too few episodes per partner type.')
    _section_done('14d. G5 - Early-Warning Value per Signal Partner')
except Exception as _err:
    _section_failed('14d. G5 - Early-Warning Value per Signal Partner', _err)


### 14e. Deep-Dive: The 10 Most-Signalling Households

High-volume households are a normal part of the data — none are excluded. This cell profiles
the ten most-signalling households to show what kind of recurrence they represent. The
timeline plot helps tell two patterns apart: evenly spaced ~30-day dots from one or two creditors
suggest **administrative re-reporting**; clustered bursts across several creditors suggest
**repeated distress**. Households are labelled by rank only — hashes are never shown.


In [ ]:
_report_section('14e. Deep-Dive: The 10 Most-Signalling Households')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _cnt = signalen_x.dropna(subset=['Hash']).groupby('Hash').size().sort_values(ascending=False)
    TOP10 = _cnt.head(10)
    print(f"Top 10 households: {TOP10.sum():,} of {int(_cnt.sum()):,} signals "
          f"({TOP10.sum()/_cnt.sum()*100:.1f}% of total volume)")

    rows = []
    for _rank, (_h, _n) in enumerate(TOP10.items(), 1):
        g = signalen_x[signalen_x['Hash'] == _h].sort_values('Datum melding')
        dts = g['Datum melding'].dropna()
        span_m = max((dts.max() - dts.min()).days / 30.44, 1) if len(dts) > 1 else 1
        gaps = dts.diff().dt.days.dropna()
        monthly_pct = (gaps.between(25, 35)).mean() * 100 if len(gaps) else np.nan
        ntypes = int(g['Type melder'].nunique())
        _ep = epi[epi['Hash'] == _h]
        n_ep = int(_ep['epnum'].max()) if len(_ep) else 0
        ge = eff[eff['Hash'] == _h]
        if pd.notna(monthly_pct) and monthly_pct >= 50 and ntypes <= 2:
            pattern = 'monthly re-reporting'
        elif n_ep >= 3:
            pattern = 'episodic bursts'
        else:
            pattern = 'mixed / single episode'
        rows.append((f'HH-{_rank}', _n, round(span_m), round(_n / span_m, 1), ntypes, n_ep,
                     round(gaps.median()) if len(gaps) else np.nan,
                     round(monthly_pct) if pd.notna(monthly_pct) else np.nan,
                     'yes' if (len(ge) and ge['reached'].any()) else 'no',
                     'yes' if (len(ge) and (ge['arm'] == 'C_accepted').any()) else 'no',
                     int((EXPORT_DATE - dts.max()).days) if len(dts) else np.nan,
                     pattern))
    tab = pd.DataFrame(rows, columns=['household', 'signals', 'span (months)', 'signals/month',
                                      'creditor types', 'episodes', 'median gap (d)', '% gaps 25-35d',
                                      'ever reached', 'ever accepted help', 'days since last signal',
                                      'pattern'])
    display(tab)
    print("'monthly re-reporting' = regular ~30d gaps from <=2 creditors -> suggests administrative cadence,")
    print("possibly one unresolved debt reported month after month.")
    print("'episodic bursts' = distinct episodes -> consistent with repeated distress.")

    # ── Signal timelines ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, 5))
    for i, (_h, _n) in enumerate(TOP10.items()):
        dts = signalen_x.loc[signalen_x['Hash'] == _h, 'Datum melding'].dropna()
        ax.scatter(dts, [len(TOP10) - i] * len(dts), s=14, color=COLORS[i % len(COLORS)])
    ax.set_yticks(range(len(TOP10), 0, -1))
    ax.set_yticklabels([f'HH-{r}' for r in range(1, len(TOP10) + 1)])
    ax.set_xlabel('date'); ax.set_title('Signal timelines - 10 most-signalling households')
    plt.tight_layout(); plt.show()

    # ── Are heavy households treated / responding differently? ────────────────────
    _heavy = set(TOP10.index)
    gh = eff[eff['Hash'].isin(_heavy)]; gr = eff[~eff['Hash'].isin(_heavy)]
    if len(gh) >= 10 and len(gr) >= 100:
        print('\nHeavy-tail vs all other households (closed cases):')
        print(f"  cases:            {len(gh):>6,}  vs {len(gr):>6,}")
        _rk = lambda x: x.loc[x['reach_status'].isin(['reached', 'not reached']), 'reach_status'].eq('reached').mean() * 100
        print(f"  reach rate (known status): {_rk(gh):5.1f}%  vs {_rk(gr):5.1f}%")
        print(f"  help accepted:    {(gh['arm']=='C_accepted').mean()*100:5.1f}%  vs {(gr['arm']=='C_accepted').mean()*100:5.1f}%")
        print(f"  90d return:       {cuminc_str(gh['duration'], gh['event'], 90, 5)}  vs {cuminc_str(gr['duration'], gr['event'], 90, 5)}")
        print('If heavy households are rarely reached, the system may be repeatedly signalling')
        print('exactly the households it never manages to contact.')
    else:
        print('\nToo few closed cases among the top-10 households for a comparison.')
    _section_done('14e. Deep-Dive: The 10 Most-Signalling Households')
except Exception as _err:
    _section_failed('14e. Deep-Dive: The 10 Most-Signalling Households', _err)


## 15. Module 3 — Policy Shocks & Operational Planning (H2, F1, F2, H1)

H2 comes first: it measures how much of "recurrence" is reporting rhythm, which calibrates the
headline metric and is a prerequisite for reading F1's trends.


### 15a. H2 — Reporting Cadence per Partner (metric calibration)

How much of measured recurrence is household behaviour, and how much is partners' reporting
rhythm? Measures inter-report intervals per partner **type** and per individual **reporter**
(`Melder ID`), and recomputes the headline return rate under alternative minimum-gap
definitions — the spread shows how sensitive the rate is to that definition (it does not measure
the administrative share; §9d repeats this by year and group).


In [ ]:
_report_section('15a. H2 - Reporting Cadence per Partner')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _ts3 = (signalen_x.dropna(subset=['Hash', 'Datum melding', 'Type melder'])
            .sort_values(['Hash', 'Type melder', 'Datum melding']).copy())
    _ts3['_gap'] = _ts3.groupby(['Hash', 'Type melder'])['Datum melding'].diff().dt.days
    gaps = _ts3.dropna(subset=['_gap'])
    gaps = gaps[gaps['_gap'] > 0]
    rows = []
    for _ty, g in gaps.groupby('Type melder'):
        if len(g) < 50:
            continue
        rows.append((_ty, len(g), g['_gap'].median(),
                     round((g['_gap'].between(25, 35)).mean() * 100, 1),
                     round((g['_gap'] <= MIN_GAP_DAYS).mean() * 100, 1)))
    if rows:
        _h2 = pd.DataFrame(rows, columns=['partner type', 'intervals', 'median days',
                                          '% in 25-35d (monthly cadence)', f'% <= {MIN_GAP_DAYS}d'])
        if 'Melder ID' in signalen_x.columns:
            _tr = (signalen_x.dropna(subset=['Hash', 'Datum melding', 'Melder ID', 'Type melder'])
                   .sort_values(['Hash', 'Melder ID', 'Datum melding']).copy())
            _tr['_gap'] = _tr.groupby(['Hash', 'Melder ID'])['Datum melding'].diff().dt.days
            _tr = _tr[_tr['_gap'] > 0]
            _byrep = _tr.groupby('Type melder')['_gap'].agg(
                **{'median days (same reporter)': 'median',
                   '% 25-35d (same reporter)': lambda s: round(s.between(25, 35).mean() * 100, 1)})
            _h2 = _h2.merge(_byrep, left_on='partner type', right_index=True, how='left')
        print('Inter-report intervals within the same household & partner TYPE, and within the same')
        print('household & individual REPORTER (Melder ID):')
        display(_h2)
    else:
        print('Too few repeated same-type reports.')

    if len(gaps) >= 400 and gaps['Datum melding'].dt.year.nunique() > 1:
        gaps['_yr'] = gaps['Datum melding'].dt.year
        _keepc = [t for t in gaps['Type melder'].unique() if (gaps['Type melder'] == t).sum() >= 200]
        if _keepc:
            _pvh = (gaps[gaps['Type melder'].isin(_keepc)]
                    .pivot_table(index='Type melder', columns='_yr', values='_gap', aggfunc='median'))
            print('\nMedian inter-report interval per partner per year (days) - cadence stability:')
            print(_pvh.round(0).to_string())

    print('\nSensitivity: 90-day case return rate under alternative minimum-gap definitions:')
    for gp in [7, 14, 30, 60]:
        _nx = []
        for _h, _c in zip(eff['Hash'], eff['close_date']):
            arr = _sig_by_hash.get(_h)
            if not arr:
                _nx.append(pd.NaT); continue
            i = bisect.bisect_right(arr, _c + pd.Timedelta(days=gp))
            _nx.append(arr[i] if i < len(arr) else pd.NaT)
        _nx = pd.to_datetime(pd.Series(_nx, index=eff.index, dtype=object), errors='coerce')
        _ev = _nx.notna().astype(int)
        _du = (_nx.fillna(EXPORT_DATE) - eff['close_date']).dt.days
        _ok = _du > 0
        print(f'  gap >{gp:>2}d: 90d return = {cuminc_str(_du[_ok], _ev[_ok], 90, 5)}   '
              f'(events {int(_ev[_ok].sum()):,})')
    print('The spread shows how much the return rate depends on the gap definition (9d repeats this by')
    print('year and group); it does not measure how much recurrence is administrative.')
    _section_done('15a. H2 - Reporting Cadence per Partner')
except Exception as _err:
    _section_failed('15a. H2 - Reporting Cadence per Partner', _err)


### 15b. F1 — Energy Crisis 2022: Comparative Trends

Did the energy price shock create *new* distressed households or intensify existing ones, and
did relief (energietoeslag late 2022, price cap Jan 2023) show up in the series? Water,
health-insurer and housing signals serve as comparison series. Framed as comparative trends —
**not** clean difference-in-differences. The 2021 ramp (Wgs partner onboarding) is excluded.


In [ ]:
_report_section('15b. F1 - Energy Crisis 2022: Comparative Trends')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _s = signalen_x.dropna(subset=['Datum melding', 'Type melder']).copy()
    _s = _s[_s['Datum melding'] >= pd.Timestamp('2021-07-01')]     # skip Wgs onboarding ramp
    if len(_s) < 500 or _s['Datum melding'].dt.year.nunique() < 2:
        print('Data span too short for the crisis analysis - skipped.')
    else:
        _s['month'] = _s['Datum melding'].dt.to_period('M').dt.to_timestamp()
        _piv = _s.pivot_table(index='month', columns='Type melder', values='SignaalID',
                              aggfunc='count').fillna(0)
        _keep = [c for c in _piv.columns if _piv[c].sum() >= 200]
        fig, ax = plt.subplots(figsize=(13, 5))
        for i, c in enumerate(_keep):
            ax.plot(_piv.index, _piv[c].rolling(3, center=True).mean(),
                    label=str(c)[:26], linewidth=2, color=COLORS[i % len(COLORS)])
        for d0, lab in [('2022-03-01', 'price surge'), ('2022-11-01', 'energietoeslag'),
                        ('2023-01-01', 'price cap')]:
            if pd.Timestamp(d0) > _piv.index.min():
                ax.axvline(pd.Timestamp(d0), color='red', linestyle='--', alpha=0.5)
                ax.text(pd.Timestamp(d0), ax.get_ylim()[1] * 0.97, lab, rotation=90,
                        fontsize=8, va='top')
        ax.set_title('F1 - Monthly signals per partner type (3-month rolling mean)')
        ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()

        _per = [('2021H2', '2021-07-01', '2022-01-01'), ('2022 crisis', '2022-01-01', '2023-01-01'),
                ('2023 cap', '2023-01-01', '2024-01-01'), ('2024+', '2024-01-01', '2027-01-01')]
        rows = []
        for lab, a, b in _per:
            m = _s[(_s['Datum melding'] >= a) & (_s['Datum melding'] < b)]
            if len(m) == 0:
                continue
            months = max((pd.Timestamp(b) - pd.Timestamp(a)).days / 30.44, 1)
            months = min(months, max((m['Datum melding'].max() - pd.Timestamp(a)).days / 30.44, 1))
            r0 = {'period': lab}
            for c in _keep:
                r0[str(c)[:20]] = round((m['Type melder'] == c).sum() / months, 1)
            rows.append(r0)
        print('Signals per month by period (values behind the chart):')
        display(pd.DataFrame(rows).set_index('period'))

        _pm = _piv[_keep].astype(int).copy()
        _pm.index = _pm.index.strftime('%Y-%m')
        print('Monthly signal counts per partner type (full series behind the chart,')
        print('for future re-analysis of the trends):')
        print(_pm.to_string())

        _first_seen = signalen_x.dropna(subset=['Hash', 'Datum melding']).groupby('Hash')['Datum melding'].min()
        _s['is_new'] = _s['Datum melding'] <= _s['Hash'].map(_first_seen)
        _is_en = _s['Type melder'].astype(str).str.lower().str.contains('energie')
        rows = []
        for lab, a, b in _per:
            m = _s[(_s['Datum melding'] >= a) & (_s['Datum melding'] < b)]
            en = m[_is_en.reindex(m.index, fill_value=False)]
            ot = m[~_is_en.reindex(m.index, fill_value=False)]
            if len(en) >= 30 and len(ot) >= 30:
                rows.append((lab, round(en['is_new'].mean() * 100, 1), round(ot['is_new'].mean() * 100, 1)))
        if rows:
            print('\nShare of signals from FIRST-TIME households (new inflow vs known population):')
            display(pd.DataFrame(rows, columns=['period', 'energy signals: % new hh', 'other signals: % new hh']))
        print('Caveats: comparative trends, not DiD; "new" = first observed since 2021;')
        print('check H2 that partner reporting practices were stable before reading effects.')
    _section_done('15b. F1 - Energy Crisis 2022: Comparative Trends')
except Exception as _err:
    _section_failed('15b. F1 - Energy Crisis 2022: Comparative Trends', _err)


### 15c. F2 — Seasonality by Mechanism

Do signal types follow their expected seasonal mechanisms (energy after winter bills, health
insurance after the January premium reset, water as the low-seasonality control)? Directly
usable for outreach capacity planning. Only **complete calendar years** from 2022 are used (2021
is the Wgs ramp year; the export year is only partly observed and would add an extra year to the
first months, faking a first-half peak), and each year is normalised to its own monthly average
so volume growth between years does not leak into the pattern.


In [ ]:
_report_section('15c. F2 - Seasonality by Mechanism')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _s2 = signalen_x.dropna(subset=['Datum melding', 'Type melder']).copy()
    # Only COMPLETE calendar years from 2022 (2021 = Wgs ramp). A partly observed year - e.g. the
    # export year, which ends mid-year - adds an extra year to some calendar months only and fakes
    # a seasonal pattern. Each year is normalised to its own monthly average, so volume growth
    # between years does not leak into the pattern either.
    _lastd = _s2['Datum melding'].max()
    _full_yrs = [y for y in range(2022, _lastd.year + 1) if pd.Timestamp(f'{y}-12-31') <= _lastd]
    _s2 = _s2[_s2['Datum melding'].dt.year.isin(_full_yrs)]
    if len(_s2) < 500 or len(_full_yrs) < 2:
        print('Fewer than two complete calendar years since 2022 - seasonality profile skipped.')
    else:
        print(f'Complete calendar years used: {_full_yrs[0]}-{_full_yrs[-1]} '
              '(partly observed years left out; each year normalised to its own average)')
        _s2['cal_month'] = _s2['Datum melding'].dt.month.astype(int)
        _s2['_yr'] = _s2['Datum melding'].dt.year.astype(int)
        _pv = (_s2.pivot_table(index=['_yr', 'cal_month'], columns='Type melder', values='SignaalID',
                               aggfunc='count')
                  .reindex(pd.MultiIndex.from_product([_full_yrs, range(1, 13)],
                                                      names=['_yr', 'cal_month']))
                  .fillna(0))
        _keep = [c for c in _pv.columns if _pv[c].sum() >= 300]
        if not _keep:
            print('No partner type with enough volume.')
        else:
            _ymean = _pv[_keep].groupby(level='_yr').transform('mean').replace(0, np.nan)
            _idx = (_pv[_keep] / _ymean * 100).groupby(level='cal_month').mean()
            fig, ax = plt.subplots(figsize=(12, 5))
            for i, c in enumerate(_keep):
                ax.plot(_idx.index, _idx[c], marker='o', label=str(c)[:24], color=COLORS[i % len(COLORS)])
            ax.axhline(100, color='gray', linestyle='--')
            ax.set_xticks(range(1, 13)); ax.set_xlabel('calendar month')
            ax.set_title('F2 - Seasonality index per partner type (100 = type average, complete years)')
            ax.legend(fontsize=8)
            plt.tight_layout(); plt.show()
            print('Seasonality index (values behind the chart):')
            display(_idx.round(0))
        eff['_close_q'] = eff['close_date'].dt.quarter
        print('\n90-day return by case-close quarter:')
        for q, g in eff.dropna(subset=['_close_q']).groupby('_close_q'):
            if len(g) >= 100:
                print(f'  Q{int(q)}: {cuminc_str(g["duration"], g["event"], 90, 5)}  (n={len(g):,})')
    _section_done('15c. F2 - Seasonality by Mechanism')
except Exception as _err:
    _section_failed('15c. F2 - Seasonality by Mechanism', _err)


### 15d. H1 — Risk Stratification at First Contact

Which first-episode features are associated with becoming a chronic returner (≥3 episodes
within 18 months)? Interpretable logistic model on intake-time features only, with **temporal
validation** (train on households first seen in 2022–2023, test on 2024+) and calibration —
not just AUC. Households first seen in **2021** are left out of training: monitoring started
then, so many of them already had problems for years and look far more chronic than new
households (start-up effect). Episodes use the 45-day rule from §13, so monthly re-reports of
one debt do not count as new episodes. Framing: transparent risk-stratification research, not a decision model.

This is a **cohort hold-out** (households first seen in 2022–2023 vs 2024), not a simulation of
a model deployed at a fixed date: 2023 households need 18 months of follow-up before their label
is known. Accuracy applies to the households that could be scored (exclusions are printed). A
model that does not converge or separates perfectly is reported as not estimable — no AUC,
calibration or odds ratios are shown for it. H1 and H1b are estimated independently, so one
failing does not stop the other.

**H1b** adds a second target: households at or above the training years' 90th percentile of
signal volume within 18 months — predicting future *workload*, which partly reflects reporting
cadence (see H2). Because many households share the same count, the share at or above the
threshold can differ from exactly 10% (it is printed).


In [ ]:
_report_section('15d. H1 - Risk Stratification at First Contact')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    _FOLLOW = 548   # 18 months
    _h1 = epi.groupby('Hash').agg(first=('start', 'min')).reset_index()
    _h1 = _h1[_h1['first'] <= EXPORT_DATE - pd.Timedelta(days=_FOLLOW)]
    _epw = epi.merge(_h1, on='Hash')
    _epw = _epw[_epw['start'] <= _epw['first'] + pd.Timedelta(days=_FOLLOW)]
    _h1['y'] = (_h1['Hash'].map(_epw.groupby('Hash')['epnum'].max()) >= 3).astype(int)

    # H1b target: signal volume within the same 18-month window (workload burden)
    _sw = signalen_x.dropna(subset=['Hash', 'Datum melding']).merge(_h1[['Hash', 'first']], on='Hash')
    _sw = _sw[(_sw['Datum melding'] >= _sw['first']) &
              (_sw['Datum melding'] <= _sw['first'] + pd.Timedelta(days=_FOLLOW))]
    _h1['sig18'] = _h1['Hash'].map(_sw.groupby('Hash').size()).fillna(0)

    # the household's first signal ROW (intake-time features only; groupby.first() would fill
    # gaps with values from later signals)
    _fs = (signalen_x.dropna(subset=['Hash', 'Datum melding']).sort_values('Datum melding', kind='stable')
           .drop_duplicates('Hash', keep='first'))
    _h1 = _h1.merge(_fs[['Hash', 'Type melder', 'age_band', 'debt_band', 'Minderjarige kinderen']],
                    on='Hash', how='left')
    _h1['children'] = ja_nee(_h1['Minderjarige kinderen']).astype(float)
    d = _h1.dropna(subset=['Type melder', 'age_band', 'debt_band', 'children']).copy()

    # 2021 = monitoring start-up: households 'first seen' then include long-standing problems and are
    # far more chronic than truly new households, so they are left out of training.
    H1_TRAIN_START = pd.Timestamp('2022-01-01')
    train = d[(d['first'] >= H1_TRAIN_START) & (d['first'] < pd.Timestamp('2024-01-01'))].copy()
    test  = d[d['first'] >= pd.Timestamp('2024-01-01')].copy()
    print('Inclusion flow (who this analysis describes):')
    print(f'  households with >=1 episode:              {epi["Hash"].nunique():,}')
    print(f'  first episode >= {_FOLLOW}d before export:  {len(_h1):,}')
    print(f'  with complete first-episode features:     {len(d):,}')
    if len(d):
        print(f'  chronic (>=3 episodes in 18mo):           {d["y"].mean()*100:.1f}%')
    print(f'  temporal split: train (2022-2023) n={len(train):,} | test (2024+) n={len(test):,}')
    print(f'  left out of training (2021 start-up cohort): {int((d["first"] < H1_TRAIN_START).sum()):,}')
    print('  (cohort hold-out, not a simulation of a model deployed at a fixed date)')
    if len(d) and d['first'].dt.year.nunique() > 1:
        _cy1 = (d.assign(_yr=d['first'].dt.year).groupby('_yr')['y']
                .agg(households='count', chron='mean'))
        _cy1['chronic_%'] = (_cy1['chron'] * 100).round(1)
        print('\nChronic rate by first-episode cohort year (each cohort has the same 18-month window):')
        print(_cy1.drop(columns='chron').to_string())

    def _auc(y, p):
        y = np.asarray(y); p = np.asarray(p)
        r = pd.Series(p).rank().values
        n1 = y.sum(); n0 = len(y) - n1
        return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 * n0 > 0 else np.nan

    # Stabilise categorical levels before fitting. A level that is tiny in TRAIN (e.g. reporter
    # type 'Overige signaalpartners', which mostly appears from 2025) can be perfectly separated
    # (all chronic / all not) -> singular Hessian ("Singular matrix"). Households whose level has
    # fewer than _MIN_LVL train households are left out of the model and counted below; test
    # households with a level never seen in train cannot be scored and are left out too. Level
    # order is kept, so reference levels and the odds-ratio table are unchanged when nothing
    # needs excluding.
    _MIN_LVL = 30
    _LVL_COLS = ['Type melder', 'age_band', 'debt_band']
    _n0 = (len(train), len(test))
    for _c in _LVL_COLS:
        _cats = ([str(k) for k in train[_c].cat.categories] if hasattr(train[_c], 'cat')
                 else sorted(train[_c].astype(str).unique()))
        _s_tr, _s_te = train[_c].astype(str), test[_c].astype(str)
        _n_tr = _s_tr.value_counts()
        _keep = [k for k in _cats if _n_tr.get(k, 0) >= _MIN_LVL]
        _rare = sorted((set(_s_tr) | set(_s_te)) - set(_keep))
        if _rare:
            print(f'  NOTE: {_c}: levels with <{_MIN_LVL} train households excluded from the model: {_rare}')
        train[_c] = pd.Categorical(_s_tr.where(_s_tr.isin(_keep)), categories=_keep)
        test[_c]  = pd.Categorical(_s_te.where(_s_te.isin(_keep)), categories=_keep)
    train = train.dropna(subset=_LVL_COLS)
    test  = test.dropna(subset=_LVL_COLS)
    if (len(train), len(test)) != _n0:
        print(f'  NOTE: excluded {_n0[0] - len(train)} train and {_n0[1] - len(test)} test household(s) (rare levels).')

    _FORMULA_H1 = 'C(Q("Type melder")) + C(age_band) + C(debt_band) + children'
    # ── H1: chronic returner (>=3 episodes within 18 months) ──────────────────────
    if len(train) >= 300 and train['y'].sum() >= 30 and len(test) >= 100 and test['y'].nunique() > 1:
        m, _why = _logit_fit('y ~ ' + _FORMULA_H1, train)
        if m is None:
            print(f'\nH1 model not estimable ({_why}) - no AUC, calibration or odds ratios reported.')
        else:
            pt = m.predict(train); pv = m.predict(test)
            print(f"\nAUC train = {_auc(train['y'], pt):.3f}   AUC test (2024+) = {_auc(test['y'], pv):.3f}")
            _cb = pd.DataFrame({'p': pv, 'y': test['y'].values})
            _cb['bin'] = pd.qcut(_cb['p'], 5, duplicates='drop')
            cal = _cb.groupby('bin', observed=True).agg(predicted=('p', 'mean'),
                                                        observed=('y', 'mean'), n=('y', 'size'))
            print('\nCalibration on the test set (chronic rate, %):')
            display((cal[['predicted', 'observed']] * 100).round(1).join(cal['n']))
            _or = pd.DataFrame({'OR': np.exp(m.params), 'p': m.pvalues.round(3)}).drop(
                index='Intercept', errors='ignore').round(3)
            print('Feature associations (odds ratios, train set):')
            display(_or.sort_values('OR', ascending=False).head(12))
    else:
        print('Insufficient data for temporal validation of H1 - model skipped.')

    # ── H1b: heavy-burden target - runs independently of H1 ───────────────────────
    if len(train) >= 300 and len(test) >= 100:
        _thr = train['sig18'].quantile(0.90)
        train['y2'] = (train['sig18'] >= _thr).astype(int)
        test['y2']  = (test['sig18']  >= _thr).astype(int)
        print(f"\nH1b - heavy-burden target: at or above the training 90th percentile of signals within {_FOLLOW}d")
        print(f"      (threshold >= {_thr:.0f} signals, computed on TRAIN years only - no leakage;")
        print(f"      {train['y2'].mean()*100:.1f}% of training households are at or above it)")
        if train['y2'].sum() >= 20 and test['y2'].nunique() > 1:
            m2, _why2 = _logit_fit('y2 ~ ' + _FORMULA_H1, train)
            if m2 is None:
                print(f'  H1b model not estimable ({_why2}).')
            else:
                print(f"AUC train = {_auc(train['y2'], m2.predict(train)):.3f}   "
                      f"AUC test (2024+) = {_auc(test['y2'], m2.predict(test)):.3f}")
                _or2 = pd.DataFrame({'OR': np.exp(m2.params), 'p': m2.pvalues.round(3)}).drop(
                    index='Intercept', errors='ignore').round(3)
                display(_or2.sort_values('OR', ascending=False).head(10))
        else:
            print('  Too few heavy-burden households in train, or no variation in test - skipped.')
        print('Caveat: signal volume partly reflects creditor reporting cadence (see H2), so')
        print('H1b predicts future WORKLOAD - not purely household behaviour.')
    else:
        print('\nH1b - insufficient data for temporal validation - skipped.')
    print('\nFraming: transparent risk stratification for prioritisation research - NOT a decision model.')
    _section_done('15d. H1 - Risk Stratification at First Contact')
except Exception as _err:
    _section_failed('15d. H1 - Risk Stratification at First Contact', _err)


### 15e. W — Workflow: When Are Households Receptive, and Which Method When?

**Hypothesis:** households contacted at the very start of a problem often refuse help ("I can
fix this myself"); after months of persistent signals (recidive maand-2+ positive) they are
more receptive. If true, municipalities can allocate cheap methods to the early phase and
reserve expensive ones (phone, home visit) for the receptive phase.
**W1** measures help acceptance among *reached* cases by problem persistence (case number,
maand-depth, tenure, contact speed), with an adjusted model and a within-household check. The
adjusted model controls for long-run history (`prior_band`), not the recidive flag, which
measures nearly the same thing as maand-depth. The within-household check comes with a
**selection check**: households helped after accepting early have fewer later cases, which
pushes the check towards "accepted later".
W1 uses reached cases with a **known** help answer (accepted or declined).
**W2** describes **outbound** channels per phase (first vs repeat case of a household): attempts,
cases first reached through each channel and how many of those accepted help — *accepted cases
per 100 attempts*, with each accepted case attributed to the channel of its first successful
contact. Callbacks (the client made contact) are listed separately: the labels combine channels
and do not say which earlier attempt prompted them. Channels are chosen per case, so W2 is a
description, not a fair ranking of strategies. NB: the Wgs 4-week duty means this informs
**method choice**, not delay.


In [ ]:
_report_section('15e. W - Workflow: Receptiveness Timing & Method Allocation')
# Errors are recorded in the report and the run continues (see the banner at the top).
try:
    _require_data()      # refuses data left over from an earlier run
    # W1: is help acceptance higher when the problem is older/more persistent?
    _rc = eff_sorted[eff_sorted['arm'].isin(['B_refused', 'C_accepted'])].copy()   # reached, help answer known
    _rc['accepted'] = (_rc['arm'] == 'C_accepted')
    print(f"Population: {len(_rc):,} reached closed cases with a known help answer "
          f"({int(_rc['accepted'].sum()):,} accepted help = {_rc['accepted'].mean()*100:.1f}%)")
    print("NB: the Wgs requires municipal action within 4 weeks of each signal - this experiment")
    print("informs the CHOICE OF METHOD per phase, never delaying contact.\n")

    # persistence measures at the case's opening signal
    _mcols2 = [c for c in signalen.columns if c.startswith('Recidive maand')]
    if _mcols2:
        _sl2 = signalen.dropna(subset=['Meldingnummer']).sort_values('Datum melding', kind='stable')
        _dep = _sl2.drop_duplicates('Meldingnummer', keep='first').set_index('Meldingnummer')[_mcols2]
        _depth_n = sum((_dep[c].astype(str).str.strip().str.lower() == 'ja') for c in _mcols2)
        _rc = _rc.merge(_depth_n.rename('maand_depth'), left_on='Meldingnummer',
                        right_index=True, how='left')
        _rc['depth_band'] = pd.cut(_rc['maand_depth'], bins=[-1, 0, 1, 99],
                                   labels=['0 (new problem)', '1 month', '2+ (persistent)'])
    _first_seen2 = signalen.dropna(subset=['Hash', 'Datum melding']).groupby('Hash')['Datum melding'].min()
    _rc['tenure_d'] = (_rc['signal_date'] - _rc['Hash'].map(_first_seen2)).dt.days
    _rc['tenure_band'] = pd.cut(_rc['tenure_d'], bins=[-1, 0, 60, 180, 1e9],
                                labels=['first signal', '1-60d', '61-180d', '>180d'])
    _rc['epn_band'] = pd.cut(_rc['case_epnum'], bins=[0, 1, 2, 99], labels=['1st case', '2nd', '3rd+'])
    _rc['reachdel_band'] = pd.cut(_rc['days_to_reach'], bins=[-1, 14, 30, 1e9],
                                  labels=['<=14d', '15-30d', '>30d'])

    print('W1 - Help acceptance among reached cases, by problem persistence (n>=30 per band):')
    for _col, _lab in [('epn_band', 'Case number for this household'),
                       ('depth_band', 'Recidive-maand depth of the opening signal'),
                       ('tenure_band', 'Time since household first observed'),
                       ('reachdel_band', 'Days from signal to first successful contact')]:
        if _col not in _rc.columns:
            continue
        g = _rc.dropna(subset=[_col]).groupby(_col, observed=True)['accepted'].agg(n='count', acc='mean')
        g = g[g['n'] >= 30]
        if len(g) >= 2:
            print(f'\n- {_lab}:')
            print(g.assign(**{'accepted_%': (g['acc'] * 100).round(1)}).drop(columns='acc').to_string())

    _rc['_yr'] = _rc['close_date'].dt.year
    if _rc['_yr'].nunique() > 1:
        _ya = _rc.groupby('_yr')['accepted'].agg(n='count', acc='mean')
        _ya = _ya[_ya['n'] >= 30]
        if len(_ya) > 1:
            print('\n- Acceptance among reached cases per close year:')
            print(_ya.assign(**{'accepted_%': (_ya['acc'] * 100).round(1)}).drop(columns='acc').to_string())

    # adjusted check: persistence effect net of severity markers
    import statsmodels.formula.api as smf
    if 'depth_band' in _rc.columns:
        _d = _rc.dropna(subset=['depth_band', 'debt_band', 'age_band', 'prior_band']).copy()
        if len(_d) >= 300 and _d['accepted'].sum() >= 30 and _d['depth_band'].nunique() > 1:
            try:
                _d['acc_f'] = _d['accepted'].astype(float)
                for _bc in ['depth_band', 'debt_band', 'age_band', 'prior_band']:
                    _d[_bc] = _d[_bc].cat.remove_unused_categories()
                # clustered by household: one household's repeated cases are not independent.
                # History = prior_band, not the recidive flag (nearly the same thing as maand-depth).
                m, _why = _logit_fit('acc_f ~ C(depth_band) + C(debt_band) + C(age_band) + C(prior_band)',
                                     _d, cluster='Hash')
                if m is None:
                    raise ValueError(f'not estimable - {_why}')
                _or = pd.DataFrame({'OR': np.exp(m.params), 'p': m.pvalues.round(3)}).drop(
                    index='Intercept', errors='ignore').round(3)
                print('\nAdjusted odds ratios for acceptance (ref = new problem; controls: debt, age and')
                print('long-run history (prior_band) - not the recidive flag, which measures nearly the')
                print('same thing as maand-depth; p-values clustered by household). OR>1 for persistent =')
                print('acceptance is more common when the problem has lasted longer - an association;')
                print('persistent households may also differ in need:')
                display(_or)
            except Exception as _e:
                print(f'\nAdjusted model not reported: {_e}')
        else:
            print('\nAdjusted model skipped - too few complete reached cases.')

    # within-household: among households reached in multiple episodes with DIFFERENT responses,
    # did acceptance come in the LATER episode?
    _wd = []
    for _h, g in _rc.groupby('Hash'):
        if g['accepted'].nunique() > 1:
            _wd.append(g.loc[g['accepted'], 'case_epnum'].min() > g.loc[~g['accepted'], 'case_epnum'].min())
    if len(_wd) >= 20:
        print(f"\nWithin-household check ({len(_wd)} households reached in multiple episodes with")
        print(f"different responses): acceptance came in the LATER episode for {np.mean(_wd)*100:.0f}%.")
        print("(>50% is consistent with growing receptiveness - but see the selection check below)")
    else:
        print(f"\nWithin-household check: only {len(_wd)} discordant households - too few to read.")

    # Selection check: a household that accepted help in its FIRST reached case and was helped has
    # fewer later cases, so 'refused first, accepted later' is over-represented in the check above.
    _fr = _rc.sort_values(['Hash', 'case_epnum']).groupby('Hash')
    _first_acc = _fr['accepted'].first().astype(bool)
    _has_later = _fr.size() >= 2
    if _first_acc.sum() >= 20 and (~_first_acc).sum() >= 20:
        print(f"Selection check: a later reached case exists for {_has_later[_first_acc].mean()*100:.0f}% "
              "of households whose first reached case was ACCEPTED")
        print(f"vs {_has_later[~_first_acc].mean()*100:.0f}% whose first reached case was REFUSED. "
              "If the first is clearly lower, helped households")
        print("drop out and the check above leans towards 'later' - read it as weak evidence.")

    # ── W2: outbound channels x phase - descriptive yield ─────────────────────────
    print('\nW2 - Outbound contact channels by phase (first vs repeat case of a household; descriptive)')
    if len(contacts):
        _phase_map = dict(zip(eff_sorted['Meldingnummer'],
                              np.where(eff_sorted['case_epnum'] >= 2, 'repeat case', 'first case')))
        _ca = contacts[contacts['method_kind'].eq('outbound') & contacts['in_window']].dropna(subset=['Meldingnummer']).copy()
        _ca['_phase'] = _ca['Meldingnummer'].map(_phase_map)
        _ca = _ca.dropna(subset=['_phase'])
        _ac = eff_sorted.dropna(subset=['reach_method']).copy()            # cases with a first successful contact
        _ac['_phase'] = np.where(_ac['case_epnum'] >= 2, 'repeat case', 'first case')
        rows = []
        for _ph in ['first case', 'repeat case']:
            _cp = _ca[_ca['_phase'] == _ph]; _ap = _ac[_ac['_phase'] == _ph]
            for _chn in _cp['channel'].value_counts().index:
                _att = _cp[_cp['channel'] == _chn]
                if len(_att) < 100:
                    continue
                _via = _ap[_ap['reach_method'] == _chn]                         # first contact via this channel
                _acc_via = int(_via['arm'].eq('C_accepted').sum())
                rows.append((_ph, _chn, len(_att), _att['Meldingnummer'].nunique(), len(_via), _acc_via,
                             round(_acc_via / len(_att) * 100, 2)))
        if rows:
            display(pd.DataFrame(rows, columns=['phase', 'outbound channel', 'attempts', 'cases tried',
                                                'cases first reached via it', 'of those: help accepted',
                                                'accepted cases per 100 attempts']))
            _cb = _ac[_ac['reach_method'].astype(str).str.startswith('callback')]
            if len(_cb):
                print('\nCallbacks (the client contacted the municipality; the label states what prompted it) -')
                print('responses, not outreach methods, so they are listed separately:')
                display(_cb.groupby(['_phase', 'reach_method'])
                        .agg(cases=('Meldingnummer', 'size'),
                             help_accepted=('arm', lambda s: int((s == 'C_accepted').sum())))
                        .reset_index())
            print('\nHow to read: each accepted case is attributed to the outbound channel of its FIRST')
            print('successful contact; attempts count every outbound attempt of that channel in the phase.')
            print('Cases first reached by a callback are not attributed to one outbound channel, because the')
            print('labels combine channels (letter/email; phone/WhatsApp/SMS) and do not say which of several')
            print('earlier attempts prompted the call. Channels are chosen per case, so this describes what')
            print('happened - it is not a ranking of strategies. Weigh it against staff cost per attempt and')
            print('case mix before changing the workflow.')
        else:
            print('  Too few outbound attempts per channel and phase.')
    else:
        print('  No contact-attempt data available.')
    _section_done('15e. W - Workflow: Receptiveness Timing & Method Allocation')
except Exception as _err:
    _section_failed('15e. W - Workflow: Receptiveness Timing & Method Allocation', _err)


## 16. Generate the Results File

Run this cell **last**. It writes one self-contained, timestamped HTML file containing every
table, chart and printed value from above — this is the only file to send back. It opens in any
browser (Print → Save as PDF if needed). It contains mostly aggregates; §14e profiles the ten
most-signalling households individually (rank labels, no identifiers, dated signal timelines) and
some tables show small groups.


In [ ]:
_report_section('16. Generate the Results File to Send Back')
# Run status banner: COMPLETE only if the data were loaded in THIS run and every section finished.
_loaded = globals().get('DATA_LOADED_RUN') == _RUN_ID
_failed = [(t, m) for t, (s, m) in _SECTION_LOG.items() if s == 'FAILED']
_unfin = [t for t, (s, m) in _SECTION_LOG.items() if s == 'started' and not t.startswith('16.')]
_notrun = [t for t in EXPECTED_SECTIONS if t not in _SECTION_LOG]
_ok_all = _loaded and not (_failed or _unfin or _notrun)
RUN_STATUS = ('INCOMPLETE' if not _ok_all else 'COMPLETE, WITH DATA WARNINGS' if _DATA_WARNINGS else 'COMPLETE')
_bcol = {'COMPLETE': '#2e7d32', 'COMPLETE, WITH DATA WARNINGS': '#b26a00', 'INCOMPLETE': '#c62828'}[RUN_STATUS]
_bl = [f"Run {_RUN_ID} | input: {globals().get('INPUT_INFO', 'not loaded in this run')}",
       f"Data loaded in this run: {'yes' if _loaded else 'NO - results may be missing or come from another run'}",
       f"Sections finished: {sum(1 for s, _ in _SECTION_LOG.values() if s == 'ok')} of {len(EXPECTED_SECTIONS)}"]
if _failed:
    _bl += ['', 'Sections that FAILED (the run continued):'] + [f'  - {t}: {m}' for t, m in _failed]
if _unfin:
    _bl += ['', 'Sections started but not finished (interrupted?):'] + [f'  - {t}' for t in _unfin]
if _notrun:
    _bl += ['', 'Sections NOT RUN in this session:'] + [f'  - {t}' for t in _notrun]
if _DATA_WARNINGS:
    _bl += ['', 'Data warnings:'] + [f'  - {w}' for w in _DATA_WARNINGS]
_banner = (f"<div style='border:3px solid {_bcol};padding:6px 12px;border-radius:6px;margin:12px 0'>"
           f"<h2 style='color:{_bcol};margin:6px 0'>Run status: {RUN_STATUS}</h2>"
           f"<pre>{_html.escape(chr(10).join(_bl))}</pre></div>")
# Run summary: every skipped / not-estimable / failed analysis and model warning, listed at the
# top of the report with the section it came from.
_KEYS = ('skipped', 'not estimable', 'not reported', 'failed', 'model warning', 'too few',
         'not available', 'warning:')
_issues, _sec = [], ''
for _b in _REPORT_BLOCKS:
    if _b.startswith("<h2 class='sec'>"):
        _sec = _html.unescape(_b[len("<h2 class='sec'>"):-len('</h2>')])
    elif _b.startswith('<pre>'):
        _txt = _html.unescape(_b[5:-6]).strip()
        if _txt and not _txt.startswith('(n/a') and any(k in _txt.lower() for k in _KEYS):
            _issues.append((_sec, _txt.splitlines()[0][:160]))
_summary = ("<h2 class='sec'>Run summary - messages printed by the analyses (skips, not-estimable "
            "models, warnings); not a full validation</h2>"
            + ("<pre>" + _html.escape('\n'.join(f'[{s}] {t}' for s, t in _issues)) + "</pre>"
               if _issues else "<pre>None - every analysis produced output.</pre>"))
_doc = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>ELSA Vroegsignalering — Results</title>
<style>
 body{{font-family:-apple-system,Segoe UI,Arial,sans-serif;max-width:1000px;margin:24px auto;
       padding:0 18px;color:#222;line-height:1.4}}
 h1{{border-bottom:3px solid #2c7fb8;padding-bottom:6px}}
 h2.sec{{background:#2c7fb8;color:#fff;padding:6px 12px;border-radius:4px;margin-top:34px}}
 pre{{background:#f6f8fa;padding:8px 10px;border-radius:4px;white-space:pre-wrap;
      font-size:13px;font-family:SFMono-Regular,Consolas,monospace}}
 table{{border-collapse:collapse;margin:10px 0;font-size:13px}}
 th,td{{border:1px solid #ccc;padding:4px 9px;text-align:right}}
 th{{background:#eef3f7}}
 img{{max-width:100%;margin:12px 0;border:1px solid #eee;border-radius:4px}}
 .meta{{color:#666;font-size:13px}}
</style></head><body>
<h1>ELSA Vroegsignalering — Effectiveness Results</h1>
<p class="meta">Generated {_dt.datetime.now():%Y-%m-%d %H:%M} &middot; source file: {_html.escape(str(FILE))}
&middot; {len(_REPORT_BLOCKS)} content blocks</p>
<p class="meta">{_html.escape(globals().get('RUN_ENV', ''))}</p>
<p class="meta"><em>Mostly aggregated statistics, tables and charts. Section 14e profiles the ten
most-signalling households individually (rank labels, no identifiers, dated signal timelines);
some tables show small groups.</em></p>
{_banner}
{_summary}
{''.join(_REPORT_BLOCKS)}
</body></html>'''

# Write next to the notebook's working directory, timestamped so runs never overwrite
REPORT_PATH = Path(REPORT_PATH).resolve()
REPORT_PATH = REPORT_PATH.with_name(f"{REPORT_PATH.stem}_{_dt.datetime.now():%Y%m%d-%H%M%S}{REPORT_PATH.suffix}")
REPORT_PATH.write_text(_doc, encoding='utf-8')

_orig_print("=" * 70)
_orig_print(f"RUN STATUS: {RUN_STATUS}")
_orig_print("RESULTS FILE CREATED — this is the file to send back:")
_orig_print(f"   {REPORT_PATH}")
_orig_print("=" * 70)
_orig_print(f"  {len(_REPORT_BLOCKS)} content blocks captured (text, tables, charts).")
_orig_print(f"  Working directory: {Path.cwd()}")
_orig_print("  Open it in any browser; use Print -> Save as PDF if a PDF is preferred.")

# Clickable download link in Jupyter / JupyterLab
try:
    from IPython.display import FileLink
    _ipy_display(FileLink(str(REPORT_PATH)))
except Exception:
    pass

# In Google Colab, trigger a direct browser download automatically
try:
    from google.colab import files as _colab_files   # noqa
    _colab_files.download(str(REPORT_PATH))
except Exception:
    pass
